# Library

In [1]:
import pandas as pd
import re
import numpy as np

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
from datetime import datetime, timedelta
import time
import pickle

import warnings
warnings.filterwarnings('ignore')

import os
import string
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from styleframe import StyleFrame
from IPython.display import display, HTML

def pretty_print(df):
    return display( HTML( df.to_html().replace("\\n","<br>") ) )

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)
    

# Macro data

## FRED

### Testing out the FREDapi

In [7]:
# # if i dont have it
# !pip install fredapi

In [3]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key="90e8717fe5d6a68ad1d16f49cec89651")

def get_macro_series_single(series_id, vintage_dates=None):
    """
    Pull a macro series from FRED/ALFRED.
    
    series_id: e.g., "GDPC1", "CPIAUCSL", "UNRATE"
    vintage_dates: None, a single date string, or a list of date strings
    """
    start_date='1980-01-01'
    # Case 1: latest available vintage → do NOT pass vintage_dates
    if vintage_dates is None:
        data = fred.get_series(series_id)
    
    # Case 2: specific vintage(s)
    else:
        data = fred.get_series(series_id, vintage_dates=vintage_dates)
    
    # Convert to DataFrame for consistency
    df = pd.DataFrame(data)
    df.index.name = "observation_date"

    # Column naming logic
    if isinstance(vintage_dates, str) or vintage_dates is None:
        df.columns = [series_id]
    else:
        df.columns = data.columns

    # Restrict to desired period
    df = df[df.index >= start_date]

    return df

    

In [5]:
# This is for sanity check later
gdp_latest = get_macro_series_single("GDPC1")
gdp_latest


,GDPC1
observation_date,
1980-01-01,7341.557
1980-04-01,7190.289
1980-07-01,7181.743
1980-10-01,7315.677
1981-01-01,7459.022
...,...
2024-10-01,23586.542
2025-01-01,23548.210
2025-04-01,23770.976


In [37]:
# see the 2005 and see how much of a difference it is
check = gdp_latest[gdp_latest.index < "2005-02-01"]
check

,GDPC1
observation_date,
1980-01-01,7341.557
1980-04-01,7190.289
1980-07-01,7181.743
1980-10-01,7315.677
1981-01-01,7459.022
...,...
2004-01-01,15248.680
2004-04-01,15366.850
2004-07-01,15512.619


In [156]:
# pulling vintage from the lastest version
dates = "2005-05-10"
gdp_multi = get_macro_series_single("GDPC1", dates)
gdp_multi


,GDPC1
observation_date,
1980-01-01,5221.3
1980-04-01,5115.9
1980-07-01,5107.4
1980-10-01,5202.1
1981-01-01,5307.5
...,...
2004-01-01,10697.5
2004-04-01,10784.7
2004-07-01,10891.0


### Try adding in the data

In [5]:
df = pd.read_csv('FedSpeechesPC.csv', parse_dates = ['date'])
print(df.shape)


(7239, 24)


In [7]:
# checking my data, and there are missing dates
df['date'] = pd.to_datetime(df['date'], errors='coerce')
print(df[df['date'].isna()].shape)

# So i drop them
df = df.dropna(subset=['date'])
print(df.shape)


(8, 24)
(7231, 24)


In [9]:
speech_dates = pd.to_datetime(df['date'])

# Step 1: convert to strings for ALFRED
speech_dates_str = speech_dates.dt.strftime("%Y-%m-%d")

# Step 2: unique dates to avoid repeated API calls
unique_dates = sorted(set(speech_dates_str))

print(len(unique_dates))


4598


In [13]:
def fetch_with_batches(series_id, unique_buckets, lags, compute_growth, batch_size=150, sleep_seconds=120):
    cache = {}
    total = len(unique_buckets)

    for i in range(0, total, batch_size):
        batch = unique_buckets[i:i+batch_size]
        print(f"Processing batch {i//batch_size + 1} of { (total-1)//batch_size + 1 }")

        for b in batch:
            b_str = b.strftime("%Y-%m-%d")
        
            success = False
            for attempt in range(3):  # up to 3 attempts
                try:
                    df = get_macro_series_single(series_id, vintage_dates=b_str)
                    df = df.sort_index()

                    # Compute lags
                    for k in range(1, lags+1):
                        df[f"{series_id}_lag{k}"] = df[series_id].shift(k)
                    
                    # Compute growth
                    if compute_growth:
                        df[f"{series_id}_change"] = df[series_id].pct_change()
                        for k in range(1, lags+1):
                            df[f"{series_id}_change_lag{k}"] = df[f"{series_id}_change"].shift(k)

                    cache[b] = df
                    success = True
                    break  # exit retry loop
                except Exception as e:
                    print(f"Error on {b_str} (attempt {attempt+1}/3): {e}")
        
                    if attempt < 2:
                        # Wait longer before retrying
                        print("Waiting 30 seconds before retry...")
                        time.sleep(30)
                    else:
                        print(f"Failed after 3 attempts. Skipping {b_str}.")
                        # Mark as missing so you can re-run later
                        cache[b] = None
        
            # Optional: small pause between individual calls to be extra safe
            time.sleep(1)

        # Sleep between batches
        print(f"Sleeping {sleep_seconds} seconds to avoid rate limits...")
        time.sleep(sleep_seconds)

    return cache

def fetch_vintage_cache(series_id, speech_dates, lags=1, compute_growth=True):
    speech_dates = pd.to_datetime(speech_dates)

    # Get earliest and latest available real-time vintages
    releases = fred.get_series_all_releases(series_id)
    first_vintage = pd.to_datetime(releases["realtime_start"].min())
    last_vintage  = pd.to_datetime(releases["realtime_start"].max())

    # Clamp speech dates
    effective_dates = speech_dates.copy()
    effective_dates = effective_dates.where(effective_dates <= last_vintage, last_vintage)
    effective_dates = effective_dates.where(effective_dates >= first_vintage, first_vintage)

    # Monthly buckets
    buckets = effective_dates.dt.to_period("M").dt.to_timestamp("M")
    unique_buckets = sorted(buckets.unique())

    # Fetch with batching
    cache = fetch_with_batches(series_id, unique_buckets, lags, compute_growth)

    return cache

def extract_macro_from_cache(df, date_col, series_id, cache, lags=1, compute_growth=True):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Precompute min/max bucket dates from cache
    cache_keys = sorted(cache.keys())
    first_bucket = cache_keys[0]
    last_bucket = cache_keys[-1]

    # Prepare output columns
    df[f"{series_id}"] = None
    for k in range(1, lags+1):
        df[f"{series_id}_lag{k}"] = None

    if compute_growth:
        df[f"{series_id}_change"] = None
        for k in range(1, lags+1):
            df[f"{series_id}_change_lag{k}"] = None

    # Loop through rows in original order
    for idx, row in df.iterrows():
        speech_date = row[date_col]

        # Clamp to available vintage range
        effective_date = max(min(speech_date, last_bucket), first_bucket)

        # Bucket based on clamped date
        bucket = effective_date.to_period("M").to_timestamp("M")
        vintage_df = cache[bucket]

        # Extract the most recent observation available at the speech date
        df_cut = vintage_df[vintage_df.index <= speech_date]
        last_row = df_cut.iloc[-1]

        # Fill values
        df.at[idx, series_id] = last_row[series_id]

        for k in range(1, lags+1):
            df.at[idx, f"{series_id}_lag{k}"] = last_row[f"{series_id}_lag{k}"]

        if compute_growth:
            df.at[idx, f"{series_id}_change"] = last_row[f"{series_id}_change"]
            for k in range(1, lags+1):
                df.at[idx, f"{series_id}_change_lag{k}"] = last_row[f"{series_id}_change_lag{k}"]

    return df
    

#### Testing

In [45]:
cache_gdp = fetch_vintage_cache("GDPC1", df["date"])
with open("cache_GDPC1.pkl", "wb") as f:
    pickle.dump(cache_gdp, f)
    

Processing batch 1 of 3
Sleeping 120 seconds to avoid rate limits...
Processing batch 2 of 3
Sleeping 120 seconds to avoid rate limits...
Processing batch 3 of 3
Error on 2019-06-30 (attempt 1/3): Internal Server Error
Waiting 30 seconds before retry...
Sleeping 120 seconds to avoid rate limits...


In [97]:
df_gdp = extract_macro_from_cache(df,"date", "GDPC1", cache_gdp)


In [103]:
# this is to have a quick sanity check as well
print(df_gdp.shape)
print(df_gdp.columns.tolist())
df_gdp[["date","GDPC1"]].head()


(7099, 28)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1']


,date,GDPC1
0,2025-06-25,23512.717
1,2024-09-30,23223.906
2,2024-06-18,22758.752
3,2024-05-08,22749.846
4,2024-04-16,22768.866


#### Main Run

In [15]:
listVar = ["GDPC1","INDPRO", "UNRATE","PCEPILFE", "CPIAUCSL","MICH"]
listVarChange = ["GDPC1","PCEPILFE","CPIAUCSL","INDPRO"]

for var in listVar:
    print("Starting for "+var)
    if var in listVarChange:
        dfcache = fetch_vintage_cache(var, df["date"])
        with open("MacroData/ALFRED/cache_"+var+".pkl", "wb") as f:
            pickle.dump(dfcache, f)
        dfdum = extract_macro_from_cache(df,"date", var, dfcache)
        print(dfdum.shape)
        print(dfdum.columns.tolist())
        df = dfdum.copy()
    else:
        dfcache = fetch_vintage_cache(var, df["date"])
        dfdum = extract_macro_from_cache(df,"date", var, dfcache,compute_growth=False)
        with open("MacroData/ALFRED/cache_"+var+".pkl", "wb") as f:
            pickle.dump(dfcache, f)
        print(dfdum.shape)
        print(dfdum.columns.tolist())
        df = dfdum.copy()
    print("All done")
    print("\n")


Starting for GDPC1
Processing batch 1 of 3
Error on 1992-05-31 (attempt 1/3): Internal Server Error
Waiting 30 seconds before retry...
Error on 1995-11-30 (attempt 1/3): Internal Server Error
Waiting 30 seconds before retry...
Error on 1997-06-30 (attempt 1/3): Internal Server Error
Waiting 30 seconds before retry...
Sleeping 120 seconds to avoid rate limits...
Processing batch 2 of 3
Error on 2008-06-30 (attempt 1/3): Internal Server Error
Waiting 30 seconds before retry...
Error on 2015-01-31 (attempt 1/3): Internal Server Error
Waiting 30 seconds before retry...
Sleeping 120 seconds to avoid rate limits...
Processing batch 3 of 3
Sleeping 120 seconds to avoid rate limits...
(7231, 28)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'G

In [17]:
# Final check before exporting
print(df.shape)
print(df.columns.tolist())
df.head()


(7231, 44)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 'UNRATE', 'UNRATE_lag1', 'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'MICH', 'MICH_lag1']


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,PCEPILFE,PCEPILFE_lag1,PCEPILFE_change,PCEPILFE_change_lag1,CPIAUCSL,CPIAUCSL_lag1,CPIAUCSL_change,CPIAUCSL_change_lag1,MICH,MICH_lag1
0,Boston,0.0,False,False,Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",2025-06-25,0.408672,0.320187,...,125.512,125.288,0.001788,0.001359,320.58,320.321,0.000809,0.002209,5.0,6.6
1,Boston,0.0,False,False,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,2024-09-30,0.083191,0.874509,...,122.863,122.703,0.001304,0.001575,314.121,313.534,0.001872,0.001549,2.7,2.8
2,Boston,0.0,False,False,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",2024-06-18,0.327147,0.590322,...,122.045,121.944,0.000828,0.00259,313.225,313.207,0.000057,0.003129,3.0,3.3
3,Boston,0.0,False,False,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",2024-05-08,0.467524,0.391142,...,121.909,121.606,0.002492,0.003342,313.207,312.23,0.003129,0.003781,3.3,3.2
4,Boston,1.0,True,False,Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,2024-04-16,0.228169,0.700974,...,121.615,121.231,0.003168,0.002663,312.23,311.054,0.003781,0.004421,3.2,2.9


In [19]:
# df.to_csv('FedSpeecheswithMoreControl.csv', date_format='%Y-%m-%d %H:%M:%S', index = False)


#### Miscellaneous stuffs
- get yoy version inflation 

In [23]:
# to get the yoy inflation
def add_yoy_to_cache(cache, series_id):
    for bucket, df in cache.items():
        df = df.sort_index()
        df[f"{series_id}_yoy"] = df[series_id].pct_change(12)
        df[f"{series_id}_yoy_lag1"] = df[f"{series_id}_yoy"].shift(1)
        cache[bucket] = df
    return cache

def add_quantile_to_cache(cache, quantNum, series_id):
    for bucket, df in cache.items():
        df = df.sort_index()
        df[f"{series_id}_growth"] = df[series_id].pct_change()
        q = df[f"{series_id}_growth"].quantile(quantNum)
        
        indicator_col = f"{series_id}_low_q{int(quantNum*100)}"
        df[indicator_col] = (df[f"{series_id}_growth"] <= q).astype(int)
        df[indicator_col+"_lag1"] = df[indicator_col].shift(1)
        
        df = df.drop(columns = [f"{series_id}_growth"])
        cache[bucket] = df
    return cache
    
with open("MacroData/ALFRED/cache_PCEPILFE.pkl", "rb") as f:
    cache_pce = pickle.load(f)

with open("MacroData/ALFRED/cache_CPIAUCSL.pkl", "rb") as f:
    cache_cpi = pickle.load(f)

with open("MacroData/ALFRED/cache_GDPC1.pkl", "rb") as f:
    cache_gdp = pickle.load(f)

cache_pce = add_yoy_to_cache(cache_pce, "PCEPILFE")
cache_cpi = add_yoy_to_cache(cache_cpi, "CPIAUCSL")
quantNum = 0.1
cache_gdp = add_quantile_to_cache(cache_gdp, quantNum , "GDPC1")

# start adding it in:
df["date"] = pd.to_datetime(df["date"])

for series_id, cache in zip(["PCEPILFE","CPIAUCSL","GDPC1"],[cache_pce, cache_cpi, cache_gdp]):
    # Precompute min/max bucket dates from cache
    cache_keys = sorted(cache.keys())
    first_bucket = cache_keys[0]
    last_bucket = cache_keys[-1]
    
    # Prepare output columns
    if series_id == "GDPC1":
        df[f"{series_id}_low_q{int(quantNum*100)}"] = None
        df[f"{series_id}_low_q{int(quantNum*100)}_lag1"] = None
    else:
        df[f"{series_id}_yoy"] = None
        df[f"{series_id}_yoy_lag1"] = None
    
    # Loop through rows in original order
    for idx, row in df.iterrows():
        speech_date = row["date"]
    
        # Clamp to available vintage range
        effective_date = max(min(speech_date, last_bucket), first_bucket)
    
        # Bucket based on clamped date
        bucket = effective_date.to_period("M").to_timestamp("M")
        vintage_df = cache[bucket]
    
        # Extract the most recent observation available at the speech date
        df_cut = vintage_df[vintage_df.index <= speech_date]
        last_row = df_cut.iloc[-1]
    
        # Fill values
        if series_id == "GDPC1":
            df.at[idx, f"{series_id}_low_q{int(quantNum*100)}"] = last_row[f"{series_id}_low_q{int(quantNum*100)}"]
            df.at[idx, f"{series_id}_low_q{int(quantNum*100)}_lag1"] = last_row[f"{series_id}_low_q{int(quantNum*100)}_lag1"]
        else:
            df.at[idx, f"{series_id}_yoy"] = last_row[f"{series_id}_yoy"]
            df.at[idx, f"{series_id}_yoy_lag1"] = last_row[f"{series_id}_yoy_lag1"]





In [24]:
print(df.shape)
print(df.columns.tolist())
df.head()


(7231, 50)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 'UNRATE', 'UNRATE_lag1', 'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'MICH', 'MICH_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1']


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,CPIAUCSL_change,CPIAUCSL_change_lag1,MICH,MICH_lag1,PCEPILFE_yoy,PCEPILFE_yoy_lag1,CPIAUCSL_yoy,CPIAUCSL_yoy_lag1,GDPC1_low_q10,GDPC1_low_q10_lag1
0,Boston,0.0,False,False,Perspectives on the Economy from Susan M. Collins,Susan M. Collins,"Massachusetts towns of Fall River, New Bedford...",2025-06-25,0.408672,0.320187,...,0.000809,0.002209,5.0,6.6,0.026775,0.025774,0.023759,0.023337,0.0,0.0
1,Boston,0.0,False,False,Welcoming Remarks at the forum on “Meeting the...,Susan M. Collins,Federal Reserve Bank of Boston,2024-09-30,0.083191,0.874509,...,0.001872,0.001549,2.7,2.8,0.026785,0.026494,0.025912,0.029236,0.0,0.0
2,Boston,0.0,False,False,A Partnership for Progress,Susan M. Collins,"Lawrence, Massachusetts",2024-06-18,0.327147,0.590322,...,0.000057,0.003129,3.0,3.3,0.025726,0.027832,0.032502,0.033577,0.0,0.0
3,Boston,0.0,False,False,Reflections on Uncertainty and Patience in Mon...,Susan M. Collins,"Cambridge, Massachusetts",2024-05-08,0.467524,0.391142,...,0.003129,0.003781,3.3,3.2,0.027537,0.028128,0.033577,0.034751,0.0,0.0
4,Boston,1.0,True,False,Remarks for the National Association of Corpor...,Susan M. Collins,Federal Reserve Bank of Boston,2024-04-16,0.228169,0.700974,...,0.003781,0.004421,3.2,2.9,0.028204,0.028401,0.034751,0.031657,0.0,0.0


In [132]:
df.to_csv('FedSpeecheswithMoreControl.csv', date_format='%Y-%m-%d %H:%M:%S', index = False)


## Green/Tealbook

In [27]:
# We use the row format
def load_greenbook_sheet(
    excel_path,
    var_name,
    gbdate_col="GBdate",
    keep_suffixes=("B1", "F0", "F1")
):
    """
    Load one Greenbook/Tealbook sheet for a given variable.
    Keeps GBdate and selected suffix columns (e.g. B1, F0, F1).
    """
    # Read the sheet named like the variable
    df_gb = pd.read_excel(excel_path, sheet_name=var_name)

    # Ensure date is datetime
    df_gb[gbdate_col] = pd.to_datetime(df_gb[gbdate_col].astype(str), format="%Y%m%d")

    # Build the list of columns to keep
    cols_to_keep = [gbdate_col] + [f"{var_name}{suf}" for suf in keep_suffixes]
    df_gb = df_gb[cols_to_keep]

    # Sort by GBdate
    df_gb = df_gb.sort_values(gbdate_col).reset_index(drop=True)

    # Trick: ensure speeches before first GBdate still get the earliest forecast
    first_row = df_gb.iloc[0].copy()
    first_row[gbdate_col] = pd.Timestamp("1900-01-01")  # very early date
    df_gb = pd.concat([pd.DataFrame([first_row]), df_gb], ignore_index=True)
    df_gb = df_gb.sort_values(gbdate_col).reset_index(drop=True)

    return df_gb

def merge_greenbook_variable(
    df_speeches,
    date_col,
    excel_path,
    var_name,
    gbdate_col="GBdate",
    keep_suffixes=("B1", "F0", "F1")
):
    """
    For a single variable (e.g. 'gRGDP'), match Greenbook forecasts to speeches.
    Uses nearest GBdate <= speech date (backward merge).
    Returns df_speeches with new columns added.
    """
    df = df_speeches.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    # Load Greenbook sheet for this variable
    df_gb = load_greenbook_sheet(
        excel_path=excel_path,
        var_name=var_name,
        gbdate_col=gbdate_col,
        keep_suffixes=keep_suffixes
    )

    # Sort both for merge_asof
    df_sorted = df.sort_values(date_col)
    df_gb_sorted = df_gb.sort_values(gbdate_col)

    # Merge: for each speech date, get latest GBdate <= speech date
    merged = pd.merge_asof(
        left=df_sorted,
        right=df_gb_sorted,
        left_on=date_col,
        right_on=gbdate_col,
        direction="backward"
    )

    # Drop GBdate column if you don't want it in the final df
    merged = merged.drop(columns=[gbdate_col])

    # Restore original order
    merged = merged.sort_index()

    return merged

def add_greenbook_forecasts(
    df_speeches,
    date_col,
    excel_path,
    var_list,
    gbdate_col="GBdate",
    keep_suffixes=("B1", "F0", "F1")
):
    """
    For each variable in var_list, merge its Greenbook forecasts into df_speeches.
    Returns df with all variables added.
    """
    df = df_speeches.copy()

    for var in var_list:
        print(f"Matching Greenbook forecasts for {var}...")
        df = merge_greenbook_variable(
            df_speeches=df,
            date_col=date_col,
            excel_path=excel_path,
            var_name=var,
            gbdate_col=gbdate_col,
            keep_suffixes=keep_suffixes
        )

    return df



In [28]:
excel_path = "MacroData/GreenTealbook/GBweb_Row_Format.xlsx"
vars_greenbook = ["gRGDP", "gPGDP", "UNEMP", "HSTART", "gIP"]

df_with_gb = add_greenbook_forecasts(
    df_speeches=df,          # your original speech DataFrame
    date_col="date",         # column with speech dates
    excel_path=excel_path,
    var_list=vars_greenbook,
    keep_suffixes=("B1", "F0", "F1")  # past 1q, current, next
)


Matching Greenbook forecasts for gRGDP...
Matching Greenbook forecasts for gPGDP...
Matching Greenbook forecasts for UNEMP...
Matching Greenbook forecasts for HSTART...
Matching Greenbook forecasts for gIP...


In [31]:
# Final check before exporting
print(df_with_gb.shape)
print(df_with_gb.columns.tolist())
df_with_gb[["date",'gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
            'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
            'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
            'HSTARTB1', 'HSTARTF0', 'HSTARTF1', 
            'gIPB1', 'gIPF0', 'gIPF1']].head()


(7231, 65)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 'UNRATE', 'UNRATE_lag1', 'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'MICH', 'MICH_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1', 'gRGDPB1', 'gRGDPF0', 'gRGDPF1', 'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 'HSTARTB1', 'HSTARTF0', 'HSTARTF1', 'gIPB1', 'gIPF0', 'gIPF1']


,date,gRGDPB1,gRGDPF0,gRGDPF1,gPGDPB1,gPGDPF0,gPGDPF1,UNEMPB1,UNEMPF0,UNEMPF1,HSTARTB1,HSTARTF0,HSTARTF1,gIPB1,gIPF0,gIPF1
0,1986-01-06,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
1,1986-01-08,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
2,1986-01-15,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
3,1986-01-15,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
4,1986-01-16,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9


In [33]:
# I need all the date after the end of the year 2020 to be set to 0
cutoff = pd.Timestamp("2020-12-31")
mask = df_with_gb["date"] > cutoff

gb_cols = [
    c for c in df_with_gb.columns
    if any(prefix in c for prefix in ["gRGDP", "gPGDP", "UNEMP", "HSTART", "gIP"])
]

df_with_gb.loc[mask, gb_cols] = pd.NA

# at the same time just check everything to make sure it looks alright
pd.set_option("display.max_columns", None)
df_with_gb.head(10)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,negativeFin,hawkishMal,n_hawk_pair,n_dove_pair,board_role,IsBpres,timeOnly,score,abstract,info,read,disunity,strain,strainUpper,GDPC1,GDPC1_lag1,GDPC1_change,GDPC1_change_lag1,INDPRO,INDPRO_lag1,INDPRO_change,INDPRO_change_lag1,UNRATE,UNRATE_lag1,PCEPILFE,PCEPILFE_lag1,PCEPILFE_change,PCEPILFE_change_lag1,CPIAUCSL,CPIAUCSL_lag1,CPIAUCSL_change,CPIAUCSL_change_lag1,MICH,MICH_lag1,PCEPILFE_yoy,PCEPILFE_yoy_lag1,CPIAUCSL_yoy,CPIAUCSL_yoy_lag1,GDPC1_low_q10,GDPC1_low_q10_lag1,gRGDPB1,gRGDPF0,gRGDPF1,gPGDPB1,gPGDPF0,gPGDPF1,UNEMPB1,UNEMPF0,UNEMPF1,HSTARTB1,HSTARTF0,HSTARTF1,gIPB1,gIPF0,gIPF1
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,0.293320,1.000000,26,0,Not Board,False,NaN,NaN,1.514411,0.590045,-0.453618,0.429384,-0.185253,-0.671299,4390.5,4333.5,0.013153,0.005686,126.0,125.1,0.007194,0.005627,6.9,7.0,70.66,70.41,0.003551,0.002420,327.8,326.4,0.004289,0.005545,2.9,3.5,0.041261,0.041876,0.037342,0.035533,0.0,0.0,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,0.168316,0.837838,34,3,Not Board,True,NaN,NaN,2.961107,-0.537846,-1.080441,-1.152238,0.206547,-0.449667,4390.5,4333.5,0.013153,0.005686,126.0,125.1,0.007194,0.005627,6.9,7.0,70.66,70.41,0.003551,0.002420,327.8,326.4,0.004289,0.005545,2.9,3.5,0.041261,0.041876,0.037342,0.035533,0.0,0.0,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,0.236043,0.846154,12,1,Not Board,False,NaN,NaN,2.359110,0.187259,0.392625,0.960709,0.538797,-0.121827,4390.5,4333.5,0.013153,0.005686,126.0,125.1,0.007194,0.005627,6.9,7.0,70.66,70.41,0.003551,0.002420,327.8,326.4,0.004289,0.005545,2.9,3.5,0.041261,0.041876,0.037342,0.035533,0.0,0.0,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,0.273496,0.933333,29,1,Not Board,False,NaN,NaN,1.626665,0.684213,-0.421159,0.581613,-0.115528,-0.773794,4390.5,4333.5,0.013153,0.005686,126.0,125.1,0.007194,0.005627,6.9,7.0,70.66,70.41,0.003551,0.002420,327.8,326.4,0.004289,0.005545,2.9,3.5,0.041261,0.041876,0.037342,0.035533,0.0,0.0,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,0.335514,0.800000,9,1,Not Board,False,NaN,NaN,-0.264672,-0.489864,0.010725,0.297671,0.056723,-0.569484,4390.5,4333.5,0.013153,0.005686,126.0,125.1,0.007194,0.005627,6.9,7.0,70.66,70.41,0.003551,0.002420,327.8,326.4,0.004289,0.005545,2.9,3.5,0.041261,0.041876,0.037342,0.035533,0.0,0.0,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
5,BoardGovernors,NaN,False,False,Statement before the Subcommittee on Financial...,Emmett John Rice,Board of Governors of the Federal Reserve,1986-01-28,0.256045,0.547524,0.196432,0.000000,0,0,Governor,False,NaN,NaN,-1.683337,-0.566956,0.347836,2.042900,0.775399,0.630956,4390.5,4333.5,0.013153,0.005686,126.0,125.1,0.007194,0.005627,6.9,7.0,70.66,70.41,0.003551,0.002420,327.8,326.4,0.004289,0.005545,2.9,3.5,0.041261,0.041876,0.037342,0.035533,0.0,0.0,4.3,2.6,2.0,2.3,3.7,4.0,7.2,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9
6,BoardGovernors,NaN,False,False,Statement before the Subcommittee on Domestic ...,Paul A Volcker,Board of Governors of the Federal Reserve,1986-01-29,0.173745,0.773427,0.052828,-0.333333,1,2,Chair,False,NaN,NaN,0.154294,0.767038,0.588473,1.006859,1.145506,2.244231,4390.5,4

In [35]:
# just a quick side note, making indicator 
df_with_gb['is_monday'] = (df['date'].dt.weekday == 0).astype(int)
df_with_gb['is_friday'] = (df['date'].dt.weekday == 4).astype(int)


In [36]:
df_with_gb.to_csv('FedSpeecheswithMoreControl.csv', date_format='%Y-%m-%d %H:%M:%S', index = False)


# Import data with macro control already

In [5]:
df = pd.read_csv("FedSpeecheswithMoreControl.csv", parse_dates = ['date'])
print(df.shape)
df.head()


(7231, 67)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,UNEMPF0,UNEMPF1,HSTARTB1,HSTARTF0,HSTARTF1,gIPB1,gIPF0,gIPF1,is_monday,is_friday
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9,0,0
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9,1,0
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9,0,0
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9,0,0
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,7.1,7.1,1.66,1.75,1.75,1.6,1.7,1.9,0,0


In [7]:
print(df.columns.tolist())

['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 'UNRATE', 'UNRATE_lag1', 'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'MICH', 'MICH_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1', 'gRGDPB1', 'gRGDPF0', 'gRGDPF1', 'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 'HSTARTB1', 'HSTARTF0', 'HSTARTF1', 'gIPB1', 'gIPF0', 'gIPF1', 'is_monday', 'is_friday']


In [9]:
df['date'].min()

Timestamp('1986-01-06 00:00:00')

# More speaker variable
- 2 sources! They are merged into a single file
- The first coming from Riboni, A., & Ruge‑Murcia, F. (2025)
- The second coming from Conti‑Brown & Nygaard Federal Reserve Bank Boards of Directors Biographical Database
- The end results of the combination are imported


## Investigate, combine Alessandro & Francisco data + salt vs fresh water


In [6]:
# # import and print columns; replace -99 with missing
# dfold = pd.read_excel("SpeakerData/Ales/characteristics_FOMC.xlsx")
# print(dfold.shape)
# print(dfold.columns.tolist())
# dfold.head(10)


(113, 9)
['name', 'Gender', 'Phd', 'Age', 'Employment', 'State Born', 'University', 'Education', 'Post']


,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
0,Angell,male,1.0,1930.0,"academia, state politics and banking",Kansas,University of Kansas,economics,private
1,Balles,male,1.0,1921.0,academia and Fed,Illinois,Ohio State,economics,private
2,Baughman,male,0.0,1915.0,Fed,Iowa,Minnesota,agriculture and economics,low
3,Bernanke,male,1.0,1953.0,academia,Georgia,MIT,economics,medium
4,Bies,female,1.0,1947.0,academia and banking,New York,Northwestern,economics,private
5,Black,male,1.0,1927.0,Fed,Kentucky,University of Virginia,economics,medium
6,Blinder,male,1.0,1945.0,academia,New York,MIT,economics,medium
7,Boehne,male,1.0,1926.0,Fed,Indiana,Indiana University,economics,private
8,Bostic,male,1.0,1966.0,Fed and government,New Jersey,Stanford,economics,NaN
9,Boykin,male,0.0,1926.0,Fed,New Mexico,University of Texas at Austin,law,NaN


In [8]:
# dfnew = pd.read_excel("SpeakerData/Ales/new_names_2024.xlsx")
# print(dfnew.shape)
# print(dfnew.columns.tolist())
# dfnew.head(10)


(43, 13)
['name', 'MEMBER', 'Gender', 'Phd', 'Age', 'Education', 'University', 'Post', 'Employment', 'State Born', 'University.1', 'BANK', 'APPOINTMENT']


,name,MEMBER,Gender,Phd,Age,Education,University,Post,Employment,State Born,University.1,BANK,APPOINTMENT
0,Barkin,201,1,0,-99,-99,-99,-99,NaN,NaN,NaN,1,-99
1,Barr,202,1,0,-99,-99,-99,-99,NaN,NaN,NaN,0,1
2,Black Meredith,204,-99,-99,-99,-99,-99,-99,NaN,NaN,NaN,1,-99
3,Bostic,205,1,1,-99,-99,1,-99,NaN,NaN,Stanford,1,-99
4,Bowman,206,0,0,-99,-99,-99,-99,NaN,NaN,NaN,0,-1
5,Brainard,207,0,1,-99,-99,1,-99,NaN,NaN,Harvard,0,1
6,Bullard,208,1,1,-99,-99,3,-99,NaN,NaN,Indiana,1,-99
7,Clarida,209,1,1,-99,-99,1,-99,NaN,NaN,Harvard,0,-1
8,Collins,210,0,1,-99,-99,1,-99,NaN,NaN,MIT,1,-99
9,Cook,211,0,1,-99,-99,1,-99,NaN,NaN,Berkeley,0,1


In [10]:
# # "fixing" the new dataset to conform with the old
# # new columns, do i need them? Change name
# dfnew = dfnew.drop(columns = ["MEMBER","BANK","APPOINTMENT","University"])
# dfnew = dfnew.rename(columns = {"University.1":"University"})

# # turn everything with -99 to missing value
# dfnew = dfnew.replace(-99, np.nan)

# # gender needs to be consistent to integer
# genderMap = {"male":1 ,"female":0 }
# dfold["Gender"] = dfold["Gender"].map(genderMap)


In [12]:
# # Merging
# commonCols = ['name', 'Gender', 'Phd', 'Age', 
#               'Employment', 'State Born', 
#               'University', 'Education', 'Post']

# # make sure they indeed have the same columns
# dfold = dfold[commonCols]

# # For new_ales, if some columns are missing, create them as NaN
# for col in commonCols:
#     if col not in dfnew.columns:
#         dfnew[col] = np.nan

# dfnew = dfnew[commonCols]

# # merge
# dfspeakerAles = pd.concat([dfold, dfnew], ignore_index=True)


In [14]:
# # checking for duplicates
# dupes = dfspeakerAles[dfspeakerAles.duplicated(subset="name", keep=False)]
# duplicate_names = dupes["name"].unique()
# print(duplicate_names)

# for n in duplicate_names:
#     print("\n===== DUPLICATE:", n, "=====")
#     display(dupes[dupes["name"] == n])
    
# dfspeakerAles = dfspeakerAles.drop_duplicates(subset="name", keep="first")


['Bostic' 'Brainard' 'Bullard' 'Dudley' 'Fischer' 'George' 'Harker'
 'Kaplan' 'Kashkari' 'Kocherlakota' 'Lockhart' 'Mester' 'Powell' 'Quarles'
 'Raskin' 'Stein' 'Tarullo' 'Williams' 'Clarida' 'Bowman' 'Barkin' 'Daly']

===== DUPLICATE: Bostic =====


,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
8,Bostic,1.0,1.0,1966.0,Fed and government,New Jersey,Stanford,economics,NaN
116,Bostic,1.0,1.0,NaN,NaN,NaN,Stanford,NaN,NaN



===== DUPLICATE: Brainard =====


,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
10,Brainard,0.0,1.0,1962.0,academia and government,Foreign (Germany),Harvard,economics,low
118,Brainard,0.0,1.0,NaN,NaN,NaN,Harvard,NaN,NaN



===== DUPLICATE: Bullard =====


...

,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
107,Bowman,0.0,0.0,1971.0,banking and government,Hawaii,Washburn,law,NaN
117,Bowman,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



===== DUPLICATE: Barkin =====


,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
111,Barkin,1.0,0.0,1961.0,banking and consulting,Florida,Harvard,law and mba,NaN
113,Barkin,1.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN



===== DUPLICATE: Daly =====


,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
112,Daly,0.0,1.0,1962.0,Fed,Missouri,Syracuse,economics,NaN
123,Daly,0.0,1.0,NaN,NaN,NaN,Syracuse,NaN,NaN


In [16]:
# # best to check again on some of the name to see if i do indeed keep the right version
# print(dfspeakerAles[dfspeakerAles["name"] == "Daly"])

# # this is simply to check if any new rows provide better information
# for n in duplicate_names:
#     rows = dupes[dupes["name"] == n]
#     old_row = rows.iloc[0]
#     new_row = rows.iloc[1]

#     diff = new_row.notna() & old_row.isna()
#     if diff.any():
#         print("\nPotential useful info in new row for:", n)
#         print(new_row[diff])
        

     name  Gender  Phd     Age Employment State Born University  Education  \
112  Daly     0.0  1.0  1962.0        Fed   Missouri   Syracuse  economics   

    Post  
112  NaN  


In [18]:
# # the name bro gonna be a problem
# dfspeakerAles["name"] = (
#     dfspeakerAles["name"]
#     .str.strip()
#     .str.replace(r"\s+", " ", regex=True)
# )
# dfspeakerAles.head()

,name,Gender,Phd,Age,Employment,State Born,University,Education,Post
0,Angell,1.0,1.0,1930.0,"academia, state politics and banking",Kansas,University of Kansas,economics,private
1,Balles,1.0,1.0,1921.0,academia and Fed,Illinois,Ohio State,economics,private
2,Baughman,1.0,0.0,1915.0,Fed,Iowa,Minnesota,agriculture and economics,low
3,Bernanke,1.0,1.0,1953.0,academia,Georgia,MIT,economics,medium
4,Bies,0.0,1.0,1947.0,academia and banking,New York,Northwestern,economics,private


## Investigate Kaleb data + merge with Alessandro

### import the data

In [23]:
# dfKaleb = pd.read_excel("SpeakerData/KalebNygaard/FOMCbiodata.xlsx")
# print(dfKaleb.shape)
# print(dfKaleb.columns.tolist())
# dfKaleb.head()


(191, 29)
['Name', 'Org', 'Start Year', 'End Year', 'Race', 'Gender', 'Birth Year', 'Age at Start', 'Age at End', 'Title', 'Degree1', 'Major/Field1', 'School1', 'Year1', 'Degree2', 'Major/Field2', 'School2', 'Year2', 'Degree3', 'Major/Field3', 'School3', 'Year3', 'TD: Degree', 'TD: Major/Field', 'TD: Major/Field Category', 'TD: School', 'TD: Year', 'Pre-Fed Career', 'Pre-Fed Org']


,Name,Org,Start Year,End Year,Race,Gender,Birth Year,Age at Start,Age at End,Title,...,Major/Field3,School3,Year3,TD: Degree,TD: Major/Field,TD: Major/Field Category,TD: School,TD: Year,Pre-Fed Career,Pre-Fed Org
0,Mark H. Willes,Minneapolis,1977,1980.0,W,M,1941.0,36.0,39.0,President,...,NaN,NaN,NaN,PhD,Economics,NaN,Columbia University,NaN,Professor,University of Pennsylvania
1,Kevin M. Warsh,Board of Governors,2006,2011.0,W,M,1970.0,36.0,41.0,Governor,...,NaN,NaN,NaN,JD,Law,NaN,Harvard University,1995.0,Banker,Morgan Stanley
2,Manuel H. Johnson,Board of Governors,1986,1990.0,W,M,1949.0,37.0,41.0,Governor/Vice Chair,...,Economics,Florida State University,1977.0,PhD,Economics,NaN,Florida State University,1977.0,Professor,George Mason University
3,Lawrence B. Lindsey,Board of Governors,1991,1997.0,W,M,1954.0,37.0,43.0,Governor,...,Economics,Harvard University,1985.0,PhD,Economics,NaN,Harvard University,1985.0,Professor,Harvard University
4,Karen N. Horn,Cleveland,1982,1987.0,W,F,1944.0,38.0,43.0,President,...,NaN,NaN,NaN,PhD,Economics,NaN,Johns Hopkins University,1971.0,Economist,First National Bank of Boston


In [25]:
# # thinking about what variable to use:
# keepCols = ['Name', 'Start Year', 'End Year', 'Race', 'Gender', 'Birth Year',
#             'TD: Degree', 'TD: Major/Field', 'TD: School', 'Pre-Fed Career']
# dfKaleb = dfKaleb[keepCols]


In [27]:
# # there are ppl appearing multiple times! because they change their roles
# # I need to investigate this:
# # Identify duplicated names (people with multiple roles)
# dupes = dfKaleb[dfKaleb.duplicated(subset="Name", keep=False)]

# # List unique duplicated names
# duplicate_names = dupes["Name"].unique()
# print(duplicate_names)

# for n in duplicate_names:
#     print("\n===== DUPLICATE:", n, "=====")
#     display(dfKaleb[dfKaleb["Name"] == n])


['E. Gerald Corrigan' 'Philip E. Coldwell' 'Janet L. Yellen'
 'Paul A. Volcker' 'John C. Williams' 'Oliver S. Powell']

===== DUPLICATE: E. Gerald Corrigan =====


,Name,Start Year,End Year,Race,Gender,Birth Year,TD: Degree,TD: Major/Field,TD: School,Pre-Fed Career
6,E. Gerald Corrigan,1980,1984.0,W,M,1941.0,PhD,Economics,Fordham University,Economist
23,E. Gerald Corrigan,1985,1993.0,W,M,1941.0,PhD,Economics,Fordham University,Economist



===== DUPLICATE: Philip E. Coldwell =====


,Name,Start Year,End Year,Race,Gender,Birth Year,TD: Degree,TD: Major/Field,TD: School,Pre-Fed Career
35,Philip E. Coldwell,1968,1974.0,W,M,1922.0,PhD,Economics,University of Wisconsin,Professor
82,Philip E. Coldwell,1974,1980.0,W,M,1922.0,PhD,Economics,University of Wisconsin,Professor



===== DUPLICATE: Janet L. Yellen =====


...

,Name,Start Year,End Year,Race,Gender,Birth Year,TD: Degree,TD: Major/Field,TD: School,Pre-Fed Career
53,Paul A. Volcker,1975,1979.0,W,M,1927.0,Master's,Public Administration,Harvard University,Economist
83,Paul A. Volcker,1979,1987.0,W,M,1927.0,Master's,Public Administration,Harvard University,Economist



===== DUPLICATE: John C. Williams =====


,Name,Start Year,End Year,Race,Gender,Birth Year,TD: Degree,TD: Major/Field,TD: School,Pre-Fed Career
66,John C. Williams,2011,2018.0,W,M,1962.0,PhD,Economics,Stanford Univeristy,Economist
133,John C. Williams,2018,NaN,W,M,1962.0,PhD,Economics,Stanford University,Economist



===== DUPLICATE: Oliver S. Powell =====


,Name,Start Year,End Year,Race,Gender,Birth Year,TD: Degree,TD: Major/Field,TD: School,Pre-Fed Career
102,Oliver S. Powell,1950,1952.0,W,M,1896.0,Bachelor's,Economics,University of Minnesota,Economist
126,Oliver S. Powell,1952,1957.0,W,M,1896.0,Bachelor's,Economics,University of Minnesota,Economist


This is ok, since there are only a few instances and their background remains identical!

### name problems

In [32]:
# # this is to prep for merging the two datasets!
# phdMap = {1:"PhD" ,0:"None/unknown" }
# dfspeakerAles["Phd"] = dfspeakerAles["Phd"].map(phdMap)

# dfspeakerAles.rename(columns = {"name":"Name", 
#                                 "Age":"Birth Year",
#                                 "University":"TD: School",
#                                 "Education":"TD: Major/Field",
#                                 "Employment":"Pre-Fed Career",
#                                 "Phd":"TD: Degree"}, inplace=True)

# dfspeakerAles.drop(columns = ["State Born","Post"], inplace=True)

# genderMap = {"M":1 ,"F":0 }
# dfKaleb["Gender"] = dfKaleb["Gender"].map(genderMap)


In [34]:
# # Clean names is not enough at all
# dfKaleb["Name"] = (
#     dfKaleb["Name"].str.strip().str.replace(r"\s+", " ", regex=True)
# )
# dfspeakerAles["Name"] = (
#     dfspeakerAles["Name"].str.strip().str.replace(r"\s+", " ", regex=True)
# )

# # how do i deal with this before moving to merging?
# # first get last name of Kaleb
# dfKaleb["last_name"] = (
#     dfKaleb["Name"]
#     .str.replace(",", "")
#     .str.split()
#     .str[-1]
# )

# last_counts = dfKaleb["last_name"].value_counts()
# duplicate_lastnames = last_counts[last_counts > 1].index
# print(duplicate_lastnames)

# for ln in duplicate_lastnames:
#     print("\n===== LAST NAME:", ln, "=====")
#     display(dfKaleb[dfKaleb["last_name"] == ln][["Name", "Start Year", "End Year","Birth Year"]])
    
# problem_lastnames = []

# for ln in duplicate_lastnames:
#     full_names = dfKaleb[dfKaleb["last_name"] == ln]["Name"].unique()
#     if len(full_names) > 1:
#         problem_lastnames.append((ln, full_names))

# problem_lastnames


Index(['Jr.', 'Powell', 'Williams', 'Yellen', 'Corrigan', 'Miller', 'Volcker',
       'Young', 'Coldwell', 'Evans'],
      dtype='object', name='last_name')

===== LAST NAME: Jr. =====


,Name,Start Year,End Year,Birth Year
7,"G. H. King, Jr.",1959,1963.0,1920.0
21,"David W. Mullins, Jr.",1990,1994.0,1946.0
25,William McChesney Martin Jr.,1951,1970.0,1906.0
33,Roger W. Ferguson Jr.,1997,2006.0,1951.0
34,Hugh D. Galusha Jr.,1965,1971.0,1919.0
54,Robert D. Mcteer Jr.,1991,2004.0,1943.0
81,"James K. Vardaman, Jr.",1946,1958.0,1894.0
89,William S. Mclarin Jr.,1941,1951.0,1889.0
103,"ALl. Mills, Jr.",1952,1965.0,1898.0
109,J. Alfred Broaddus Jr.,1993,2004.0,1939.0



===== LAST NAME: Powell =====


,Name,Start Year,End Year,Birth Year
102,Oliver S. Powell,1950,1952.0,1896.0
126,Oliver S. Powell,1952,1957.0,1896.0
157,Jerome H. Powell,2012,NaN,1953.0



===== LAST NAME: Williams =====


...


===== LAST NAME: Coldwell =====


,Name,Start Year,End Year,Birth Year
35,Philip E. Coldwell,1968,1974.0,1922.0
82,Philip E. Coldwell,1974,1980.0,1922.0



===== LAST NAME: Evans =====


,Name,Start Year,End Year,Birth Year
67,Charles L. Evans,2007,2023.0,1958.0
80,Rudolph M. Evans,1942,1954.0,1890.0


[('Jr.',
  array(['G. H. King, Jr.', 'David W. Mullins, Jr.',
         'William McChesney Martin Jr.', 'Roger W. Ferguson Jr.',
         'Hugh D. Galusha Jr.', 'Robert D. Mcteer Jr.',
         'James K. Vardaman, Jr.', 'William S. Mclarin Jr.',
         'ALl. Mills, Jr.', 'J. Alfred Broaddus Jr.',
         'Edward W. Kelley Jr.', 'Philip C. Jackson, Jr.'], dtype=object)),
 ('Powell', array(['Oliver S. Powell', 'Jerome H. Powell'], dtype=object)),
 ('Williams', array(['Alfred H. Williams', 'John C. Williams'], dtype=object)),
 ('Miller', array(['G. William Miller', 'Paul E. Miller'], dtype=object)),
 ('Young', array(['Roy A. Young', 'Clifford S. Young'], dtype=object)),
 ('Evans', array(['Charles L. Evans', 'Rudolph M. Evans'], dtype=object))]

Unfortunately, there are duplicate, but now we both have birth year, this is key to merge on top of name!

In [37]:
# # for Ales, check multi name words
# ales_multiword = dfspeakerAles[dfspeakerAles["Name"].str.split().str.len() > 1]
# ales_multiword


,Name,Gender,TD: Degree,Birth Year,Pre-Fed Career,TD: School,TD: Major/Field
115,Black Meredith,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
# # check duplicates
# ales_dupes = dfspeakerAles[dfspeakerAles.duplicated(subset="Name", keep=False)]
# ales_dupes


,Name,Gender,TD: Degree,Birth Year,Pre-Fed Career,TD: School,TD: Major/Field


In [41]:
# ales_problem = dfspeakerAles[dfspeakerAles["Name"].isin([p[0] for p in problem_lastnames])]
# ales_problem


,Name,Gender,TD: Degree,Birth Year,Pre-Fed Career,TD: School,TD: Major/Field
20,Evans,1.0,PhD,1958.0,academia and Fed,Carnegie Mellon,economics
65,Miller,1.0,None/unknown,1925.0,industry,Berkeley,law
78,Powell,1.0,None/unknown,1953.0,"legal, banking, government",Georgetown,law
98,Williams,1.0,PhD,1962.0,Fed and government,Stanford,economics


### merging

In [45]:
# print(dfspeakerAles.shape)
# print(dfKaleb.shape)

# merged = dfspeakerAles.merge(
#     dfKaleb,
#     left_on=["Name", "Birth Year"],  
#     right_on=["last_name", "Birth Year"],
#     how="outer",
#     suffixes=("_ales", "_kaleb")
# )

# print(merged.shape)
# merged.head()


(134, 7)
(191, 11)
(234, 17)


,Name_ales,Gender_ales,TD: Degree_ales,Birth Year,Pre-Fed Career_ales,TD: School_ales,TD: Major/Field_ales,Name_kaleb,Start Year,End Year,Race,Gender_kaleb,TD: Degree_kaleb,TD: Major/Field_kaleb,TD: School_kaleb,Pre-Fed Career_kaleb,last_name
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Carl E. Allen,1956.0,1961.0,W,1.0,Bachelor's,NaN,Dartmouth College,Banker,Allen
1,Angell,1.0,PhD,1930.0,"academia, state politics and banking",University of Kansas,economics,Wayne D. Angell,1986.0,1994.0,W,1.0,PhD,Economics,Ottawa University,Politician,Angell
2,NaN,NaN,NaN,1897.0,NaN,NaN,NaN,C. Canby Balderston,1954.0,1966.0,W,1.0,PhD,Law,University of Pennsylvania,Professor,Balderston
3,Balles,1.0,PhD,1921.0,academia and Fed,Ohio State,economics,John J. Balles,1972.0,1986.0,W,1.0,Master's,Economics,Ohio State University,Economist,Balles
4,Barkin,1.0,None/unknown,1961.0,banking and consulting,Harvard,law and mba,Thomas I. Barkin,2018.0,NaN,W,1.0,JD,Law,Harvard University,Consultant,Barkin


Now I am checking the merge

In [48]:
# # how much is matched
# matched_ales = merged[merged["Name_ales"].notna()]
# len(matched_ales["Name_kaleb"].unique())


92

In [50]:
# # Ales name that not match Kaleb
# unmatched_ales = merged[merged["Name_kaleb"].isna() & merged["Name_ales"].notna()]
# print(unmatched_ales.shape)
# unmatched_ales[["Name_ales", "Birth Year"]]


(43, 17)


,Name_ales,Birth Year
6,Barr,NaN
11,Black Meredith,NaN
13,Boehne,1926.0
21,Broaddus,1939.0
33,Collins,NaN
35,Cook,NaN
38,Cumming,1953.0
45,Dubbert,NaN
55,Feldman,NaN
56,Ferguson,1951.0


In [52]:
# # this is the other way around
# unmatched_kaleb = merged[merged["Name_ales"].isna() & merged["Name_kaleb"].notna()]
# print(unmatched_kaleb.shape)
# unmatched_kaleb[["Name_kaleb", "Start Year", "End Year"]]


(94, 17)


,Name_kaleb,Start Year,End Year
0,Carl E. Allen,1956.0,1961.0
2,C. Canby Balderston,1954.0,1966.0
5,Michael S. Barr,2022.0,NaN
14,Edward G. Boehne,1981.0,2000.0
15,Karl R. Bopp,1958.0,1970.0
...,...,...,...
222,Edward A. Wayne,1961.0,1968.0
223,Laurence F. Whittemore,1946.0,1948.0
225,Alfred H. Williams,1941.0,1958.0
232,Roy A. Young,1930.0,1942.0


In [54]:
# print(merged["Name_ales"].value_counts()[merged["Name_ales"].value_counts() > 1])
# print(merged["Name_kaleb"].value_counts()[merged["Name_kaleb"].value_counts() > 1])


Name_ales
Yellen      3
Coldwell    2
Williams    2
Corrigan    2
Volcker     2
Name: count, dtype: int64
Name_kaleb
Janet L. Yellen       3
Paul A. Volcker       2
E. Gerald Corrigan    2
John C. Williams      2
Oliver S. Powell      2
Philip E. Coldwell    2
Name: count, dtype: int64


In [56]:
# # how many unique people?
# unique_people = merged["Name_kaleb"].fillna(merged["Name_ales"]).nunique()
# unique_people


227

In [59]:
# # let us just merge them for now, and deal with the rest later
# rawList = ["Name","Gender","TD: Degree",'Pre-Fed Career', 'TD: School', 'TD: Major/Field']
# for var in rawList:
#     merged[var+"_final"] = np.where(
#         merged[var+"_kaleb"].notna(),
#         merged[var+"_kaleb"],
#         merged[var+"_ales"]
#     )
    

In [61]:
# # so now we need to decide which to keep and which to not keep
# print(merged.isnull().sum())


Name_ales                 94
Gender_ales              104
TD: Degree_ales          105
Birth Year                28
Pre-Fed Career_ales      116
TD: School_ales          109
TD: Major/Field_ales     119
Name_kaleb                43
Start Year                43
End Year                  59
Race                      43
Gender_kaleb              43
TD: Degree_kaleb          43
TD: Major/Field_kaleb     82
TD: School_kaleb          59
Pre-Fed Career_kaleb      49
last_name                 43
Name_final                 0
Gender_final              10
TD: Degree_final          11
Pre-Fed Career_final      27
TD: School_final          31
TD: Major/Field_final     62
dtype: int64


In [63]:
# merged_final = merged[["Name_final","Name_kaleb","Gender_final",
#                   "Birth Year","Start Year","End Year","Race",
#                   "TD: Degree_final","Pre-Fed Career_final",
#                   "TD: School_final","TD: Major/Field_final"]]
# print(merged_final.shape)

# # There are still missing value!
# # Identify background columns
# background_cols = [
#     "Gender_final", "Birth Year","Start Year","End Year","Race",
#                   "TD: Degree_final","Pre-Fed Career_final",
#                   "TD: School_final","TD: Major/Field_final"
# ]

# # Rows where all background fields are missing
# only_name_rows = merged_final[
#     merged_final[background_cols].isna().all(axis=1)
# ]
# print(only_name_rows.shape)
# display(only_name_rows)

# # just drop these, no harm done
# merged_final = merged_final[~merged_final[background_cols].isna().all(axis=1)]
# print(merged_final.shape)


(234, 11)
(10, 11)


,Name_final,Name_kaleb,Gender_final,Birth Year,Start Year,End Year,Race,TD: Degree_final,Pre-Fed Career_final,TD: School_final,TD: Major/Field_final
11,Black Meredith,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
45,Dubbert,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
55,Feldman,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
72,Gooding,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,Gould,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
145,Meder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
155,Montgomery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159,Mullinix,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
165,O’Neill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,Shukla,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(224, 11)


In [65]:
# # checking missing value again
# print(merged_final.isnull().sum())


Name_final                0
Name_kaleb               33
Gender_final              0
Birth Year               18
Start Year               33
End Year                 49
Race                     33
TD: Degree_final          1
Pre-Fed Career_final     17
TD: School_final         21
TD: Major/Field_final    52
dtype: int64


In [67]:
# listCheck = merged_final.columns.tolist()

# for col in listCheck:
#     print(f"\n===== Missing {col} =====")
#     missing_rows = merged_final[merged_final[col].isna()]
#     display(missing_rows)



===== Missing Name_final =====


,Name_final,Name_kaleb,Gender_final,Birth Year,Start Year,End Year,Race,TD: Degree_final,Pre-Fed Career_final,TD: School_final,TD: Major/Field_final



===== Missing Name_kaleb =====


,Name_final,Name_kaleb,Gender_final,Birth Year,Start Year,End Year,Race,TD: Degree_final,Pre-Fed Career_final,TD: School_final,TD: Major/Field_final
6,Barr,NaN,1.0,NaN,NaN,NaN,NaN,None/unknown,NaN,NaN,NaN
13,Boehne,NaN,1.0,1926.0,NaN,NaN,NaN,PhD,Fed,Indiana University,economics
21,Broaddus,NaN,1.0,1939.0,NaN,NaN,NaN,PhD,Fed and government,Indiana University,NaN
33,Collins,NaN,0.0,NaN,NaN,NaN,NaN,PhD,NaN,MIT,NaN
35,Cook,NaN,0.0,NaN,NaN,NaN,NaN,PhD,NaN,Berkeley,NaN
38,Cumming,NaN,0.0,1953.0,NaN,NaN,NaN,PhD,Fed,Minnesota,economics
56,Ferguson,NaN,1.0,1951.0,NaN,NaN,NaN,PhD,consulting,Harvard,economics
61,Ford,NaN,1.0,1939.0,NaN,NaN,NaN,PhD,academia and banking,University of Michigan,economics
74,Goolsbee,NaN,1.0,NaN,NaN,NaN,NaN,PhD,NaN,MIT,NaN
80,Guynn,NaN,1.0,1942.0,NaN,NaN,NaN,None/unknown,Fed,Georgia Tech,mba



===== Missing Gender_final =====


...

,Name_final,Name_kaleb,Gender_final,Birth Year,Start Year,End Year,Race,TD: Degree_final,Pre-Fed Career_final,TD: School_final,TD: Major/Field_final
5,Michael S. Barr,Michael S. Barr,1.0,1965.0,2022.0,NaN,W,JD,NaN,Yale University,Law
6,Barr,NaN,1.0,NaN,NaN,NaN,NaN,None/unknown,NaN,NaN,NaN
32,Susan M. Collins,Susan M. Collins,0.0,1959.0,2022.0,NaN,NW,PhD,NaN,Massachusetts Institute of Technology,Economics
33,Collins,NaN,0.0,NaN,NaN,NaN,NaN,PhD,NaN,MIT,NaN
34,Lisa D. Cook,Lisa D. Cook,0.0,1964.0,2022.0,NaN,NW,PhD,NaN,University of California at Berkeley,Economics
35,Cook,NaN,0.0,NaN,NaN,NaN,NaN,PhD,NaN,Berkeley,NaN
73,Austan D. Goolsbee,Austan D. Goolsbee,1.0,1969.0,2023.0,NaN,W,PhD,NaN,Massachusetts Institute of Technology,Economics
74,Goolsbee,NaN,1.0,NaN,NaN,NaN,NaN,PhD,NaN,MIT,NaN
83,Hammack,NaN,0.0,NaN,NaN,NaN,NaN,None/unknown,NaN,NaN,NaN
90,Hendricks,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



===== Missing TD: School_final =====


,Name_final,Name_kaleb,Gender_final,Birth Year,Start Year,End Year,Race,TD: Degree_final,Pre-Fed Career_final,TD: School_final,TD: Major/Field_final
6,Barr,NaN,1.0,NaN,NaN,NaN,NaN,None/unknown,NaN,NaN,NaN
42,William A. Day,William A. Day,1.0,1876.0,1936.0,1945.0,W,None/unknown,Banker,NaN,NaN
48,C. E. Earhart,C. E. Earhart,1.0,1890.0,1946.0,1956.0,W,None/unknown,Banker,NaN,NaN
50,Marriner S. Eccles,Marriner S. Eccles,1.0,1890.0,1934.0,1951.0,W,None/unknown,Banker,NaN,NaN
59,Ralph E. Flanders,Ralph E. Flanders,1.0,1880.0,1944.0,1946.0,W,None/unknown,Businessman,NaN,NaN
60,Matthew J. Fleming,Matthew J. Fleming,1.0,1879.0,1935.0,1944.0,W,None/unknown,Banker,NaN,NaN
67,William B. Geery,William B. Geery,1.0,1867.0,1927.0,1936.0,W,None/unknown,Banker,NaN,NaN
71,Robert R. Gilbert,Robert R. Gilbert,1.0,1888.0,1939.0,1953.0,W,None/unknown,Banker,NaN,NaN
82,George H. Hamilton,George H. Hamilton,1.0,1875.0,1932.0,1941.0,W,None/unknown,Banker,NaN,NaN
83,Hammack,NaN,0.0,NaN,NaN,NaN,NaN,None/unknown,NaN,NaN,NaN



===== Missing TD: Major/Field_final =====


,Name_final,Name_kaleb,Gender_final,Birth Year,Start Year,End Year,Race,TD: Degree_final,Pre-Fed Career_final,TD: School_final,TD: Major/Field_final
0,Carl E. Allen,Carl E. Allen,1.0,NaN,1956.0,1961.0,W,Bachelor's,Banker,Dartmouth College,NaN
6,Barr,NaN,1.0,NaN,NaN,NaN,NaN,None/unknown,NaN,NaN,NaN
21,Broaddus,NaN,1.0,1939.0,NaN,NaN,NaN,PhD,Fed and government,Indiana University,NaN
33,Collins,NaN,0.0,NaN,NaN,NaN,NaN,PhD,NaN,MIT,NaN
35,Cook,NaN,0.0,NaN,NaN,NaN,NaN,PhD,NaN,Berkeley,NaN
41,Chester C. Davis,Chester C. Davis,1.0,1887.0,1941.0,1951.0,W,Bachelor's,Businessman,Grinnell College,NaN
42,William A. Day,William A. Day,1.0,1876.0,1936.0,1945.0,W,None/unknown,Banker,NaN,NaN
44,Ernest G. Draper,Ernest G. Draper,1.0,1885.0,1938.0,1950.0,W,Bachelor's,Assistant Secretary of Commerce,Amherst College,NaN
48,C. E. Earhart,C. E. Earhart,1.0,1890.0,1946.0,1956.0,W,None/unknown,Banker,NaN,NaN
50,Marriner S. Eccles,Marriner S. Eccles,1.0,1890.0,1934.0,1951.0,W,None/unknown,Banker,NaN,NaN


### make sure data is usable in regressions by transforming

In [71]:
# # fine list, just check to be safe but probably fine
# listFine = ["Gender_final","Race","TD: Degree_final"]
# for var in listFine:
#     print(merged_final[var].value_counts())


Gender_final
1.0    197
0.0     27
Name: count, dtype: int64
Race
W     182
NW      9
Name: count, dtype: int64
TD: Degree_final
PhD             98
None/unknown    47
Master's        37
Bachelor's      23
JD              18
Name: count, dtype: int64


In [73]:
# # the problematic variables: Pre-Fed Career_final	TD: School_final	TD: Major/Field_final
# listCheck = ["TD: Major/Field_final","Pre-Fed Career_final"]
# for var in listCheck:
#     print(merged_final[var].value_counts())
#     print(merged_final[var].value_counts().index.tolist())


TD: Major/Field_final
Economics                       87
Law                             31
Business Administration         14
economics                       10
Public Administration            4
law                              3
mba                              3
Political Science                1
Pomology                         1
business                         1
Finance & Business Economics     1
Engineering                      1
economics and finance            1
Electrical Engineering           1
Monetary Economics               1
International Ecomonics          1
Mechanical Engineering           1
political science                1
Business                         1
na                               1
Statistics                       1
Civil & Urban Engineering        1
Management Development           1
Agriculture                      1
Civil Engineering                1
unknown                          1
Finance                          1
Name: count, dtype: int64
['Econo

In [75]:
# def clean_major(x):
#     if pd.isna(x):
#         return np.nan
    
#     x_low = x.lower().strip()
    
#     # treat these as missing
#     if x_low in ["na", "n/a", "unknown", "none", ""]:
#         return np.nan

#     if "finance" in x_low:
#         return "Finance"
#     if "econ" in x_low:
#         return "Economics"
#     if "law" in x_low:
#         return "Law"
#     if "business" in x_low or "mba" in x_low:
#         return "Business"
#     if "engineer" in x_low or "statistics" in x_low:
#         return "Engineering/STEM"
#     if "political" in x_low or "public" in x_low:
#         return "Political Science/Public Policy"
#     if "agri" in x_low or "pomology" in x_low:
#         return "Agriculture/Other"
    
#     return "Other"

# merged_final["Major_clean"] = merged_final["TD: Major/Field_final"].apply(clean_major)
# merged_final["Major_clean"].value_counts()


Major_clean
Economics                          98
Law                                34
Business                           19
Political Science/Public Policy     6
Engineering/STEM                    6
Finance                             3
Agriculture/Other                   2
Other                               2
Name: count, dtype: int64

In [77]:
# def clean_prefed(x):
#     if pd.isna(x):
#         return np.nan
    
#     x_low = x.lower()
    
#     # Academic economist
#     if "professor" in x_low or "academia" in x_low:
#         return "Academic Economist"
    
#     # Professional economist (non-academic)
#     if "economist" in x_low:
#         return "Professional Economist"
    
#     # Banking / finance / Fed
#     if "bank" in x_low or "fed" in x_low or "asset" in x_low or "central bank" in x_low:
#         return "Banking/Finance/Fed"
    
#     # Law
#     if "law" in x_low or "legal" in x_low:
#         return "Law/Legal"
    
#     # Business / industry
#     if "business" in x_low or "industry" in x_low or "invest" in x_low or "equity" in x_low or "real estate" in x_low or "consult" in x_low:
#         return "Business/Industry"
    
#     # Government / public service
#     if "public" in x_low or "politic" in x_low or "secretary" in x_low or "regulator" in x_low:
#         return "Government/Public Service"
    
#     # Military
#     if "military" in x_low:
#         return "Military"
    
#     return "Other"

# merged_final["PreFed_clean"] = merged_final["Pre-Fed Career_final"].apply(clean_prefed)
# merged_final["PreFed_clean"].value_counts()


PreFed_clean
Banking/Finance/Fed          69
Professional Economist       45
Academic Economist           43
Law/Legal                    18
Business/Industry            17
Other                         6
Government/Public Service     5
Military                      4
Name: count, dtype: int64

In [79]:
# # I decided not to clean up the school and focus on how to get salt water/ fresh water list
# saltwater_keywords = [
#     "harvard", "mit", "massachusetts institute", "pennsylvania",
#     "yale", "columbia", "wisconsin", "michigan",
#     "berkeley", "princeton", "stanford", "northwestern"
# ]

# freshwater_keywords = [
#     "chicago", "carnegie mellon", "johns hopkins",
#     "ucla", "los angeles", "iowa state",
#     "ohio state", "brown", "virginia"
# ]

# def classify_school(row):
#     school = str(row["TD: School_final"]).lower()
#     phd = (row["TD: Degree_final"] == "PhD")
#     econ = (row["Major_clean"] == "Economics")
    
#     # Only classify Econ PhDs
#     if phd and econ:
#         if any(s in school for s in saltwater_keywords):
#             return 1
#         if any(s in school for s in freshwater_keywords):
#             return 2
#         return 3  # Econ PhD but not salt/fresh
    
#     # Everyone else → missing
#     return np.nan

# merged_final["School_category"] = merged_final.apply(classify_school, axis=1)
# merged_final["School_category"].value_counts()


School_category
1.0    45
3.0    25
2.0    11
Name: count, dtype: int64

In [81]:
# # rename and we are done!
# merged_final.drop(columns = ["TD: School_final","TD: Major/Field_final","Pre-Fed Career_final"], inplace=True)
# merged_final = merged_final.rename(columns = {"Name_final":"NameMix",
#                                              "Name_kaleb":"FullName",
#                                              "Gender_final":"Gender",
#                                              "TD: Degree_final":"Degree",
#                                              "Major_clean":"Major",
#                                              "School_category":"SaltFresh"})

# merged_final.head()


,NameMix,FullName,Gender,Birth Year,Start Year,End Year,Race,Degree,Major,PreFed_clean,SaltFresh
0,Carl E. Allen,Carl E. Allen,1.0,NaN,1956.0,1961.0,W,Bachelor's,NaN,Banking/Finance/Fed,NaN
1,Wayne D. Angell,Wayne D. Angell,1.0,1930.0,1986.0,1994.0,W,PhD,Economics,Government/Public Service,3.0
2,C. Canby Balderston,C. Canby Balderston,1.0,1897.0,1954.0,1966.0,W,PhD,Law,Academic Economist,NaN
3,John J. Balles,John J. Balles,1.0,1921.0,1972.0,1986.0,W,Master's,Economics,Professional Economist,NaN
4,Thomas I. Barkin,Thomas I. Barkin,1.0,1961.0,2018.0,NaN,W,JD,Law,Business/Industry,NaN


In [83]:
# # a few more cleaning
# merged_final["Start Year"] = merged_final["Start Year"].fillna(-9999)
# merged_final["End Year"] = merged_final["End Year"].fillna(9999)


In [85]:
# merged_final.isnull().sum()

NameMix           0
FullName         33
Gender            0
Birth Year       18
Start Year        0
End Year          0
Race             33
Degree            1
Major            54
PreFed_clean     17
SaltFresh       143
dtype: int64

In [87]:
# merged_final.to_csv("SpeakerData/merged.csv", index = False)


## Merge to main

In [89]:
merged_final = pd.read_csv("SpeakerData/merged.csv")
print(merged_final.shape)
merged_final.head()


(224, 11)


,NameMix,FullName,Gender,Birth Year,Start Year,End Year,Race,Degree,Major,PreFed_clean,SaltFresh
0,Carl E. Allen,Carl E. Allen,1.0,NaN,1956.0,1961.0,W,Bachelor's,NaN,Banking/Finance/Fed,NaN
1,Wayne D. Angell,Wayne D. Angell,1.0,1930.0,1986.0,1994.0,W,PhD,Economics,Government/Public Service,3.0
2,C. Canby Balderston,C. Canby Balderston,1.0,1897.0,1954.0,1966.0,W,PhD,Law,Academic Economist,NaN
3,John J. Balles,John J. Balles,1.0,1921.0,1972.0,1986.0,W,Master's,Economics,Professional Economist,NaN
4,Thomas I. Barkin,Thomas I. Barkin,1.0,1961.0,2018.0,9999.0,W,JD,Law,Business/Industry,NaN


In [90]:
print(df.shape)

print(merged_final.shape)
print(merged_final[merged_final["FullName"].isna()].shape)
print(merged_final[merged_final["FullName"].notna()].shape)

backgroundVar = ['Gender', 'Birth Year', 'Start Year', 'End Year', 
                 'Race', 'Degree', 'Major', 'PreFed_clean', 'SaltFresh']


(7231, 67)
(224, 11)
(33, 11)
(191, 11)


In [92]:
dfmain = df.copy()

def clean_name(x):
    if not isinstance(x, str):
        return ""
    x = x.lower().replace(".", "")
    parts = x.split()
    if len(parts) <= 2:
        return " ".join(parts)
    return parts[0] + " " + parts[-1]

dfmain["speaker_clean"] = dfmain["speaker"].apply(clean_name)
merged_final["FullName_clean"] = merged_final["FullName"].apply(clean_name)

dfmain["speech_year"] = pd.to_datetime(dfmain["date"]).dt.year
dfmain["row_id"] = dfmain.index

temp = dfmain.merge(
    merged_final,
    left_on="speaker_clean",
    right_on="FullName_clean",
    how="left"
)
print(temp.shape)

temp = temp[
    (temp["speech_year"] >= temp["Start Year"]) &
    (temp["speech_year"] <= temp["End Year"])
]
print(temp.shape)

dfmain = dfmain.merge(
    temp.drop_duplicates("row_id")[["row_id"]+backgroundVar],
    on="row_id",
    how="left"
)
print(dfmain.shape)

# checking for weird stuffs
print(dfmain["row_id"].is_unique)
print(dfmain["row_id"].value_counts().max() == 1)

dfmain.head()

(7774, 82)
(6437, 82)
(7231, 79)
True
True


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,row_id,Gender,Birth Year,Start Year,End Year,Race,Degree,Major,PreFed_clean,SaltFresh
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,0,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,1,1.0,NaN,1985.0,1998.0,W,Master's,Engineering/STEM,Banking/Finance/Fed,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,2,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,3,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,4,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN


In [94]:
backup = dfmain.copy()

In [96]:
dfmain = backup.copy()

# now just about the last_name only data
ales_only = merged_final[
    merged_final["FullName"].isna() &
    merged_final["NameMix"].notna()
].copy()
print(ales_only.shape)

ales_only["last_name_mix"] = (
    ales_only["NameMix"]
    .str.lower()
    .str.replace(".", "", regex=False)
    .str.split()
    .str[-1]
)

# unmatched rows in dfmain
unmatched = dfmain[dfmain["Gender"].isna()].copy()

unmatched["last_name"] = (
    unmatched["speaker_clean"]
    .str.split()
    .str[-1]
)
print(unmatched.shape)

# start the merge
fallback = unmatched.merge(
    ales_only,
    left_on="last_name",
    right_on="last_name_mix",
    how="left",
    suffixes=("", "_ales")
)
print(fallback.shape)

dfmain = dfmain.merge(
    fallback[["row_id"]+[c+"_ales" for c in backgroundVar]],
    on="row_id",
    how="left"
)
print(dfmain.shape)

# a few checking
mask = dfmain["SaltFresh"].notna().to_numpy() & dfmain["SaltFresh_ales"].notna().to_numpy()
display(dfmain.loc[mask])

ales_only["last_name_mix"].value_counts().loc[lambda x: x > 1]

dfmain.head()


(33, 12)
(809, 80)
(809, 93)
(7231, 88)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,SaltFresh,Gender_ales,Birth Year_ales,Start Year_ales,End Year_ales,Race_ales,Degree_ales,Major_ales,PreFed_clean_ales,SaltFresh_ales


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,SaltFresh,Gender_ales,Birth Year_ales,Start Year_ales,End Year_ales,Race_ales,Degree_ales,Major_ales,PreFed_clean_ales,SaltFresh_ales
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
backup = dfmain.copy()

In [100]:
dfmain = backup.copy()
# now just fill in and finish it up
ales_cols = [c+"_ales" for c in backgroundVar]
print(dfmain[ales_cols].notna().sum())
print(dfmain[backgroundVar].notna().sum())

for var in backgroundVar:
    dfmain[var] = dfmain[var].fillna(dfmain[var+"_ales"])

dfmain = dfmain.drop(columns=ales_cols)
print(dfmain[backgroundVar].notna().sum())



Gender_ales          384
Birth Year_ales      292
Start Year_ales      384
End Year_ales        384
Race_ales              0
Degree_ales          384
Major_ales           260
PreFed_clean_ales    292
SaltFresh_ales       187
dtype: int64
Gender          6422
Birth Year      6301
Start Year      6422
End Year        6422
Race            6422
Degree          6422
Major           6422
PreFed_clean    6202
SaltFresh       3999
dtype: int64
Gender          6806
Birth Year      6593
Start Year      6806
End Year        6806
Race            6422
Degree          6806
Major           6682
PreFed_clean    6494
SaltFresh       4186
dtype: int64


In [102]:
# final check 
ales_only = merged_final[
    merged_final["FullName"].isna() &
    merged_final["NameMix"].notna()
].copy()

ales_only["last_name_mix"] = (
    ales_only["NameMix"]
    .str.lower()
    .str.replace(".", "", regex=False)
    .str.split()
    .str[-1]
)

unmatched = dfmain[dfmain["Gender"].isna()].copy()

unmatched["last_name"] = unmatched["speaker_clean"].str.split().str[-1]

overlap = set(ales_only["last_name_mix"]) & set(unmatched["last_name"])
overlap

set()

In [104]:
# just listing out the unmatched
unmatched = dfmain[dfmain["Gender"].isna()].copy()

remaining_unmatched_original = unmatched["speaker"].unique().tolist()
print(len(remaining_unmatched_original))

print(unmatched["speaker"].value_counts().to_string())


74
speaker
Tom Barkin                                       96
Simon M. Potter                                  33
Paul M. Connolly                                 21
Ernest T Patrikis                                21
Kevin Stiroh                                     20
Thomas C Baxter                                  16
Michael Held                                     13
James McAndrews                                  12
Roberto Perli                                    11
Michelle Neal                                     9
William L Rutledge                                8
Brian P Sack                                      8
Sarah Dahlgren                                    7
Kartik Athreya                                    7
Patrick K Barron                                  6
Terrence J Checki                                 6
Beverly Hirtle                                    6
Nathaniel Wuerffel                                5
Richard Dzina                                     5
J

In [106]:
# my final results: 
print(dfmain.shape)
print(dfmain.columns.tolist())
dfmain.head()

(7231, 79)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 'UNRATE', 'UNRATE_lag1', 'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'MICH', 'MICH_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1', 'gRGDPB1', 'gRGDPF0', 'gRGDPF1', 'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 'HSTARTB1', 'HSTARTF0', 'HSTARTF1', 'gIPB1', 'gIPF0', 'gIPF1', 'is_monday', 'is_friday', 'speaker_clean', 'speech_year', 'row_id', 'Gender', 'Birth Year', 'Start Year', 'E

,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,row_id,Gender,Birth Year,Start Year,End Year,Race,Degree,Major,PreFed_clean,SaltFresh
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,0,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,1,1.0,NaN,1985.0,1998.0,W,Master's,Engineering/STEM,Banking/Finance/Fed,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,2,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,3,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,4,1.0,1931.0,1983.0,1996.0,W,JD,Law,Law/Legal,NaN


# Financial data


In [110]:
dfPC = dfmain.copy()

## Overview

In [113]:
# just want to know how many data with time
print("Is null")
print(dfPC[dfPC["timeOnly"].isnull()].shape)
print("Is at 00:00:00")
print(dfPC[dfPC["timeOnly"] == "00:00:00"].shape)
print("Can be merged by time")
print(dfPC[~dfPC["timeOnly"].isnull()].shape[0] - dfPC[dfPC["timeOnly"] == "00:00:00"].shape[0])


Is null
(5189, 79)
Is at 00:00:00
(613, 79)
Can be merged by time
1429


In [115]:
dfTime = dfPC[~dfPC["timeOnly"].isnull()]
dfTime = dfTime[dfTime["timeOnly"] != "00:00:00"]
print(dfTime.shape)
print(max(dfTime["date"]))
print(min(dfTime["date"]))


(1429, 79)
2023-08-25 00:00:00
2012-10-22 00:00:00


## Functions

In [153]:
from pathlib import Path

# we need to merge on demand, with daily data for each observations
# First, we need to construct usable financial data
def make_financial_features(df, haveVol = True):
    """
    Takes a raw OHLCV dataframe (with columns:
    ['Open','High','Low','Close','Volume'])
    and returns a dataframe with regression-ready features.
    """

    out = pd.DataFrame(index=df.index)

    # 1. Price levels
    out["price"] = df["Close"]

    # 2. Returns
    out["ret"] = df["Close"].pct_change()
    out["log_ret"] = np.log(df["Close"]).diff()

    # 3. Volatility proxies
    out["abs_ret"] = out["ret"].abs()
    out["ret_sq"] = out["ret"]**2

    # Parkinson high-low volatility estimator
    out["hl_range"] = np.log(df["High"] / df["Low"])

    # 4. Volume (liquidity proxy)
    if haveVol:
        out["volume"] = df["Volume"] 

    # 5. Rolling measures (optional but useful)
    out["roll_vol_5"] = out["ret"].rolling(5).std()
    out["roll_vol_22"] = out["ret"].rolling(22).std()

    out["roll_mean_5"] = out["ret"].rolling(5).mean()
    out["roll_mean_22"] = out["ret"].rolling(22).mean()

    # 6. Abnormal returns (de-meaned)
    out["abn_ret_5"] = out["ret"] - out["roll_mean_5"]
    out["abn_ret_22"] = out["ret"] - out["roll_mean_22"]

    return out

# this merge function is for daily data, thus merge based on date_col
def merge_financial_features(
    speech_df,
    fin_df,
    suffix,
    lags=4,
    leads=1,
    date_col="date"
):
    """
    speech_df: dataframe with one row per speech, must contain a date column
    fin_df: financial features dataframe indexed by date
    suffix: string to append to variable names (e.g., "GLD")
    lags: number of lag days to include
    leads: number of future days to include
    date_col: name of the date column in speech_df
    """

    # Ensure date formats
    speech_df = speech_df.copy()
    speech_df[date_col] = pd.to_datetime(speech_df[date_col])
    fin_df = fin_df.copy()
    fin_df.index = pd.to_datetime(fin_df.index)

    # Add suffix to financial variable names
    fin_df = fin_df.add_suffix(f"_{suffix}")

    # Build a dataframe to merge into speech_df
    merged = speech_df.copy()

    # Current-day merge
    merged = merged.merge(
        fin_df,
        left_on=date_col,
        right_index=True,
        how="left"
    )

    # Add lags
    for k in range(1, lags + 1):
        lagged = fin_df.shift(k)
        lagged = lagged.add_suffix(f"_lag{k}")
        merged = merged.merge(
            lagged,
            left_on=date_col,
            right_index=True,
            how="left"
        )

    # Add leads
    for k in range(1, leads + 1):
        leaded = fin_df.shift(-k)
        leaded = leaded.add_suffix(f"_fut{k}")
        merged = merged.merge(
            leaded,
            left_on=date_col,
            right_index=True,
            how="left"
        )

    return merged

# bloomberg data i got raw form needs to be merge before usable

def bloomberg_mergeOld(filepath):
    # Read all sheets
    xls = pd.ExcelFile(filepath)
    dfs = []

    for sheet in xls.sheet_names:
        df = pd.read_excel(filepath, sheet_name=sheet)

        # Ensure datetime is parsed
        df["DateTime"] = pd.to_datetime(df["Date"])
        dfs.append(df)

    # Combine all sheets
    full = pd.concat(dfs, ignore_index=True)

    # Drop duplicates
    full = full.drop_duplicates(subset=["DateTime"])

    # Sort by time
    full = full.sort_values("DateTime").reset_index(drop=True)
    full["DateTime"] += timedelta(minutes=15)
    full = full.drop(columns = ["Date"])

    return full

def parse_bloomberg_intraday_file(path: str | Path) -> pd.DataFrame:
    # Read raw, no automatic header
    raw = pd.read_excel(path, header=None, engine="openpyxl")
    
    # First row = column names
    cols = raw.iloc[0].tolist()
    raw = raw.iloc[1:].reset_index(drop=True)  # drop header row, keep summary + rest
    raw.columns = cols
    
    # Drop summary row (assume it's the first data row)
    raw = raw.iloc[1:].reset_index(drop=True)
    
    records = []
    current_date = None
    
    for _, row in raw.iterrows():
        time_interval = str(row[cols[0]]).strip()
        
        # Skip empty rows
        if time_interval == "nan" or time_interval == "":
            continue
        
        # Detect date row: e.g. "03JAN2012_00:00:00.000000"
        # Simple rule: contains '_' and no '-'
        if "_" in time_interval and "-" not in time_interval:
            current_date = time_interval.split("_")[0]  # "03JAN2012"
            continue
        
        # Otherwise, this should be an interval row like "09:30 - 09:45"
        if current_date is None:
            # Safety: interval before any date → skip
            continue
        
        # Extract end time (right side of " - ")
        # e.g. "09:30 - 09:45" → "09:45"
        if "-" not in time_interval:
            # malformed, skip
            continue
        
        start_str, end_str = [p.strip() for p in time_interval.split("-")]
        end_time_str = end_str  # "09:45"
        
        # Build full timestamp: date + end time
        # current_date like "03JAN2012"
        dt_str = f"{current_date} {end_time_str}"
        # Adjust format if needed
        ts = pd.to_datetime(dt_str, format="%d%b%Y %H:%M")
        
        # Build record
        rec = {
            "DateTime": ts,
            "Open": row.get("Open"),
            "High": row.get("High"),
            "Low": row.get("Low"),
            "Close": row.get("Close"),
            "Volume": row.get("Volume"),
        }
        records.append(rec)
    
    out = pd.DataFrame(records)
    out = out.sort_values("DateTime").reset_index(drop=True)
    return out

def bloomberg_mergeNew(path) -> pd.DataFrame:
    folder = Path(path)
    all_files = sorted(folder.glob("*.xls*"))
    
    dfs = []
    for f in all_files:
        df = parse_bloomberg_intraday_file(f)
        dfs.append(df)
    
    if not dfs:
        return pd.DataFrame()
    
    out = pd.concat(dfs, ignore_index=True)
    # Remove duplicates across files
    out = out.drop_duplicates(subset=["DateTime"], keep="first")
    
    out = out.sort_values("DateTime").reset_index(drop=True)
    return out


# merge but for intraday data
def merge_financial_features_bloomberg(
    speech_df,
    fin_df,
    suffix,
    window=[15,30,45,60],
    date_col="date",
    time_col="timeOnly"
):

    # Ensure datetime exists in speech_df
    speech_df = speech_df.copy()
    speech_df.loc[speech_df[time_col] == "00:00:00", time_col] = np.nan
    mask_valid_time = speech_df[time_col].notna()

    speech_df["speech_dt"] = pd.NaT  # initialize
    
    speech_df.loc[mask_valid_time, "speech_dt"] = pd.to_datetime(
        speech_df.loc[mask_valid_time, date_col].astype(str)
        + " "
        + speech_df.loc[mask_valid_time, time_col].astype(str),
        errors="coerce",
    )
    speech_df["speech_dt_rounded"] = speech_df["speech_dt"].dt.round("15min")

    # Round to nearest 15-min bar
    speech_df["speech_dt_rounded"] = speech_df["speech_dt"].dt.round("15min")
    
    # Ensure financial df is indexed by datetime
    fin_df = fin_df.rename(columns={"DateTime": "datetime"})
    fin_df = fin_df.sort_values("datetime")
    fin_df = fin_df.set_index("datetime")

    # Compute recommended features
    fin_df["ret"] = fin_df["Close"].pct_change()
    fin_df["abs_ret"] = fin_df["ret"].abs()
    fin_df["range"] = fin_df["High"] - fin_df["Low"]
    # Volume already exists
    has_volume = "Volume" in fin_df.columns

    for w in window:
        print(f"Calculating for window {w} mins")

        # Symmetric window variables
        before_ret  = f"{suffix}_ret_before_{w}"
        after_ret   = f"{suffix}_ret_after_{w}"
        react_ret   = f"{suffix}_ret_reaction_{w}"

        before_abs  = f"{suffix}_absret_before_{w}"
        after_abs   = f"{suffix}_absret_after_{w}"
        react_abs   = f"{suffix}_absret_reaction_{w}"

        before_rng  = f"{suffix}_range_before_{w}"
        after_rng   = f"{suffix}_range_after_{w}"
        react_rng   = f"{suffix}_range_reaction_{w}"

        # Pure post-speech variables
        post_ret  = f"{suffix}_ret_post_{w}"
        post_abs  = f"{suffix}_absret_post_{w}"
        post_rng  = f"{suffix}_range_post_{w}"

        if has_volume:
            before_vol  = f"{suffix}_vol_before_{w}"
            after_vol   = f"{suffix}_vol_after_{w}"
            react_vol   = f"{suffix}_vol_reaction_{w}"
            post_vol  = f"{suffix}_vol_post_{w}"

        # Initialize columns
        for col in [
            before_ret, after_ret, react_ret,
            before_abs, after_abs, react_abs,
            before_rng, after_rng, react_rng,
            post_ret, post_abs, post_rng
        ]:
            speech_df[col] = np.nan

        if has_volume:
            for col in [before_vol, after_vol, react_vol, post_vol]:
                speech_df[col] = np.nan
        
        for i, row in speech_df.iterrows():
            ts = row["speech_dt_rounded"]
            if pd.isna(ts):
                continue

            ts_before = ts - pd.Timedelta(minutes=w)
            ts_after  = ts + pd.Timedelta(minutes=w)

            # Values at t-w, t, t+w
            before_row = fin_df.loc[ts_before] if ts_before in fin_df.index else None
            at_row     = fin_df.loc[ts]        if ts        in fin_df.index else None
            after_row  = fin_df.loc[ts_after]  if ts_after  in fin_df.index else None

            # --- Symmetric window ---
            if before_row is not None:
                speech_df.at[i, before_ret] = before_row["ret"]
                speech_df.at[i, before_abs] = before_row["abs_ret"]
                speech_df.at[i, before_rng] = before_row["range"]
                if has_volume:
                    speech_df.at[i, before_vol] = before_row["Volume"]

            if after_row is not None:
                speech_df.at[i, after_ret] = after_row["ret"]
                speech_df.at[i, after_abs] = after_row["abs_ret"]
                speech_df.at[i, after_rng] = after_row["range"]
                if has_volume:
                    speech_df.at[i, after_vol] = after_row["Volume"]

            if before_row is not None and after_row is not None:
                speech_df.at[i, react_ret] = after_row["ret"] - before_row["ret"]
                speech_df.at[i, react_abs] = after_row["abs_ret"] - before_row["abs_ret"]
                speech_df.at[i, react_rng] = after_row["range"] - before_row["range"]
                if has_volume:
                    speech_df.at[i, react_vol] = after_row["Volume"] - before_row["Volume"]

            if at_row is not None and after_row is not None:
                speech_df.at[i, post_ret] = after_row["ret"] - at_row["ret"]
                speech_df.at[i, post_abs] = after_row["abs_ret"] - at_row["abs_ret"]
                speech_df.at[i, post_rng] = after_row["range"] - at_row["range"]
                if has_volume:
                    speech_df.at[i, post_vol] = after_row["Volume"] - at_row["Volume"]

    cols_to_drop = [col for col in speech_df.columns if "_before_" in col or "_after_" in col]

    speech_df = speech_df.drop(columns=cols_to_drop)
    return speech_df
    

## Testing

In [149]:
var = "SPY"
spy_bb = bloomberg_mergeOld(f"FinData/bloomberg/{var}.xlsx")
spy_bb.head()


,Open,High,Low,Close,DateTime
0,137.8400,138.15,137.800,137.990,2012-04-17 09:45:00
1,137.9900,138.08,137.860,137.945,2012-04-17 10:00:00
2,137.9550,138.13,137.698,138.090,2012-04-17 10:15:00
3,138.0995,138.40,138.060,138.345,2012-04-17 10:30:00
4,138.3510,138.47,138.300,138.430,2012-04-17 10:45:00


In [49]:
root = "FinData/bloombergVolume/"
var = "SPY"
spy = bloomberg_mergeNew(root + var)
spy.head()


,DateTime,Open,High,Low,Close,Volume
0,2012-01-03 09:45:00,127.76,128.12,127.73,128.060,11816040
1,2012-01-03 10:00:00,128.07,128.08,127.84,127.950,8050847
2,2012-01-03 10:15:00,127.94,128.34,127.83,128.265,11864613
3,2012-01-03 10:30:00,128.26,128.38,128.18,128.190,6713132
4,2012-01-03 10:45:00,128.19,128.27,128.12,128.170,3731413


## Applying

In [121]:
backup = dfPC.copy()


In [ ]:
# download the financiala daily data from yahoo finance
try:
    import yfinance as yf
except:
    !pip install yfinance
    import yfinance as yf

tickers = {
    "SPY": "SPY",
    "SPX_index": "^GSPC",
    "ES_futures": "ES=F",
    "SHY":"SHY",
    "IEF":"IEF",
    "TLT":"TLT",
    "UUP": "UUP",
    "VXX": "VXX",
    "VIX":"^VIX",
    "GLD": "GLD"
}

start_date = "1986-01-01"
end_date = "2023-12-31"

data_dict = {}

for name, ticker in tickers.items():
    df = yf.download(ticker, start=start_date, end=end_date)
    data_dict[name] = df
    df.to_csv(f"FinData/{name}Daily.csv")   # saves each DataFrame



In [123]:
dfPC = backup.copy()
# Daily data
finDailyList = ["SPY","SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]

first = True
for finName in finDailyList:
    dffindum = pd.read_csv(f"FinData/{finName}Daily.csv")
    dffindum = dffindum.iloc[2:].reset_index(drop=True)
    dffindum = dffindum.rename(columns={dffindum.columns[0]: "Date"})
    dffindum["Date"] = pd.to_datetime(dffindum["Date"])
    dffindum = dffindum.set_index("Date")
    dffindum = dffindum.apply(pd.to_numeric, errors="coerce")

    features = make_financial_features(dffindum)

    if first:
        merged_df = merge_financial_features(
        speech_df=dfPC,
        fin_df=features,
        suffix=finName,
        lags=4,
        leads=1,
        date_col="date"
        )
        first = False
    else:
        merged_df = merge_financial_features(
        speech_df=merged_df,
        fin_df=features,
        suffix=finName,
        lags=4,
        leads=1,
        date_col="date"
        )

print(merged_df.shape)
merged_df.head()


(7231, 859)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,abs_ret_GLD_fut1,ret_sq_GLD_fut1,hl_range_GLD_fut1,volume_GLD_fut1,roll_vol_5_GLD_fut1,roll_vol_22_GLD_fut1,roll_mean_5_GLD_fut1,roll_mean_22_GLD_fut1,abn_ret_5_GLD_fut1,abn_ret_22_GLD_fut1
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [124]:
backup = merged_df.copy()

In [157]:
# intraday, i want to treat those at 00:00:00 to be missing value! 
# this is included in the function
merged_df = backup.copy()
print(merged_df.shape)

# only VIX is from old data
var = "VIX"
spy_bb = bloomberg_mergeOld(f"FinData/bloomberg/{var}.xlsx")
merged_df2 = merge_financial_features_bloomberg(
    merged_df,
    spy_bb,
    suffix = var+"_BB",
    window=[15,30,45,60],
    date_col="date",
    time_col="timeOnly"
)
print(merged_df2.shape)

# The rest is from new data
listBB = ["SPY","SHY","IEF","TLT"]
root = "FinData/bloombergVolume/"
for var in listBB:
    spy_bb = bloomberg_mergeNew(root + var)
    merged_df2 = merge_financial_features_bloomberg(
        merged_df2,
        spy_bb,
        suffix = var+"_BB",
        window=[15,30,45,60],
        date_col="date",
        time_col="timeOnly"
    )
    print(merged_df2.shape)

print(merged_df2.shape)
merged_df2.head()


(7231, 859)
Calculating for window 15 mins
Calculating for window 30 mins
Calculating for window 45 mins
Calculating for window 60 mins
(7231, 885)
Calculating for window 15 mins
Calculating for window 30 mins
Calculating for window 45 mins
Calculating for window 60 mins
(7231, 917)
Calculating for window 15 mins
Calculating for window 30 mins
Calculating for window 45 mins
Calculating for window 60 mins
(7231, 949)
Calculating for window 15 mins
Calculating for window 30 mins
Calculating for window 45 mins
Calculating for window 60 mins
(7231, 981)
Calculating for window 15 mins
Calculating for window 30 mins
Calculating for window 45 mins
Calculating for window 60 mins
(7231, 1013)
(7231, 1013)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,TLT_BB_vol_reaction_45,TLT_BB_vol_post_45,TLT_BB_ret_reaction_60,TLT_BB_absret_reaction_60,TLT_BB_range_reaction_60,TLT_BB_ret_post_60,TLT_BB_absret_post_60,TLT_BB_range_post_60,TLT_BB_vol_reaction_60,TLT_BB_vol_post_60
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [160]:
# checking: just drop nan for the timeOnly and check 1-2 instances
dfcheck = merged_df2.dropna(subset = ["timeOnly"])
dfcheck[["timeOnly","SPY_BB_ret_post_60","SPY_BB_ret_reaction_60"]].head()


,timeOnly,SPY_BB_ret_post_60,SPY_BB_ret_reaction_60
4514,13:30:00,0.002172,0.000421
4515,20:00:00,NaN,NaN
4516,17:00:00,NaN,NaN
4517,12:30:00,-0.000420,0.000035
4520,12:00:00,-0.000500,-0.000886


In [164]:
# a quick info to the missing
print("The total observation with timestamp = "+ str(dfcheck.shape[0]))
for w in [15,45,60]:
    print("For window = " +str(w))
    cols = [
    "SPY_BB_ret_post_"+str(w),
    "SPY_BB_ret_reaction_"+str(w)
]
    print(dfcheck[cols].isna().sum())



The total observation with timestamp = 1429
For window = 15
SPY_BB_ret_post_15        593
SPY_BB_ret_reaction_15    613
dtype: int64
For window = 45
SPY_BB_ret_post_45        632
SPY_BB_ret_reaction_45    777
dtype: int64
For window = 60
SPY_BB_ret_post_60        642
SPY_BB_ret_reaction_60    810
dtype: int64


## Export complete regression data

In [166]:
print(merged_df2.shape)
merged_df2.head()


(7231, 1013)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,TLT_BB_vol_reaction_45,TLT_BB_vol_post_45,TLT_BB_ret_reaction_60,TLT_BB_absret_reaction_60,TLT_BB_range_reaction_60,TLT_BB_ret_post_60,TLT_BB_absret_post_60,TLT_BB_range_post_60,TLT_BB_vol_reaction_60,TLT_BB_vol_post_60
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [168]:
merged_df2.isnull().sum()


District                     0
MultSpeakers              6606
VideoForm                    0
Article                      0
title                        0
                          ... 
TLT_BB_ret_post_60        6444
TLT_BB_absret_post_60     6444
TLT_BB_range_post_60      6444
TLT_BB_vol_reaction_60    6612
TLT_BB_vol_post_60        6444
Length: 1013, dtype: int64

In [68]:
merged_df2.to_csv('FedSpeechesRegression.csv', date_format='%Y-%m-%d %H:%M:%S', index = False)


C:\Users\Lynn\anaconda3\Lib\site-packages\executing\executing.py:713: DeprecationWarning:ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
C:\Users\Lynn\anaconda3\Lib\ast.py:587: DeprecationWarning:Attribute s is deprecated and will be removed in Python 3.14; use value instead
C:\Users\Lynn\anaconda3\Lib\site-packages\executing\executing.py:713: DeprecationWarning:ast.Str is deprecated and will be removed in Python 3.14; use ast.Constant instead
C:\Users\Lynn\anaconda3\Lib\ast.py:587: DeprecationWarning:Attribute s is deprecated and will be removed in Python 3.14; use value instead


NameError: name 'merged_df2' is not defined

In [61]:
# dfmain.to_csv('FedSpeechesRegression.csv', date_format='%Y-%m-%d %H:%M:%S', index = False)


# Regressions

## Import data

In [39]:
dfmain = pd.read_csv("FedSpeechesRegression.csv", parse_dates = ['date'])
dfmain = dfmain.drop_duplicates(subset=['speaker', 'date','title'])
print(dfmain.shape)
dfmain.head()


(7228, 1013)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,TLT_BB_vol_reaction_45,TLT_BB_vol_post_45,TLT_BB_ret_reaction_60,TLT_BB_absret_reaction_60,TLT_BB_range_reaction_60,TLT_BB_ret_post_60,TLT_BB_absret_post_60,TLT_BB_range_post_60,TLT_BB_vol_reaction_60,TLT_BB_vol_post_60
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
listDistrict = ['NewYork', 'Atlanta', 'SanFrancisco', 'Chicago', 'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston', 'Minneapolis', 'Kansas']

print(dfmain[(dfmain['District'].isin(listDistrict)) & (dfmain['IsBpres']== True)].shape)
print(dfmain[(dfmain['District'].isin(listDistrict))].shape)
print(dfmain[(dfmain['District'].isin(['BoardGovernors']))].shape)



(4002, 1013)
(4351, 1013)
(2877, 1013)


In [43]:
# About topics
df = pd.read_csv("FedSpeechesTopic.csv", parse_dates = ['date'])
df = df.dropna(subset=['date'])
df = df.drop_duplicates(subset=['speaker', 'date','title'])
df = df[['speaker', 'date','title']+["Topic "+str(i) for i in range(20)]]
print(df.shape)
df.head()


(7228, 23)


,speaker,date,title,Topic 0,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,...,Topic 10,Topic 11,Topic 12,Topic 13,Topic 14,Topic 15,Topic 16,Topic 17,Topic 18,Topic 19
0,Susan M. Collins,2025-06-25,Perspectives on the Economy from Susan M. Collins,0.000094,0.000094,0.021017,0.000094,0.000094,0.000094,0.090077,...,0.054367,0.000094,0.000094,0.000094,0.000094,0.000094,0.000094,0.157189,0.174159,0.000094
1,Susan M. Collins,2024-09-30,Welcoming Remarks at the forum on “Meeting the...,0.000171,0.000171,0.211306,0.000171,0.000171,0.139555,0.049671,...,0.000171,0.000171,0.000171,0.061778,0.000171,0.000171,0.000171,0.000171,0.512725,0.022568
2,Susan M. Collins,2024-06-18,A Partnership for Progress,0.000036,0.013062,0.008668,0.000036,0.000036,0.034661,0.017051,...,0.007728,0.000036,0.000036,0.088362,0.023008,0.000036,0.000036,0.000036,0.567451,0.000036
3,Susan M. Collins,2024-05-08,Reflections on Uncertainty and Patience in Mon...,0.000028,0.000028,0.000028,0.000028,0.000028,0.006715,0.029123,...,0.024423,0.000028,0.000028,0.044043,0.054702,0.014881,0.000028,0.145100,0.094085,0.000028
4,Susan M. Collins,2024-04-16,Remarks for the National Association of Corpor...,0.006756,0.000010,0.000010,0.000010,0.000010,0.000010,0.031587,...,0.000029,0.000010,0.000010,0.554396,0.055425,0.000010,0.000010,0.040715,0.097247,0.008184


In [44]:
group_map = {
    # Group A — Monetary Policy & Inflation
    5:  "monetary_policy_inflation",
    7:  "monetary_policy_inflation",
    8:  "monetary_policy_inflation",
    9:  "monetary_policy_inflation",
    17: "monetary_policy_inflation",

    # Group B — Financial Stability & Banking Regulation
    0:  "financial_stability_regulation",
    1:  "financial_stability_regulation",
    3:  "financial_stability_regulation",
    16: "financial_stability_regulation",
    19: "financial_stability_regulation",

    # Group C — Real Economic Activity & Productivity
    6:  "real_economy_productivity",
    10: "real_economy_productivity",
    11: "real_economy_productivity",

    # Group D — Community Development & Inequality
    2:  "community_development",
    18: "community_development",

    # Group E — International Economics & Global Spillovers
    12: "international_economics",

    # Group F — Federal Reserve Operations & Payment Systems
    14: "fed_operations_payments",
    15: "fed_operations_payments",
}

macro_groups = set(group_map.values())
for g in macro_groups:
    df[g] = 0.0
    
for topic_idx, group_name in group_map.items():
    col = f"Topic {topic_idx}"
    if col in df.columns:
        df[group_name] += df[col]
    else:
        print(f"Warning: {col} not found in df_topics")

df = df.drop(columns=[f"Topic {i}" for i in range(20)])

df.head()



,speaker,date,title,real_economy_productivity,financial_stability_regulation,monetary_policy_inflation,fed_operations_payments,community_development,international_economics
0,Susan M. Collins,2025-06-25,Perspectives on the Economy from Susan M. Collins,0.144538,0.000472,0.659343,0.000189,0.195175,0.000094
1,Susan M. Collins,2024-09-30,Welcoming Remarks at the forum on “Meeting the...,0.050014,0.023253,0.140240,0.000342,0.724030,0.000171
2,Susan M. Collins,2024-06-18,A Partnership for Progress,0.024814,0.013204,0.274385,0.023043,0.576119,0.000036
3,Susan M. Collins,2024-05-08,Reflections on Uncertainty and Patience in Mon...,0.053574,0.000141,0.738489,0.069583,0.094113,0.000028
4,Susan M. Collins,2024-04-16,Remarks for the National Association of Corpor...,0.031627,0.014971,0.246293,0.055435,0.097257,0.000010


In [46]:
# merge to main just the topic
dfmain = dfmain.merge(df, on=['speaker', 'date','title'], how='left')
print(dfmain.shape)
dfmain.head()


(7228, 1019)


,District,MultSpeakers,VideoForm,Article,title,speaker,location,date,positiveFin,neutralFin,...,TLT_BB_absret_post_60,TLT_BB_range_post_60,TLT_BB_vol_reaction_60,TLT_BB_vol_post_60,real_economy_productivity,financial_stability_regulation,monetary_policy_inflation,fed_operations_payments,community_development,international_economics
0,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-06,0.497780,0.208901,...,NaN,NaN,NaN,NaN,0.439026,0.046963,0.016287,0.000059,0.000059,0.464426
1,StLouis,NaN,False,False,"1986: What We Know, What We Don't Know, and Wh...",Thomas C. Melzer,Federal Reserve Bank of St Louis,1986-01-08,0.338883,0.492801,...,NaN,NaN,NaN,NaN,0.219863,0.008706,0.522837,0.000124,0.000124,0.080379
2,Atlanta,NaN,False,False,The United States in the World Economy : Remar...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.455167,0.308791,...,NaN,NaN,NaN,NaN,0.234458,0.011162,0.042240,0.004594,0.000062,0.707422
3,Atlanta,NaN,False,False,The Economic Outlook for 1986 : Remarks to the...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-15,0.526953,0.199551,...,NaN,NaN,NaN,NaN,0.474519,0.044495,0.000141,0.000057,0.000057,0.468289
4,Atlanta,NaN,False,False,International Currency Changes : Remarks to th...,Robert P. Forrestal,Federal Reserve Bank of Atlanta,1986-01-16,0.306015,0.358471,...,NaN,NaN,NaN,NaN,0.051767,0.013328,0.129503,0.009902,0.000093,0.795313


In [50]:
# if I want to eliminate the differences in format
# excluding video, mult, article: cannot use == False because there are missing values in Mult
dfmain = dfmain[(dfmain['MultSpeakers'] != True) & (dfmain['VideoForm'] != True) & (dfmain['Article'] != True)]
print(len(dfmain["speaker"].value_counts().index))
print(dfmain["District"].value_counts().index)

# # Removing tops outliers for info
# n = 2
# top_n_indices = dfmain.nlargest(n, 'info').index
# dfmain = dfmain.drop(index=top_n_indices)

# # also removing tops outliers for read
# n = 2
# top_n_indices = dfmain.nlargest(n, 'read').index
# dfmain = dfmain.drop(index=top_n_indices)

backup = dfmain.copy()



149
Index(['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
       'StLouis', 'Cleveland', 'Philadelphia', 'Richmond', 'Dallas', 'Boston',
       'Kansas', 'Minneapolis'],
      dtype='object', name='District')


In [51]:
bankList = [ 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                         'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                         'Kansas', 'Minneapolis']

## Functions

In [59]:
import statsmodels.api as sm 
from statsmodels.iolib.summary2 import summary_col
from scipy import stats

def OLSFin(df, finName, finVar, outputList, controlList, Ftest = True, sepNull = False, lagn = 4, feffect = "individual"):
    '''
    feffect: role - using IsBpres; institution - using District; 
    individual - using speaker; none - not using any
    '''
    resultReg = []
    inputlist = [finVar+"_"+finName+"_lag"+str(i+1) for i in range(lagn)]+controlList
    if  feffect == "role":
        df["IsBpres"] = df["IsBpres"].astype(int)
        exog = sm.add_constant(df[inputlist+["IsBpres"]])
        showvar = inputlist+["IsBpres"]
    elif feffect == "institution":
        desired_order = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                         'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                         'Kansas', 'Minneapolis']
        df["District"] = pd.Categorical(df["District"], categories=desired_order)
        dummies = pd.get_dummies(df["District"], prefix="inst", drop_first=True)
        dummies = dummies.astype(float)
        exog = sm.add_constant(pd.concat([df[inputlist], dummies], axis=1))
        showvar = inputlist + ["inst_"+name for name in desired_order[1:]]
    elif feffect == "individual":
        dummies = pd.get_dummies(df["speaker"], prefix="speaker", drop_first=True)
        dummies = dummies.astype(float)
        exog = sm.add_constant(pd.concat([df[inputlist], dummies], axis=1))
        showvar = inputlist
    elif feffect == "none":
        exog =  sm.add_constant(df[inputlist])
        showvar = inputlist
    else:
        print("category not recognized, choosing between role, institution, individual")
        return None
    
    for output in outputList:
        # dfdum = dfdum.replace([np.inf, -np.inf], np.nan)
        # print(exog.dtypes)
        reg = sm.OLS(df[output], exog, missing='drop').fit(cov_type="HAC", cov_kwds={"kernel": "bartlett", "maxlags": 3}, use_t=True)
        resultReg.append(reg)

    if not Ftest:
        # https://github.com/statsmodels/statsmodels/blob/main/statsmodels/iolib/summary2.py#L472
        resultTable = summary_col(resultReg,stars=True,float_format='%0.2f',
                                  model_names= outputList,
                                  regressor_order = showvar,
                                  drop_omitted = True,
                                  include_r2 = False,
                                  info_dict={'R2adj': lambda x: "{}".format(round(x.rsquared_adj,3)),
                                      'N': lambda x: "{0:d}".format(int(x.nobs))})
        # resultTable.tables[0].to_excel('results/{}.xlsx'.format(finName+finVar))
        return resultTable
    elif sepNull:
        inputlist = inputlist[:4]
        results = pd.DataFrame(index=[finVar+"_"+finName], columns=outputList)
        for (reg, output) in zip(resultReg, outputList):
            # Restriction matrix: each coefficient equals zero
            beta_hat = reg.params[inputlist].values
            V = reg.cov_params().loc[inputlist, inputlist].values
            R = np.eye(len(inputlist))   # 4x4 identity
            
            theta = R @ beta_hat                 # shape (4,)
            var_theta = R @ V @ R.T              # shape (4,4)
            
            # Wald statistic
            try:
                W = theta.T @ np.linalg.inv(var_theta) @ theta
            except:
                print(inputlist)
                results.loc[finVar+"_"+finName, output] = "error"
                continue
            dfree = len(inputlist)                  # 4 restrictions
            
            # Convert to F-statistic (OLS convention)
            F_stat = W / dfree
            p_value = 1 - stats.f.cdf(F_stat, dfree, reg.df_resid)

            if p_value < 0.01:
                results.loc[finVar+"_"+finName, output] = "***"
            elif p_value < 0.05:
                results.loc[finVar+"_"+finName, output] = "**"
            elif p_value < 0.1:
                results.loc[finVar+"_"+finName, output] = "*"
            else:
                results.loc[finVar+"_"+finName, output] = 0
        return results
    else:
        inputlist = inputlist[:4]
        results = pd.DataFrame(index=[finVar+"_"+finName], columns=outputList)
        for (reg, output) in zip(resultReg, outputList):
            # i will be testing the first lagn variable coefficient jointly sum to 0
            # against the alternative that they are >0 and <0
            beta_hat = reg.params[inputlist].values
            V = reg.cov_params().loc[inputlist, inputlist].values
        
            L = np.ones(len(inputlist))  # weights for sum
            theta = L @ beta_hat
            var_theta = L @ V @ L.T
            se_theta = np.sqrt(var_theta)
        
            t_stat = theta / se_theta
            df_resid = reg.df_resid
        
            # One-sided p-values
            p_pos = 1 - stats.t.cdf(t_stat, df_resid)   # H1: theta > 0
            p_neg = stats.t.cdf(t_stat, df_resid)       # H1: theta < 0
            
            # Decision rule
            if theta > 0:
                if p_pos < 0.01:
                    results.loc[finVar+"_"+finName, output] = "+++"
                elif p_pos < 0.05:
                    results.loc[finVar+"_"+finName, output] = "++"
                elif p_pos < 0.1:
                    results.loc[finVar+"_"+finName, output] = "+"
                else:
                    results.loc[finVar+"_"+finName, output] = 0
            elif theta < 0:
                if p_neg < 0.01:
                    results.loc[finVar+"_"+finName, output] = "---"
                elif p_neg < 0.05:
                    results.loc[finVar+"_"+finName, output] = "--"
                elif p_neg < 0.1:
                    results.loc[finVar+"_"+finName, output] = "-"
                else:
                    results.loc[finVar+"_"+finName, output] = 0
            else:
                results.loc[finVar+"_"+finName, output] = 0

        return results


In [60]:
def OLSFin2(df, finNameList, finVarList, inputList, Ftest = True, sepNull = False):
    results = pd.DataFrame(index=finNameList, columns=finVarList)
    testList = inputList[:6]
    for finVar in finVarList:
        resultReg = []
        model_names = []
        for finName in finNameList:
            # dfdum = dfdum.replace([np.inf, -np.inf], np.nan)
            # print(exog.dtypes)
            output = finVar+"_"+finName+"_fut"+str(1)
            reg = sm.OLS(df[output], df[inputList], missing='drop').fit(cov_type="HAC", cov_kwds={"kernel": "bartlett", "maxlags": 3}, use_t=True)
            resultReg.append(reg)
            model_names.append(output)
        if not Ftest:
            # https://github.com/statsmodels/statsmodels/blob/main/statsmodels/iolib/summary2.py#L472
            resultTable = summary_col(resultReg,stars=True,float_format='%0.2f',
                                      model_names= model_names,
                                      regressor_order = inputList,
                                      drop_omitted = True,
                                      include_r2 = False,
                                      info_dict={'R2adj': lambda x: "{}".format(round(x.rsquared_adj,3)),
                                          'N': lambda x: "{0:d}".format(int(x.nobs))})
            resultTable.tables[0].to_excel('results/reg2/{}.xlsx'.format(finVar))
            print(resultTable)
        elif sepNull:
            for (reg,finName) in zip(resultReg,finNameList):
                # Restriction matrix: each coefficient equals zero
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
                R = np.eye(len(testList))   # 4x4 identity
                
                theta = R @ beta_hat                 # shape (4,)
                var_theta = R @ V @ R.T              # shape (4,4)
                
                # Wald statistic
                try:
                    W = theta.T @ np.linalg.inv(var_theta) @ theta
                except:
                    print(testList)
                    results.loc[finName, finVar] = "error"
                    continue
                dfree = len(testList)                  # 4 restrictions
                
                # Convert to F-statistic (OLS convention)
                F_stat = W / dfree
                p_value = 1 - stats.f.cdf(F_stat, dfree, reg.df_resid)
    
                if p_value < 0.01:
                    results.loc[finName, finVar] = "***"
                elif p_value < 0.05:
                    results.loc[finName, finVar] = "**"
                elif p_value < 0.1:
                    results.loc[finName, finVar] = "*"
                else:
                    results.loc[finName, finVar] = 0
        else:
            for (reg,finName) in zip(resultReg,finNameList):
                # i will be testing the first lagn variable coefficient jointly sum to 0
                # against the alternative that they are >0 and <0
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
            
                L = np.ones(len(testList))  # weights for sum
                theta = L @ beta_hat
                var_theta = L @ V @ L.T
                se_theta = np.sqrt(var_theta)
            
                t_stat = theta / se_theta
                df_resid = reg.df_resid
            
                # One-sided p-values
                p_pos = 1 - stats.t.cdf(t_stat, df_resid)   # H1: theta > 0
                p_neg = stats.t.cdf(t_stat, df_resid)       # H1: theta < 0
                
                # Decision rule
                if theta > 0:
                    if p_pos < 0.01:
                        results.loc[finName, finVar] = "+++"
                    elif p_pos < 0.05:
                        results.loc[finName, finVar] = "++"
                    elif p_pos < 0.1:
                        results.loc[finName, finVar] = "+"
                    else:
                        results.loc[finName, finVar] = 0
                elif theta < 0:
                    if p_neg < 0.01:
                        results.loc[finName, finVar] = "---"
                    elif p_neg < 0.05:
                        results.loc[finName, finVar] = "--"
                    elif p_neg < 0.1:
                        results.loc[finName, finVar] = "-"
                    else:
                        results.loc[finName, finVar] = 0
                else:
                    results.loc[finName, finVar] = 0
    return results

In [61]:
def OLSFin3(df, finNameList, finVarList, inputList, windowList, windowAsym = True, Ftest = True, sepNull = False):
    finVarFullList = [i+str(j) for i in finVarList for j in windowList]
    results = pd.DataFrame(index=finNameList, columns=finVarFullList)
    testList = inputList[:6]
    for finName in finNameList:
        resultReg = []
        model_names = []
        for finVar in finVarList:
            for win in windowList:
                # dfdum = dfdum.replace([np.inf, -np.inf], np.nan)
                # print(exog.dtypes)
                if windowAsym:
                    df["output"] = df[finName+"_"+finVar+"_post_"+str(win)]
                else:
                    df["output"] = df[finName+"_"+finVar+"_reaction_"+str(win)]
                reg = sm.OLS(df["output"], df[inputList], missing='drop').fit(cov_type="HAC", cov_kwds={"kernel": "bartlett", "maxlags": 3}, use_t=True)
                resultReg.append(reg)
                model_names.append(finName+"_"+finVar+str(win))
        if not Ftest:
            # https://github.com/statsmodels/statsmodels/blob/main/statsmodels/iolib/summary2.py#L472
            resultTable = summary_col(resultReg,stars=True,float_format='%0.2f',
                                      model_names= model_names,
                                      regressor_order = inputList,
                                      drop_omitted = True,
                                      include_r2 = False,
                                      info_dict={'R2adj': lambda x: "{}".format(round(x.rsquared_adj,3)),
                                          'N': lambda x: "{0:d}".format(int(x.nobs))})
            resultTable.tables[0].to_excel('results/reg3/{}.xlsx'.format(finName))
            print(resultTable)
        elif sepNull:
            for (reg,finVarFull) in zip(resultReg,finVarFullList):
                # Restriction matrix: each coefficient equals zero
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
                R = np.eye(len(testList))   # 4x4 identity
                
                theta = R @ beta_hat                 # shape (4,)
                var_theta = R @ V @ R.T              # shape (4,4)
                
                # Wald statistic
                try:
                    W = theta.T @ np.linalg.inv(var_theta) @ theta
                except:
                    print(testList)
                    results.loc[finName, finVarFull] = "error"
                    continue
                dfree = len(testList)                  # 4 restrictions
                
                # Convert to F-statistic (OLS convention)
                F_stat = W / dfree
                p_value = 1 - stats.f.cdf(F_stat, dfree, reg.df_resid)
    
                if p_value < 0.01:
                    results.loc[finName, finVarFull] = "***"
                elif p_value < 0.05:
                    results.loc[finName, finVarFull] = "**"
                elif p_value < 0.1:
                    results.loc[finName, finVarFull] = "*"
                else:
                    results.loc[finName, finVarFull] = 0
        else:
            for (reg,finVarFull) in zip(resultReg,finVarFullList):
                # i will be testing the first lagn variable coefficient jointly sum to 0
                # against the alternative that they are >0 and <0
                beta_hat = reg.params[testList].values
                V = reg.cov_params().loc[testList, testList].values
            
                L = np.ones(len(testList))  # weights for sum
                theta = L @ beta_hat
                var_theta = L @ V @ L.T
                se_theta = np.sqrt(var_theta)
            
                t_stat = theta / se_theta
                df_resid = reg.df_resid
            
                # One-sided p-values
                p_pos = 1 - stats.t.cdf(t_stat, df_resid)   # H1: theta > 0
                p_neg = stats.t.cdf(t_stat, df_resid)       # H1: theta < 0
                
                # Decision rule
                if theta > 0:
                    if p_pos < 0.01:
                        results.loc[finName, finVarFull] = "+++"
                    elif p_pos < 0.05:
                        results.loc[finName, finVarFull] = "++"
                    elif p_pos < 0.1:
                        results.loc[finName, finVarFull] = "+"
                    else:
                        results.loc[finName, finVarFull] = 0
                elif theta < 0:
                    if p_neg < 0.01:
                        results.loc[finName, finVarFull] = "---"
                    elif p_neg < 0.05:
                        results.loc[finName, finVarFull] = "--"
                    elif p_neg < 0.1:
                        results.loc[finName, finVarFull] = "-"
                    else:
                        results.loc[finName, finVarFull] = 0
                else:
                    results.loc[finName, finVarFull] = 0
    return results

In [65]:
def df_to_latex(df):
    def fmt_col(col):
        parts = str(col).replace("_", " ").split()
        s = "\\makecell{" + " \\\\ ".join(parts) + "}"
        # escape braces for Python's .format, but LaTeX will still see single braces
        s = s.replace("{", "{{").replace("}", "}}")
        return s

    latex = df.to_latex(
        index=True,
        escape=False,
        column_format="l" + "r" * len(df.columns),
        header=[fmt_col(c) for c in df.columns],
        float_format="%.3f",
    )
    return latex

def df_to_latex_intra(df):
    header = """
    \\begin{tabular}{lcccccccccccccccc}
    \\toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\\\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\\\
    \midrule
    """
    footer = r"""
    \bottomrule
    \end{tabular}
    """
    
    latex = df.to_latex(
    index=True,
    escape=False,
    column_format="l" + "r" * len(df.columns),
    header=False,   # <-- IMPORTANT
    float_format="%.3f",
    )
    
    # Remove the \begin{tabular} ... \end{tabular} wrapper
    body = latex.split("\\midrule")[1]          # everything after \midrule
    body = body.rsplit("\\bottomrule", 1)[0]    # everything before \bottomrule
    full_table = header + body + footer
    return full_table



## Simple daily regressions (Testing)

In [40]:
dfmain = backup.copy()

speechVarList = ['abstract','info','read','disunity','strain','strainUpper']
finTypeList = ["SPY","SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
# finVarList = ["price","ret","log_ret","abs_ret","ret_sq","hl_range","volume",
#               "roll_mean_5","roll_mean_22","roll_vol_5","roll_vol_22", "abn_ret_5","abn_ret_22"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]
controlList = []


### First group of regressions

In [42]:
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, lagn = 4, feffect = "individual")
        model.tables[0].to_excel("results/test/reg1/finOnComplexityDetail{}.xlsx".format("individual"+finType+finVar+str(4)))
        print(model)



             abstract  info   read  disunity strain strainUpper
---------------------------------------------------------------
ret_SPY_lag1 0.39     0.62   1.95   -0.06    -0.07  -0.61      
             (1.92)   (0.80) (1.68) (0.90)   (0.64) (0.79)     
ret_SPY_lag2 -2.01    -0.55  1.07   0.39     0.94   -0.37      
             (2.07)   (0.93) (1.72) (0.94)   (0.71) (0.82)     
ret_SPY_lag3 -2.66    0.21   -1.32  -1.01    -0.49  -0.38      
             (1.86)   (0.84) (1.79) (0.92)   (0.67) (0.74)     
ret_SPY_lag4 -0.01    -0.07  0.43   -0.16    -0.33  -0.28      
             (2.03)   (1.03) (1.88) (0.95)   (0.70) (0.82)     
N            5535     5535   5535   5535     5535   5535       
R2adj        0.319    0.194  0.231  0.449    0.46   0.313      
Standard errors in parentheses.
* p<.1, ** p<.05, ***p<.01

                   abstract  info   read  disunity strain strainUpper
---------------------------------------------------------------------
ret_SPX_index_lag1 -0.18    0.0

In [56]:
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        model = OLSFin(dfmain, finType, finVar, outputList,controlList,  Ftest = False, lagn = 4, feffect = "role")
        model.tables[0].to_excel("results/test/reg1/finOnComplexityDetail{}.xlsx".format("role"+finType+finVar+str(4)))
        print(model)



             abstract   info     read   disunity  strain  strainUpper
---------------------------------------------------------------------
ret_SPY_lag1 -1.43    0.44     1.00     -0.55    0.10     -0.43      
             (2.34)   (0.85)   (1.70)   (1.19)   (0.89)   (0.94)     
ret_SPY_lag2 -2.61    -0.46    -0.79    -0.85    -0.28    -0.57      
             (2.46)   (1.03)   (1.67)   (1.15)   (0.90)   (1.00)     
ret_SPY_lag3 -2.30    0.40     -1.46    -1.35    -0.21    0.05       
             (2.21)   (0.92)   (1.85)   (1.10)   (0.83)   (0.89)     
ret_SPY_lag4 -1.36    -0.41    -0.96    -1.02    -0.79    -0.45      
             (2.30)   (1.04)   (1.79)   (1.13)   (0.86)   (0.96)     
IsBpres      -1.44*** -0.20*** -1.41*** -1.02*** -0.77*** -0.58***   
             (0.06)   (0.02)   (0.08)   (0.03)   (0.03)   (0.02)     
N            5535     5535     5535     5535     5535     5535       
R2adj        0.102    0.014    0.072    0.161    0.156    0.092      
Standard errors in 

In [58]:
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, lagn = 4, feffect = "institution")
        model.tables[0].to_excel("results/test/reg1/finOnComplexityDetail{}.xlsx".format("institution"+finType+finVar+str(4)))
        print(model)
        


                  abstract   info     read   disunity  strain  strainUpper
--------------------------------------------------------------------------
ret_SPY_lag1      -0.76    0.34     1.25     -0.29    0.27     -0.69      
                  (2.27)   (0.87)   (1.67)   (1.15)   (0.82)   (0.87)     
ret_SPY_lag2      -2.87    -0.63    -0.91    -0.68    0.05     -0.56      
                  (2.39)   (1.03)   (1.67)   (1.11)   (0.87)   (0.96)     
ret_SPY_lag3      -2.41    0.26     -1.67    -1.76*   -0.82    -0.38      
                  (2.17)   (0.92)   (1.84)   (1.04)   (0.77)   (0.84)     
ret_SPY_lag4      -0.85    -0.36    -0.78    -0.99    -0.88    -0.40      
                  (2.28)   (1.09)   (1.79)   (1.08)   (0.82)   (0.92)     
inst_NewYork      -0.70*** -0.07    -1.01*** -0.35*** -0.42*** -0.48***   
                  (0.10)   (0.04)   (0.10)   (0.04)   (0.03)   (0.03)     
inst_Atlanta      -1.98*** -0.31*** -1.66*** -1.00*** -0.68*** -1.13***   
                  (0.11)

In [43]:
# using F-test instead
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull=False, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull=False, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/test/reg1/finOnComplexity.csv")
display(model)

# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull=True, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull=True, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/test/reg1/finOnComplexity2.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPY,0,0,0,0,0,0
ret_SPX_index,0,0,0,0,0,-
ret_ES_futures,0,0,0,-,0,0
ret_SHY,++,0,++,+,0,0
ret_IEF,0,0,++,+++,+,++
ret_TLT,0,0,0,+++,0,+++
ret_UUP,-,0,---,-,0,0
ret_VXX,0,+,0,0,0,++
ret_VIX,0,+,0,+,++,++
ret_GLD,0,0,0,0,0,0


,abstract,info,read,disunity,strain,strainUpper
ret_SPY,0,0,0,0,0,0
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,0,**,0,0,0,0
ret_IEF,0,0,0,*,0,0
ret_TLT,0,0,0,*,0,*
ret_UUP,0,0,**,0,*,*
ret_VXX,0,0,0,0,0,*
ret_VIX,0,0,0,0,*,0
ret_GLD,0,0,0,0,0,*


In [46]:
# using F-test instead
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, feffect = "institution")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, feffect = "institution")
            model = pd.concat([model,resdum])
model.to_csv("results/test/reg1/finOnComplexityInstitution.csv")
display(model)

# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull = True, feffect = "institution")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull = True,  feffect = "institution")
            model = pd.concat([model,resdum])
model.to_csv("results/test/reg1/finOnComplexityInstitution2.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPY,-,0,0,-,0,0
ret_SPX_index,--,0,0,-,0,-
ret_ES_futures,0,0,0,--,-,0
ret_SHY,+++,0,+++,++,0,0
ret_IEF,++,0,++,++,+,++
ret_TLT,++,0,++,+++,0,++
ret_UUP,--,0,--,-,0,0
ret_VXX,0,0,0,0,0,++
ret_VIX,0,+,0,+,+,++
ret_GLD,0,0,0,0,0,0


,abstract,info,read,disunity,strain,strainUpper
ret_SPY,0,0,0,0,0,0
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,*,0,*,0,0,0
ret_IEF,0,0,0,*,0,0
ret_TLT,0,0,0,0,0,0
ret_UUP,0,0,0,*,0,0
ret_VXX,0,0,0,0,0,0
ret_VIX,0,0,0,0,0,0
ret_GLD,0,0,0,0,0,0


In [68]:
# using F-test instead
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, feffect = "role")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, feffect = "role")
            model = pd.concat([model,resdum])
model.to_csv("results/test/reg1/finOnComplexityRole.csv")
display(model)

# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull = True, feffect = "role")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = True, sepNull = True,  feffect = "role")
            model = pd.concat([model,resdum])
model.to_csv("results/test/reg1/finOnComplexityRole2.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPY,-,0,0,-,0,0
ret_SPX_index,-,0,0,0,0,0
ret_ES_futures,0,0,0,--,0,0
ret_SHY,+++,0,+++,++,0,0
ret_IEF,++,0,+++,+++,+,+
ret_TLT,++,0,++,+++,+,++
ret_UUP,--,0,---,--,0,-
ret_VXX,0,+,0,0,0,++
ret_VIX,0,+,0,++,++,++
ret_GLD,0,0,0,0,0,0


[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[0. 0. 0. 0.]
['volume_VIX_lag1', 'volume_VIX_lag2', 'volume_VIX_lag3', 'volume_VIX_lag4']
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[0. 0. 0. 0.]
['volume_VIX_lag1', 'volume_VIX_lag2', 'volume_VIX_lag3', 'volume_VIX_lag4']
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[0. 0. 0. 0.]
['volume_VIX_lag1', 'volume_VIX_lag2', 'volume_VIX_lag3', 'volume_VIX_lag4']
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[0. 0. 0. 0.]
['volume_VIX_lag1', 'volume_VIX_lag2', 'volume_VIX_lag3', 'volume_VIX_lag4']
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[0. 0. 0. 0.]
['volume_VIX_lag1', 'volume_VIX_lag2', 'volume_VIX_lag3', 'volume_VIX_lag4']
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[0. 0. 0. 0.]
['volume_VIX_lag1', 'volume_VIX_lag2', 'volume_VIX_lag3', 'volume_VIX_lag4']


,abstract,info,read,disunity,strain,strainUpper
ret_SPY,0,0,0,0,0,0
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,*,0,0,0,0,0
ret_IEF,0,0,0,**,*,0
ret_TLT,0,0,0,*,*,0
ret_UUP,0,0,*,**,0,0
ret_VXX,0,0,0,0,0,**
ret_VIX,0,0,0,0,0,0
ret_GLD,0,0,0,0,0,0


### Second group of regressions

In [70]:
inputList = speechVarList
finNameList = finTypeList
OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = False)



            ret_SPY_fut1 ret_SPX_index_fut1 ret_ES_futures_fut1 ret_SHY_fut1 ret_IEF_fut1 ret_TLT_fut1 ret_UUP_fut1 ret_VXX_fut1 ret_VIX_fut1 ret_GLD_fut1
----------------------------------------------------------------------------------------------------------------------------------------------------------
abstract    0.00         0.00               0.00                0.00         0.00         0.00         -0.00        -0.00        0.00         0.00        
            (0.00)       (0.00)             (0.00)              (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)      
info        0.00         0.00               0.00                0.00         0.00         -0.00        0.00         0.00         -0.00        -0.00       
            (0.00)       (0.00)             (0.00)              (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)      
read        0.00         0.00               -0.00               -0.00

,ret,abs_ret,hl_range,volume,abn_ret_5,abn_ret_22
SPY,NaN,NaN,NaN,NaN,NaN,NaN
SPX_index,NaN,NaN,NaN,NaN,NaN,NaN
ES_futures,NaN,NaN,NaN,NaN,NaN,NaN
SHY,NaN,NaN,NaN,NaN,NaN,NaN
IEF,NaN,NaN,NaN,NaN,NaN,NaN
TLT,NaN,NaN,NaN,NaN,NaN,NaN
UUP,NaN,NaN,NaN,NaN,NaN,NaN
VXX,NaN,NaN,NaN,NaN,NaN,NaN
VIX,NaN,NaN,NaN,NaN,NaN,NaN
GLD,NaN,NaN,NaN,NaN,NaN,NaN


In [116]:
res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True)
res.to_csv("results/test/reg2/complexityOnFinFtest.csv")
display(res)

res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True, sepNull = True)
res.to_csv("results/test/reg2/complexityOnFinFtest2.csv")
display(res)


,ret,abs_ret,hl_range,volume,abn_ret_5,abn_ret_22
SPY,0,0,0,+++,0,0
SPX_index,0,0,0,+++,0,0
ES_futures,0,---,---,---,0,0
SHY,0,---,0,---,0,0
IEF,0,---,---,---,0,0
TLT,0,---,---,---,0,0
UUP,0,--,0,---,0,0
VXX,0,---,---,---,0,0
VIX,0,0,0,0,0,0
GLD,-,-,0,---,0,0


[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
[0. 0. 0. 0. 0. 0.]
['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper']


,ret,abs_ret,hl_range,volume,abn_ret_5,abn_ret_22
SPY,0,0,***,***,0,0
SPX_index,0,**,***,***,0,0
ES_futures,0,***,***,***,0,0
SHY,0,***,***,***,0,0
IEF,*,***,***,***,0,0
TLT,0,***,***,***,0,0
UUP,0,***,***,***,0,0
VXX,0,***,***,***,0,0
VIX,0,**,***,error,0,0
GLD,0,***,***,***,0,0


## Main regressions

### Side-track: enumerate some variables: speaker backgrounds, roles, topics

In [68]:
dfmain = backup.copy()
speakerBG = ['Gender', 'Race']

# Race
dfmain['Race'] = dfmain['Race'].map({'W': 1, 'NW': 0})

# major
var = "Major"
categoryOrder = ['Econ', 'Law', 'Business','Fin']
dfmain[var+'Collapsed'] = dfmain[var].replace({
    'Economics': 'Econ',
    'Law': 'Law',
    'Business': 'Business',
    'Finance': 'Fin'
})
dfmain[var+'Collapsed'] = dfmain[var+'Collapsed'].where(dfmain[var+'Collapsed'].isin(categoryOrder),'Other')
dfmain[var+'Collapsed'] = pd.Categorical(
    dfmain[var+'Collapsed'],
    categories=categoryOrder+['Other'],
    ordered=True
)
all_dummies = pd.get_dummies(dfmain[var+'Collapsed'], drop_first=True, prefix=var).astype(int)
dfmain = pd.concat([dfmain, all_dummies], axis=1)
speakerBG = speakerBG + [var+"_"+i for i in categoryOrder[1:]+['Other']]

# degree
var = "Degree"
categoryOrder = ['PhD']
dfmain[var+'Collapsed'] = dfmain[var]
dfmain[var+'Collapsed'] = dfmain[var+'Collapsed'].where(dfmain[var+'Collapsed'].isin(categoryOrder),'Other')
dfmain[var+'Collapsed'] = pd.Categorical(
    dfmain[var+'Collapsed'],
    categories=categoryOrder+['Other'],
    ordered=True
)
all_dummies = pd.get_dummies(dfmain[var+'Collapsed'], drop_first=True, prefix=var).astype(int)
dfmain = pd.concat([dfmain, all_dummies], axis=1)
speakerBG = speakerBG + [var+"_"+i for i in categoryOrder[1:]+['Other']]

# PreFed
var = "PreFed_clean"
categoryOrder = ['Econ','AcademicEcon','Finance','Business','Law']
dfmain[var+'Collapsed'] = dfmain[var].replace({
    'Professional Economist': 'Econ',
    'Academic Economist': 'AcademicEcon',
    'Banking/Finance/Fed': 'Finance',
    'Business/Industry':'Business',
    'Law/Legal':'Law'
})
dfmain[var+'Collapsed'] = dfmain[var+'Collapsed'].where(dfmain[var+'Collapsed'].isin(categoryOrder),'Other')
dfmain[var+'Collapsed'] = pd.Categorical(
    dfmain[var+'Collapsed'],
    categories=categoryOrder+['Other'],
    ordered=True
)
all_dummies = pd.get_dummies(dfmain[var+'Collapsed'], drop_first=True, prefix=var).astype(int)
dfmain = pd.concat([dfmain, all_dummies], axis=1)
speakerBG = speakerBG + [var+"_"+i for i in categoryOrder[1:]+['Other']]

# SaltFresh
var = "SaltFresh"
dfmain["Saltwater"] = (dfmain[var] == 1.0).astype(int)
dfmain["Freshwater"] = (dfmain[var] == 2.0).astype(int)
dfmain["SaltFresh_Other"] = (dfmain[var] == 3.0).astype(int)
speakerBG = speakerBG + ["Freshwater","SaltFresh_Other"]

# Age-related
# start and end yera should be converted the negative thingy back to null
dfmain.loc[dfmain["Start Year"] == -9999.0, "Start Year"] = pd.NA
dfmain.loc[dfmain["End Year"] == 9999.0, "End Year"] = pd.NA
dfmain['Age'] = dfmain['speech_year'] - dfmain['Birth Year']
dfmain['yearsIn'] = dfmain['speech_year'] - dfmain['Start Year']
speakerBG = speakerBG + ["Age","yearsIn"]

print(speakerBG)


['Gender', 'Race', 'Major_Law', 'Major_Business', 'Major_Fin', 'Major_Other', 'Degree_Other', 'PreFed_clean_AcademicEcon', 'PreFed_clean_Finance', 'PreFed_clean_Business', 'PreFed_clean_Law', 'PreFed_clean_Other', 'Freshwater', 'SaltFresh_Other', 'Age', 'yearsIn']


In [76]:
dfmain["Chair"] = (dfmain["board_role"] == "Chair").astype(int)
backup = dfmain.copy()


In [78]:
dfmain = backup.copy()
# the time data
# Convert to datetime (preserves NaT for missing)
dfmain["timeOnly_dt"] = pd.to_datetime(dfmain["timeOnly"], format="%H:%M:%S", errors="coerce")

# Extract fractional hour (e.g., 14:30 → 14.5)
dfmain["hour_frac"] = dfmain["timeOnly_dt"].dt.hour + dfmain["timeOnly_dt"].dt.minute / 60


def classify_time_of_day(h):
    if pd.isna(h):
        return None   
    if 9 <= h < 12:
        return "morning"
    elif 12 <= h < 14.5:
        return "midday"
    elif 14.5 <= h < 16.5:
        return "afternoon"
    else:
        return "evening"   # speeches after market hours

dfmain["TimeOfDay"] = dfmain["hour_frac"].apply(classify_time_of_day)
dummies = pd.get_dummies(dfmain['TimeOfDay'], prefix="tod", drop_first=True).astype(int)
print(dfmain.shape)
dfmain = pd.concat([dfmain, dummies], axis=1)
print(dfmain.shape)

# inspecting 
dfmain[dfmain['timeOnly'].notna()][['timeOnly','hour_frac','tod_midday','tod_morning','tod_evening']].head()


(7177, 1041)
(7177, 1044)


,timeOnly,hour_frac,tod_midday,tod_morning,tod_evening
4514,13:30:00,13.5,1,0,0
4515,20:00:00,20.0,0,0,1
4516,17:00:00,17.0,0,0,1
4517,12:30:00,12.5,1,0,0
4520,12:00:00,12.0,1,0,0


In [81]:
print(dfmain.shape)
print(dfmain.columns.tolist())


(7177, 1044)
['District', 'MultSpeakers', 'VideoForm', 'Article', 'title', 'speaker', 'location', 'date', 'positiveFin', 'neutralFin', 'negativeFin', 'hawkishMal', 'n_hawk_pair', 'n_dove_pair', 'board_role', 'IsBpres', 'timeOnly', 'score', 'abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper', 'GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 'UNRATE', 'UNRATE_lag1', 'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'MICH', 'MICH_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1', 'gRGDPB1', 'gRGDPF0', 'gRGDPF1', 'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 'HSTARTB1', 'HSTARTF0', 'HSTARTF1', 'gIPB1', 'gIPF0', 'gIPF1', 'is_monday', 'is_friday', 'speaker_clean', 'speech_year', 'row_id', 'Gender', 'Birth Year', 'Start Year', 

In [83]:
dfmain.to_csv('FedSpeechesRobust.csv', date_format='%Y-%m-%d %H:%M:%S', index = False)


In [84]:
speechVarList = ['abstract','info','read','disunity','strain','strainUpper']

finTypeList = ["SPY","SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]

# finVarList = ["price","ret","log_ret","abs_ret","ret_sq","hl_range","volume",
#               "roll_mean_5","roll_mean_22","roll_vol_5","roll_vol_22", "abn_ret_5","abn_ret_22"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

# macroVarFredList = ['GDPC1', 'GDPC1_lag1', 'GDPC1_change', 'GDPC1_change_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1',
#                     'INDPRO', 'INDPRO_lag1', 'INDPRO_change', 'INDPRO_change_lag1', 
#                     'UNRATE', 'UNRATE_lag1', 
#                     'PCEPILFE', 'PCEPILFE_lag1', 'PCEPILFE_change', 'PCEPILFE_change_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1',
#                     'CPIAUCSL', 'CPIAUCSL_lag1', 'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1',
#                     'MICH', 'MICH_lag1']
realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList1 = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']

macroVarGTList2 = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1',
                  'gIPB1', 'gIPF0', 'gIPF1']

topicList = ['community_development','monetary_policy_inflation','real_economy_productivity',
             'fed_operations_payments','financial_stability_regulation','international_economics']

otherList = ['is_monday', 'is_friday','Chair','tod_midday','tod_morning']+topicList



In [182]:
# This is just to see if some control variables are highly correlated, from experience
# so that i can exclude, specifically IPB
macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1', 
                  'gIPB1', 'gIPF0', 'gIPF1']

corr = dfmain[macroVarGTList].corr().abs()

# Keep only upper triangle to avoid duplicates
upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_pairs = upper.stack()[upper.stack() > 0.8]
high_pairs



gRGDPB1   gIPB1       0.895922
gRGDPF0   gIPF0       0.921039
UNEMPB1   UNEMPF0     0.801988
          UNEMPF1     0.864437
UNEMPF0   UNEMPF1     0.966047
HSTARTB1  HSTARTF0    0.968031
          HSTARTF1    0.960858
HSTARTF0  HSTARTF1    0.989211
dtype: float64

In [184]:
macroVarFredList = ['GDPC1_change', 'GDPC1_change_lag1', 'GDPC1_low_q10', 'GDPC1_low_q10_lag1',
                    'INDPRO_change', 'INDPRO_change_lag1', 
                    'UNRATE', 'UNRATE_lag1', 
                    'PCEPILFE_change', 'PCEPILFE_change_lag1', 'PCEPILFE_yoy', 'PCEPILFE_yoy_lag1',
                    'CPIAUCSL_change', 'CPIAUCSL_change_lag1', 'CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1',
                    'MICH', 'MICH_lag1']

corr = dfmain[macroVarFredList].corr().abs()

# Keep only upper triangle to avoid duplicates
upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_pairs = upper.stack()[upper.stack() > 0.8]
high_pairs


UNRATE        UNRATE_lag1          0.945790
PCEPILFE_yoy  PCEPILFE_yoy_lag1    0.987191
CPIAUCSL_yoy  CPIAUCSL_yoy_lag1    0.968317
MICH          MICH_lag1            0.884692
dtype: float64

In [190]:
finTypeList = ["SPY","SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["price","ret","log_ret","abs_ret","ret_sq","hl_range","volume",
              "roll_mean_5","roll_mean_22","roll_vol_5","roll_vol_22", "abn_ret_5","abn_ret_22"]
fullList = [var1+"_"+var2 for var1 in finVarList for var2 in finTypeList]

corr = dfmain[fullList].corr().abs()

# Keep only upper triangle to avoid duplicates
upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

high_pairs = upper.stack()[upper.stack() > 0.8]
high_pairs



price_SPY               price_SPX_index            0.997262
                        price_ES_futures           0.998553
                        price_VXX                  0.922941
price_SPX_index         price_ES_futures           0.999976
                        price_VXX                  0.912523
price_ES_futures        price_VXX                  0.915316
price_SHY               price_IEF                  0.957940
                        price_TLT                  0.892552
                        price_GLD                  0.882524
price_IEF               price_TLT                  0.977556
                        price_GLD                  0.812244
price_VIX               hl_range_ES_futures        0.805770
                        roll_vol_22_SPY            0.881943
                        roll_vol_22_SPX_index      0.881375
                        roll_vol_22_ES_futures     0.874478
ret_SPY                 ret_SPX_index              0.984684
                        ret_ES_futures  

### Checking out a few financial variables to see control coefficients

In [36]:
realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList1 = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']

macroVarGTList2 = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1',
                  'gIPB1', 'gIPF0', 'gIPF1']


In [38]:
outputList = speechVarList
controlList = realAct3 +inflaty2 + mich + unemp

for finVar in ["volume","hl_range"]:
    for finType in ["SPX_index","IEF"]:
        model = OLSFin(dfmain, finType, finVar, outputList,  controlList, Ftest = False, sepNull=False, feffect = "individual")
        display(model)


,abstract,info,read,disunity,strain,strainUpper
volume_SPX_index_lag1,-0.00,0.00,-0.00***,-0.00**,-0.00*,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag2,0.00,-0.00**,-0.00,-0.00,-0.00*,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag3,0.00,-0.00,-0.00*,0.00,0.00,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag4,-0.00*,0.00,-0.00***,-0.00**,-0.00*,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
INDPRO_change,2.92,-3.83**,-0.79,-1.53,-0.89,0.33
,(3.56),(1.60),(4.27),(1.93),(1.41),(1.68)


,abstract,info,read,disunity,strain,strainUpper
volume_IEF_lag1,-0.00**,-0.00,0.00,-0.00***,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag2,0.00,-0.00***,0.00**,0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag3,0.00,0.00,0.00,-0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag4,-0.00**,-0.00,-0.00,-0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
INDPRO_change,2.93,-1.41,-1.72,-0.79,-1.27,0.12
,(3.84),(1.72),(1.20),(1.53),(1.00),(1.61)


,abstract,info,read,disunity,strain,strainUpper
hl_range_SPX_index_lag1,3.40,0.51,-7.95**,-2.49*,-3.04***,-2.60**
,(2.62),(1.47),(3.09),(1.50),(1.13),(1.10)
hl_range_SPX_index_lag2,15.54***,1.72,2.93,2.05,0.96,0.89
,(3.40),(2.51),(4.08),(1.92),(1.45),(1.52)
hl_range_SPX_index_lag3,-5.52*,-3.16*,-5.02,-2.91*,-1.32,-2.69**
,(3.18),(1.73),(3.10),(1.62),(1.21),(1.24)
hl_range_SPX_index_lag4,-8.52***,1.15,-6.16*,-3.13,-2.13,-1.16
,(3.24),(1.44),(3.73),(1.95),(1.41),(1.42)
INDPRO_change,4.15,-2.81*,0.36,-0.86,-0.60,-0.38
,(3.51),(1.57),(4.26),(1.91),(1.39),(1.66)


,abstract,info,read,disunity,strain,strainUpper
hl_range_IEF_lag1,19.78,-1.24,-0.42,-4.84,-2.44,0.03
,(12.62),(5.17),(3.47),(5.18),(2.89),(4.66)
hl_range_IEF_lag2,37.79***,7.10,12.67***,5.40,-0.16,1.48
,(12.70),(5.86),(4.02),(4.98),(2.84),(5.42)
hl_range_IEF_lag3,9.57,3.57,2.38,3.77,2.76,4.96
,(11.84),(5.95),(3.25),(4.08),(2.88),(5.47)
hl_range_IEF_lag4,-27.65**,-6.14,-4.77,-1.77,-5.81**,-11.59***
,(13.35),(4.70),(3.96),(4.60),(2.80),(4.30)
INDPRO_change,3.52,-1.48,-1.54,-0.70,-1.65,-0.46
,(3.93),(1.78),(1.19),(1.58),(1.00),(1.66)


In [304]:
outputList = speechVarList
controlList = macroVarGTList1

for finVar in ["volume","abs_ret"]:
    for finType in ["SPX_index","IEF"]:
        model = OLSFin(dfmain, finType, finVar, outputList,  controlList, Ftest = False, sepNull=False, feffect = "individual")
        display(model)


,abstract,info,read,disunity,strain,strainUpper
volume_SPX_index_lag1,-0.00,0.00,-0.00***,-0.00***,-0.00*,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag2,0.00,-0.00*,-0.00,0.00,-0.00,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag3,-0.00,-0.00,-0.00**,0.00,-0.00,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag4,-0.00,0.00,-0.00***,-0.00,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
gRGDPB1,0.00,-0.01***,0.01,-0.00,-0.00,-0.00
,(0.01),(0.00),(0.01),(0.00),(0.00),(0.00)


,abstract,info,read,disunity,strain,strainUpper
volume_IEF_lag1,0.00,-0.00,0.00,-0.00,-0.00**,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag2,-0.00,-0.00***,0.00,0.00,-0.00,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag3,0.00,-0.00,0.00,-0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag4,-0.00**,-0.00,0.00,0.00,0.00*,0.00*
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
gRGDPB1,-0.01,-0.01**,-0.00,0.00,-0.00,-0.00
,(0.01),(0.00),(0.00),(0.00),(0.00),(0.00)


,abstract,info,read,disunity,strain,strainUpper
abs_ret_SPX_index_lag1,3.85,-0.70,-6.26**,-1.36,-2.63**,-4.34***
,(2.51),(1.20),(2.73),(1.41),(1.03),(1.00)
abs_ret_SPX_index_lag2,4.91*,2.04,-5.78**,1.21,0.04,0.22
,(2.97),(2.10),(2.74),(1.60),(1.13),(1.32)
abs_ret_SPX_index_lag3,-2.79,-2.33,-3.33,-2.83**,-1.08,-2.04**
,(2.68),(1.45),(2.81),(1.43),(1.07),(0.99)
abs_ret_SPX_index_lag4,-5.17*,1.96,-8.76***,-4.42**,-2.39*,-1.48
,(2.82),(1.30),(3.38),(1.81),(1.26),(1.18)
gRGDPB1,0.00,-0.01**,0.01*,-0.00,-0.00,-0.00
,(0.01),(0.00),(0.01),(0.00),(0.00),(0.00)


,abstract,info,read,disunity,strain,strainUpper
abs_ret_IEF_lag1,-14.81,0.05,-1.14,3.68,-2.29,-7.05
,(10.94),(4.84),(3.12),(4.47),(2.67),(4.55)
abs_ret_IEF_lag2,26.22**,7.65,3.57,-2.58,-1.12,3.17
,(10.48),(4.70),(3.17),(4.30),(2.67),(4.83)
abs_ret_IEF_lag3,-4.86,5.45,-4.06,0.52,0.41,4.00
,(10.58),(4.61),(2.86),(4.09),(2.67),(6.14)
abs_ret_IEF_lag4,-1.96,1.30,0.56,3.49,-1.21,-3.83
,(11.47),(4.53),(3.18),(4.06),(2.64),(4.58)
gRGDPB1,-0.01,-0.01**,-0.00,0.00,-0.00,-0.00
,(0.01),(0.00),(0.00),(0.00),(0.00),(0.00)


### First group of regressions

In [87]:
finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']


In [45]:
# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
controlList = realAct3+inflaty2+unemp+mich+topicList

for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/reg1/RegG1main.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,0,**,0,0,0,0
ret_IEF,0,0,0,0,0,0
ret_TLT,0,0,0,0,0,0
ret_UUP,0,0,0,0,*,0
ret_VXX,0,0,0,0,0,**
ret_VIX,0,0,0,0,*,0
ret_GLD,0,0,0,0,0,0
abs_ret_SPX_index,*,0,***,***,***,***


In [44]:
name1 = ["& SPY","& ES","& SHY","& IEF","& TLT","& UUP","& VXX","& VIX","& GLD"]
name2 = ["Return","Absolute Return","High-low Range","Volume","Abnormal Return (5 days)","Abnormal Return (22 days)"]
nameFinal = [name2[0]+' '+name1[0]]+name1[1:]
for var2 in name2[1:]:
    nameFinal += [var2+name1[0]]+name1[1:]
model.index = nameFinal
latex = df_to_latex(model)   # or whatever object you pass in
print(latex)



\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{abstract} & \makecell{info} & \makecell{read} & \makecell{disunity} & \makecell{strain} & \makecell{strainUpper} \\
\midrule
Return & SPY & 0 & 0 & 0 & 0 & 0 & 0 \\
& ES & 0 & 0 & 0 & 0 & 0 & 0 \\
& SHY & 0 & ** & 0 & 0 & 0 & 0 \\
& IEF & 0 & 0 & 0 & 0 & 0 & 0 \\
& TLT & 0 & 0 & 0 & 0 & 0 & 0 \\
& UUP & 0 & 0 & 0 & 0 & * & 0 \\
& VXX & 0 & 0 & 0 & 0 & 0 & ** \\
& VIX & 0 & 0 & 0 & 0 & * & 0 \\
& GLD & 0 & 0 & 0 & 0 & 0 & 0 \\
Absolute Return& SPY & 0 & 0 & *** & *** & *** & *** \\
& ES & 0 & 0 & 0 & ** & 0 & ** \\
& SHY & 0 & 0 & 0 & 0 & ** & 0 \\
& IEF & ** & 0 & 0 & 0 & 0 & 0 \\
& TLT & *** & 0 & ** & 0 & 0 & 0 \\
& UUP & 0 & 0 & 0 & 0 & 0 & 0 \\
& VXX & ** & ** & 0 & 0 & 0 & * \\
& VIX & 0 & 0 & 0 & 0 & 0 & 0 \\
& GLD & 0 & 0 & *** & * & * & 0 \\
High-low Range& SPY & *** & 0 & *** & *** & *** & *** \\
& ES & ** & 0 & 0 & 0 & 0 & 0 \\
& SHY & 0 & 0 & 0 & 0 & 0 & 0 \\
& IEF & ** & 0 & ** & 0 & 0 & 0 \\
& TLT & ** & 0 & *** & 0 & 0 & 0 \\

In [45]:
# using F-test instead
first = True
outputList = speechVarList
for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=False, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=False, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/reg1/RegG1robustSign.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPX_index,0,0,0,0,0,--
ret_ES_futures,0,0,0,0,0,0
ret_SHY,0,--,+,++,0,+
ret_IEF,0,0,0,++,+,++
ret_TLT,0,0,0,++,0,+++
ret_UUP,0,+,--,0,0,0
ret_VXX,0,0,0,0,0,++
ret_VIX,0,0,0,+,++,++
ret_GLD,0,0,0,0,0,0
abs_ret_SPX_index,-,0,---,---,---,---


In [46]:
name1 = ["& SPY","& ES","& SHY","& IEF","& TLT","& UUP","& VXX","& VIX","& GLD"]
name2 = ["Return","Absolute Return","High-low Range","Volume","Abnormal Return (5 days)","Abnormal Return (22 days)"]
nameFinal = [name2[0]+' '+name1[0]]+name1[1:]
for var2 in name2[1:]:
    nameFinal += [var2+name1[0]]+name1[1:]
model.index = nameFinal
latex = df_to_latex(model)   # or whatever object you pass in
print(latex)



\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{abstract} & \makecell{info} & \makecell{read} & \makecell{disunity} & \makecell{strain} & \makecell{strainUpper} \\
\midrule
Return & SPY & 0 & 0 & 0 & 0 & 0 & -- \\
& ES & 0 & 0 & 0 & 0 & 0 & 0 \\
& SHY & 0 & -- & + & ++ & 0 & + \\
& IEF & 0 & 0 & 0 & ++ & + & ++ \\
& TLT & 0 & 0 & 0 & ++ & 0 & +++ \\
& UUP & 0 & + & -- & 0 & 0 & 0 \\
& VXX & 0 & 0 & 0 & 0 & 0 & ++ \\
& VIX & 0 & 0 & 0 & + & ++ & ++ \\
& GLD & 0 & 0 & 0 & 0 & 0 & 0 \\
Absolute Return& SPY & - & 0 & --- & --- & --- & --- \\
& ES & -- & 0 & + & +++ & 0 & 0 \\
& SHY & 0 & 0 & 0 & 0 & --- & - \\
& IEF & 0 & 0 & + & 0 & - & 0 \\
& TLT & 0 & 0 & +++ & 0 & 0 & 0 \\
& UUP & - & 0 & 0 & 0 & 0 & 0 \\
& VXX & --- & 0 & 0 & 0 & 0 & 0 \\
& VIX & 0 & 0 & 0 & + & 0 & 0 \\
& GLD & 0 & 0 & 0 & ++ & 0 & 0 \\
High-low Range& SPY & 0 & 0 & --- & --- & --- & --- \\
& ES & -- & 0 & ++ & ++ & 0 & 0 \\
& SHY & 0 & 0 & 0 & + & -- & - \\
& IEF & 0 & - & 0 & 0 & - & 0 \\
& TLT & 0 & - & +++ & 0 & 0

In [48]:
first = True
outputList = speechVarList
controlList = realAct3+inflaty1+unemp+mich+topicList

for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/reg1/RegG1robustMacro1.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,0,**,0,0,0,0
ret_IEF,0,0,0,0,0,0
ret_TLT,0,0,0,0,0,0
ret_UUP,0,0,0,0,*,0
ret_VXX,0,0,0,0,0,*
ret_VIX,0,0,0,0,*,0
ret_GLD,0,0,0,0,0,0
abs_ret_SPX_index,0,0,***,***,***,***


In [49]:
# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
controlList = realAct2+inflaty2+unemp+mich+topicList

for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/reg1/RegG1robustMacro2.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,0,**,0,0,0,0
ret_IEF,0,0,0,0,0,0
ret_TLT,0,0,0,0,0,0
ret_UUP,0,0,0,0,*,0
ret_VXX,0,0,0,0,0,*
ret_VIX,0,0,0,0,*,0
ret_GLD,0,0,0,0,0,0
abs_ret_SPX_index,0,0,***,**,***,***


In [50]:
# using F-test but testing if individual coefficients are significant
first = True
outputList = speechVarList
controlList = macroVarGTList+topicList

for finVar in finVarList:
    for finType in finTypeList:
        if first:
            model = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            first = False
        else:
            resdum = OLSFin(dfmain, finType, finVar, outputList,  controlList,Ftest = True, sepNull=True, feffect = "individual")
            model = pd.concat([model,resdum])
model.to_csv("results/reg1/RegG1robustMacro3.csv")
display(model)


,abstract,info,read,disunity,strain,strainUpper
ret_SPX_index,0,0,0,0,0,0
ret_ES_futures,0,0,0,0,0,0
ret_SHY,0,**,0,0,0,0
ret_IEF,0,0,0,0,0,0
ret_TLT,0,0,0,0,0,0
ret_UUP,0,0,0,0,**,0
ret_VXX,0,0,0,*,0,***
ret_VIX,0,0,0,0,0,0
ret_GLD,0,0,0,0,0,0
abs_ret_SPX_index,0,0,***,**,***,***


In [51]:
name1 = ["& SPY","& ES","& SHY","& IEF","& TLT","& UUP","& VXX","& VIX","& GLD"]
name2 = ["Return","Absolute Return","High-low Range","Volume","Abnormal Return (5 days)","Abnormal Return (22 days)"]
nameFinal = [name2[0]+' '+name1[0]]+name1[1:]
for var2 in name2[1:]:
    nameFinal += [var2+name1[0]]+name1[1:]
model.index = nameFinal
latex = df_to_latex(model)   # or whatever object you pass in
print(latex)


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{abstract} & \makecell{info} & \makecell{read} & \makecell{disunity} & \makecell{strain} & \makecell{strainUpper} \\
\midrule
Return & SPY & 0 & 0 & 0 & 0 & 0 & 0 \\
& ES & 0 & 0 & 0 & 0 & 0 & 0 \\
& SHY & 0 & ** & 0 & 0 & 0 & 0 \\
& IEF & 0 & 0 & 0 & 0 & 0 & 0 \\
& TLT & 0 & 0 & 0 & 0 & 0 & 0 \\
& UUP & 0 & 0 & 0 & 0 & ** & 0 \\
& VXX & 0 & 0 & 0 & * & 0 & *** \\
& VIX & 0 & 0 & 0 & 0 & 0 & 0 \\
& GLD & 0 & 0 & 0 & 0 & 0 & 0 \\
Absolute Return& SPY & 0 & 0 & *** & ** & *** & *** \\
& ES & 0 & 0 & 0 & * & 0 & *** \\
& SHY & 0 & 0 & 0 & 0 & * & 0 \\
& IEF & 0 & 0 & 0 & 0 & 0 & 0 \\
& TLT & * & 0 & 0 & 0 & 0 & 0 \\
& UUP & 0 & 0 & 0 & 0 & 0 & 0 \\
& VXX & 0 & ** & 0 & 0 & 0 & * \\
& VIX & 0 & 0 & 0 & 0 & 0 & 0 \\
& GLD & 0 & 0 & ** & 0 & ** & 0 \\
High-low Range& SPY & *** & 0 & *** & *** & *** & *** \\
& ES & 0 & 0 & 0 & 0 & 0 & * \\
& SHY & 0 & 0 & 0 & 0 & 0 & 0 \\
& IEF & ** & 0 & * & 0 & 0 & 0 \\
& TLT & ** & 0 & ** & 0 & 0 & 0 \\
& UUP &

### Sub-reg of First group

In [89]:
outputList = speechVarList
controlList = speakerBG
controlList = controlList+ realAct3 +inflaty2 + mich + unemp + topicList

for finVar in ["volume","hl_range"]:
    for finType in ["SPX_index","IEF"]:
        model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, sepNull=False, feffect = "role")
        model.tables[0].to_excel(f"results/reg1/RegG1mainSub{finVar+finType}.xlsx")
        display(model)


,abstract,info,read,disunity,strain,strainUpper
volume_SPX_index_lag1,-0.00,0.00*,-0.00*,0.00,0.00**,0.00*
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag2,-0.00,-0.00,-0.00,-0.00,-0.00*,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag3,0.00,-0.00,-0.00,0.00,0.00,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag4,-0.00*,0.00,-0.00,-0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.25***,-0.19***,0.06,-0.01,-0.11***,0.13***
,(0.08),(0.04),(0.05),(0.04),(0.03),(0.03)


,abstract,info,read,disunity,strain,strainUpper
volume_IEF_lag1,-0.00***,-0.00,-0.00***,-0.00***,-0.00**,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag2,-0.00,-0.00**,0.00,0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag3,-0.00,-0.00,-0.00,-0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag4,-0.00***,-0.00,-0.00,-0.00*,-0.00,0.00*
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.25***,-0.24***,0.03,0.13***,0.02,0.12***
,(0.08),(0.04),(0.02),(0.03),(0.03),(0.04)


,abstract,info,read,disunity,strain,strainUpper
hl_range_SPX_index_lag1,1.25,0.85,-7.91**,-1.44,-1.34,-1.41
,(2.77),(1.62),(3.33),(1.69),(1.40),(1.27)
hl_range_SPX_index_lag2,8.94***,1.27,0.49,1.38,0.04,0.62
,(3.39),(2.75),(4.56),(2.16),(1.72),(1.72)
hl_range_SPX_index_lag3,-4.76,-3.27*,-4.69,-2.15,-0.40,-2.40*
,(3.42),(1.82),(3.41),(2.03),(1.54),(1.44)
hl_range_SPX_index_lag4,-5.07,2.47,-3.18,-0.83,-0.05,0.13
,(3.30),(1.58),(3.95),(2.18),(1.65),(1.58)
Gender,-0.20***,-0.20***,0.12**,-0.02,-0.12***,0.12***
,(0.08),(0.04),(0.05),(0.04),(0.03),(0.03)


,abstract,info,read,disunity,strain,strainUpper
hl_range_IEF_lag1,12.22,-4.34,2.15,-7.33,-0.51,4.29
,(13.33),(5.39),(3.56),(5.77),(3.52),(5.02)
hl_range_IEF_lag2,21.47*,5.36,9.25***,1.16,-1.36,3.39
,(12.33),(6.15),(3.47),(5.53),(3.70),(5.76)
hl_range_IEF_lag3,0.63,-1.33,2.63,0.75,3.19,4.30
,(12.87),(5.02),(3.26),(4.60),(3.45),(6.12)
hl_range_IEF_lag4,-31.15***,-8.15*,-6.11**,-6.05,-5.45*,-7.25
,(11.86),(4.71),(2.84),(4.58),(3.03),(4.55)
Gender,-0.13,-0.22***,0.04,0.16***,0.03,0.11***
,(0.08),(0.04),(0.03),(0.03),(0.03),(0.04)


In [91]:
finVar = "volume"
finType = "SPX_index"
model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, sepNull=False, feffect = "role")
columnName = ["Abstract","Informativeness","Readability","Disunity","Strain","Strain (Upperbound)"]
indexName = ["fin1","fin2","fin3","fin4","Gender","Race","Major: Law","Major: Business","Major: Finance","Major: Other",
             "Not PhD","Pre-Fed: Academic","Pre-Fed: Finance","Pre-Fed: Business","Pre-Fed: Law","Pre-Fed: Other",
             "Freshwater","Other Economist","Age","Terms Time","gINDPRO","gINDPRO lag1","Infl",
             "Infl lag1","MICH","MICH lag1","Unemp","Unemp lag1","Topic: Community Dev","Topic: MonPol–Inflation","Topic: Real Econ–Prod",
             "Topic: Fed Ops–Payments","Topic: FinStab–Reg","Topic: International Econ", "Is Bank Pres","N","$R^2_{adj}$"]
res = model.tables[0]
res.columns = columnName
res.index = [elem for pair in zip(indexName[:-2], [""] * len(indexName[:-2])) for elem in pair][:-1]+[""]+indexName[-2:]
latex = df_to_latex(res)   # or whatever object you pass in
print(latex)


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Abstract} & \makecell{Informativeness} & \makecell{Readability} & \makecell{Disunity} & \makecell{Strain} & \makecell{Strain \\ (Upperbound)} \\
\midrule
fin1 & -0.00 & 0.00* & -0.00* & 0.00 & 0.00** & 0.00* \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin2 & -0.00 & -0.00 & -0.00 & -0.00 & -0.00* & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin3 & 0.00 & -0.00 & -0.00 & 0.00 & 0.00 & -0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin4 & -0.00* & 0.00 & -0.00 & -0.00 & 0.00 & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
Gender & -0.25*** & -0.19*** & 0.06 & -0.01 & -0.11*** & 0.13*** \\
 & (0.08) & (0.04) & (0.05) & (0.04) & (0.03) & (0.03) \\
Race & 0.27* & 0.06 & 0.74*** & 0.47*** & 0.28*** & 0.56*** \\
 & (0.14) & (0.06) & (0.12) & (0.08) & (0.06) & (0.05) \\
Major: Law & 0.30* & 0.11 & 0.21** & -0.15* & 0.09 & 0.14** \\
 & (0.16) & (0.07) & (0.11) & (0.08) 

In [93]:
outputList = speechVarList
controlList = speakerBG
controlList = controlList+ realAct3 +inflaty2 + mich + unemp + topicList

for finVar in ["volume","hl_range"]:
    for finType in ["SPX_index","IEF"]:
        model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, sepNull=False, feffect = "institution")
        model.tables[0].to_excel(f"results/reg1/RegG1robustSub{finVar+finType}Institution.xlsx")
        display(model)


,abstract,info,read,disunity,strain,strainUpper
volume_SPX_index_lag1,-0.00,0.00*,-0.00***,-0.00,0.00,0.00*
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag2,-0.00,-0.00,-0.00,0.00,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag3,-0.00,-0.00,-0.00*,0.00,0.00,-0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag4,-0.00**,0.00,-0.00,-0.00,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.28***,-0.21***,0.15***,-0.17***,-0.19***,0.19***
,(0.09),(0.04),(0.05),(0.04),(0.03),(0.04)


,abstract,info,read,disunity,strain,strainUpper
volume_IEF_lag1,-0.00***,-0.00,-0.00***,-0.00***,-0.00**,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag2,-0.00,-0.00**,0.00,0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag3,-0.00,-0.00,-0.00,-0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag4,-0.00***,-0.00,-0.00,-0.00**,-0.00,0.00*
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.46***,-0.13***,-0.12***,-0.14***,-0.18***,0.19***
,(0.10),(0.05),(0.03),(0.04),(0.03),(0.04)


,abstract,info,read,disunity,strain,strainUpper
hl_range_SPX_index_lag1,0.60,0.86,-9.05***,-2.40,-2.15*,-1.03
,(2.73),(1.60),(3.32),(1.58),(1.24),(1.23)
hl_range_SPX_index_lag2,9.16***,1.55,0.70,2.22,1.00,0.94
,(3.34),(2.77),(4.56),(2.03),(1.64),(1.68)
hl_range_SPX_index_lag3,-4.86,-3.62**,-5.27,-3.52**,-1.72,-2.76**
,(3.40),(1.83),(3.36),(1.78),(1.35),(1.40)
hl_range_SPX_index_lag4,-5.75*,2.34,-4.38,-2.21,-1.44,0.12
,(3.25),(1.55),(3.91),(1.99),(1.48),(1.55)
Gender,-0.19**,-0.21***,0.25***,-0.16***,-0.19***,0.16***
,(0.09),(0.04),(0.06),(0.05),(0.03),(0.04)


,abstract,info,read,disunity,strain,strainUpper
hl_range_IEF_lag1,10.19,-5.97,1.05,-7.62,-1.89,2.34
,(13.42),(5.33),(3.39),(5.53),(3.28),(4.76)
hl_range_IEF_lag2,25.90**,5.25,10.76***,2.98,0.50,3.26
,(12.17),(6.17),(3.17),(5.00),(3.14),(5.60)
hl_range_IEF_lag3,-0.36,-2.55,0.74,-0.67,0.50,2.27
,(12.51),(5.21),(3.00),(4.27),(3.21),(5.51)
hl_range_IEF_lag4,-32.66***,-8.63*,-6.28**,-5.11,-7.27**,-8.00*
,(11.93),(4.76),(2.76),(4.37),(2.92),(4.35)
Gender,-0.31***,-0.11**,-0.10***,-0.10**,-0.18***,0.17***
,(0.10),(0.05),(0.03),(0.04),(0.03),(0.04)


In [94]:
finVar = "volume"
finType = "SPX_index"
model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, sepNull=False, feffect = "institution")
columnName = ["Abstract","Informativeness","Readability","Disunity","Strain","Strain (Upperbound)"]
indexName = ["fin1","fin2","fin3","fin4","Gender","Race","Major: Law","Major: Business","Major: Finance","Major: Other",
             "Not PhD","Pre-Fed: Academic","Pre-Fed: Finance","Pre-Fed: Business","Pre-Fed: Law","Pre-Fed: Other",
             "Freshwater","Other Economist","Age","Terms Time","gINDPRO","gINDPRO lag1","Infl",
             "Infl lag1","MICH","MICH lag1","Unemp","Unemp lag1","Topic: Community Dev","Topic: MonPol–Inflation",
             "Topic: Real Econ–Prod", "Topic: Fed Ops–Payments","Topic: FinStab–Reg",
             "Topic: International Econ"]+["Institution: "+i for i in bankList]+["N","$R^2_{adj}$"]
res = model.tables[0]
res.columns = columnName
res.index = [elem for pair in zip(indexName[:-2], [""] * len(indexName[:-2])) for elem in pair][:-1]+[""]+indexName[-2:]
latex = df_to_latex(res)   # or whatever object you pass in
print(latex)


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Abstract} & \makecell{Informativeness} & \makecell{Readability} & \makecell{Disunity} & \makecell{Strain} & \makecell{Strain \\ (Upperbound)} \\
\midrule
fin1 & -0.00 & 0.00* & -0.00*** & -0.00 & 0.00 & 0.00* \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin2 & -0.00 & -0.00 & -0.00 & 0.00 & -0.00 & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin3 & -0.00 & -0.00 & -0.00* & 0.00 & 0.00 & -0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
fin4 & -0.00** & 0.00 & -0.00 & -0.00 & -0.00 & 0.00 \\
 & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) & (0.00) \\
Gender & -0.28*** & -0.21*** & 0.15*** & -0.17*** & -0.19*** & 0.19*** \\
 & (0.09) & (0.04) & (0.05) & (0.04) & (0.03) & (0.04) \\
Race & 1.23*** & 0.22** & 1.62*** & 1.35*** & 0.91*** & 0.59*** \\
 & (0.19) & (0.11) & (0.18) & (0.12) & (0.09) & (0.10) \\
Major: Law & 0.42** & 0.23*** & 0.25** & -0.29*** & 0.18*** & 0.33*** \\
 & (0.19) & (0

In [96]:
outputList = speechVarList
controlList = speakerBG
controlList = controlList+ realAct3 +inflaty2 + mich + unemp

for finVar in ["volume","hl_range"]:
    for finType in ["SPX_index","IEF"]:
        model = OLSFin(dfmain, finType, finVar, outputList, controlList, Ftest = False, sepNull=False, feffect = "institution")
        model.tables[0].to_excel(f"results/reg1/RegG1robustSub{finVar+finType}InstitutionNoTopic.xlsx")
        display(model)


,abstract,info,read,disunity,strain,strainUpper
volume_SPX_index_lag1,-0.00,0.00,-0.00**,-0.00,0.00,0.00**
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag2,0.00,-0.00,0.00,0.00,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag3,-0.00,-0.00,-0.00*,0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_SPX_index_lag4,-0.00**,-0.00,-0.00,-0.00,-0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.16,-0.12***,0.12**,-0.15***,-0.19***,0.17***
,(0.10),(0.04),(0.06),(0.05),(0.03),(0.04)


,abstract,info,read,disunity,strain,strainUpper
volume_IEF_lag1,-0.00***,-0.00,-0.00*,-0.00***,-0.00*,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag2,-0.00,-0.00**,0.00,0.00,0.00,0.00
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag3,-0.00*,0.00,-0.00,-0.00,0.00,0.00**
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
volume_IEF_lag4,-0.00***,-0.00,-0.00,-0.00**,-0.00,0.00*
,(0.00),(0.00),(0.00),(0.00),(0.00),(0.00)
Gender,-0.34***,-0.01,-0.22***,-0.16***,-0.18***,0.21***
,(0.12),(0.05),(0.04),(0.05),(0.03),(0.04)


,abstract,info,read,disunity,strain,strainUpper
hl_range_SPX_index_lag1,3.85,1.65,-8.22**,-1.82,-2.11*,-1.21
,(2.98),(1.60),(3.33),(1.67),(1.25),(1.26)
hl_range_SPX_index_lag2,12.30***,1.78,1.90,2.15,1.14,0.86
,(3.78),(2.83),(4.57),(2.11),(1.68),(1.74)
hl_range_SPX_index_lag3,-3.49,-3.55*,-5.02,-3.42*,-1.73,-3.33**
,(3.57),(1.88),(3.42),(1.81),(1.37),(1.46)
hl_range_SPX_index_lag4,-5.24,2.20,-3.89,-2.80,-1.52,-0.53
,(3.50),(1.58),(3.95),(2.08),(1.50),(1.61)
Gender,-0.03,-0.11***,0.22***,-0.14***,-0.19***,0.12***
,(0.10),(0.04),(0.06),(0.05),(0.04),(0.04)


,abstract,info,read,disunity,strain,strainUpper
hl_range_IEF_lag1,25.24*,-1.39,0.89,-8.31,-2.22,-2.14
,(13.68),(5.53),(4.24),(5.97),(3.38),(5.20)
hl_range_IEF_lag2,47.43***,8.59,16.56***,5.42,1.59,3.16
,(14.12),(6.16),(4.43),(5.29),(3.24),(5.85)
hl_range_IEF_lag3,8.82,1.87,2.52,2.86,1.53,3.29
,(13.49),(5.71),(3.95),(4.68),(3.46),(6.02)
hl_range_IEF_lag4,-22.94,-7.19,-3.26,-3.89,-6.94**,-11.78**
,(14.44),(4.79),(4.27),(5.10),(3.21),(4.63)
Gender,-0.15,0.03,-0.20***,-0.11**,-0.18***,0.15***
,(0.12),(0.05),(0.04),(0.05),(0.03),(0.04)


### Anova analysis

- **Type II** (`typ=2`) is best when your regressors are not orthogonal but you don’t have interactions.  
- **Type III** (`typ=3`) is used when you have categorical variables and want to test each variable *conditional on all others*.

For your design (categorical backgrounds + continuous controls), **Type II** is usually the cleanest.

In [82]:
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

def anova_group_ss(df, yvar, speaker_vars, financial_vars, macro_vars, topics = False, topic_vars = []):
    # Build formula
    if topics:
        all_vars = speaker_vars + financial_vars + macro_vars + topic_vars
    else:
        all_vars = speaker_vars + financial_vars + macro_vars
    formula = yvar + " ~ " + " + ".join(all_vars)
    
    # Fit model
    model = ols(formula, data=df).fit()
    
    # ANOVA
    anova_res = anova_lm(model, typ=2)
    
    # Extract group sums of squares
    speaker_ss = anova_res.loc[speaker_vars, 'sum_sq'].sum()
    financial_ss = anova_res.loc[financial_vars, 'sum_sq'].sum()
    macro_ss = anova_res.loc[macro_vars, 'sum_sq'].sum()
    if topics:
        topic_ss = anova_res.loc[topic_vars, 'sum_sq'].sum()
    
    total_ss = anova_res['sum_sq'].sum()
    
    speaker_pct = f"{(speaker_ss / total_ss) * 100:.2f}\%"
    financial_pct = f"{(financial_ss / total_ss) * 100:.2}\%"
    macro_pct = f"{(macro_ss / total_ss) * 100:.2f}\%"
    if topics:
        topic_pct = f"{(topic_ss / total_ss) * 100:.2f}\%"
    
        return {
        'speaker_pct': speaker_pct,
        'financial_pct': financial_pct,
        'macro_pct': macro_pct,
        'topic_pct': topic_pct
        }
    else:
        return {
        'speaker_pct': speaker_pct,
        'financial_pct': financial_pct,
        'macro_pct': macro_pct
        }



#### Testing

In [64]:
finVar = "volume"
finName = "SPX_index"
financial_vars = [finVar+"_"+finName+"_lag"+str(i+1) for i in range(4)]
macro_vars = realAct3+inflaty2+unemp+mich # macroVarGTList
desired_order = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                 'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                 'Kansas', 'Minneapolis']
dfmain["District"] = pd.Categorical(dfmain["District"], categories=desired_order)
dummies = pd.get_dummies(dfmain["District"], prefix="inst", drop_first=True).astype(int)
dfmain = pd.concat([dfmain, dummies], axis=1)
speaker_vars = speakerBG + ["inst_"+var for var in desired_order[1:]]
all_vars = speaker_vars + financial_vars + macro_vars
yvar = "abstract"

# Build formula string
formula = yvar + " ~ " + " + ".join(all_vars)

print(formula)



abstract ~ Gender + Race + Major_Law + Major_Business + Major_Fin + Major_Other + Degree_Other + PreFed_clean_AcademicEcon + PreFed_clean_Finance + PreFed_clean_Business + PreFed_clean_Law + PreFed_clean_Other + Freshwater + SaltFresh_Other + Age + yearsIn + inst_NewYork + inst_Atlanta + inst_SanFrancisco + inst_Chicago + inst_Cleveland + inst_Philadelphia + inst_Richmond + inst_StLouis + inst_Dallas + inst_Boston + inst_Kansas + inst_Minneapolis + volume_SPX_index_lag1 + volume_SPX_index_lag2 + volume_SPX_index_lag3 + volume_SPX_index_lag4 + INDPRO_change + INDPRO_change_lag1 + CPIAUCSL_yoy + CPIAUCSL_yoy_lag1 + UNRATE + UNRATE_lag1 + MICH + MICH_lag1


In [80]:
model = ols(formula, data=dfmain).fit()

anova_results = anova_lm(model, typ=2)   # Type II ANOVA recommended
anova_results


C:\Users\Lynn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning:covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
C:\Users\Lynn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning:covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
C:\Users\Lynn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning:covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
C:\Users\Lynn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning:covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
C:\Users\Lynn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning:covariance of constraints does not have full rank. The number of constraints is 3, but rank is 1
C:\Users\Lynn\anaconda3\Lib\site-packages\statsmodels\base\model.py:1894: ValueW

,sum_sq,df,F,PR(>F)
Gender,15.034696,1.0,3.841053,5.006195e-02
Race,270.581768,1.0,69.128037,1.148847e-16
Major_Law,29.699851,1.0,7.587697,5.895909e-03
Major_BFin,19.374334,1.0,4.949741,2.613454e-02
Major_Other,13.793085,1.0,3.523848,6.054416e-02
Degree_Other,0.119797,1.0,0.030606,8.611293e-01
PreFed_clean_AcademicEcon,35.769514,1.0,9.138370,2.514448e-03
PreFed_clean_BFin,12.136688,1.0,3.100672,7.831491e-02
PreFed_clean_Law,52.110627,1.0,13.313186,2.659612e-04
PreFed_clean_Other,81.147871,1.0,20.731600,5.396346e-06


In [82]:
# Add a column for % of explained variance
anova_results['pct_explained'] = (
    anova_results['sum_sq'] / anova_results['sum_sq'].sum()
)


In [84]:
def group_ss(group_list):
    return anova_results.loc[group_list, 'sum_sq'].sum()

speaker_ss = group_ss(speaker_vars)
financial_ss = group_ss(financial_vars)
macro_ss = group_ss(macro_vars)

total_ss = anova_results['sum_sq'].sum()

print("Speaker backgrounds:", speaker_ss / total_ss*100)
print("Financial variables:", financial_ss / total_ss*100)
print("Macro controls:", macro_ss / total_ss*100)


Speaker backgrounds: 24.560560613319492
Financial variables: 0.09095751566834281
Macro controls: 0.09627612590789851


In [169]:
# duplication cause problems
# Combine all regressors
macro_vars = realAct1+realAct2+realAct3+inflat1+inflat2+inflaty1+inflaty2+unemp+mich # macroVarGTList
desired_order = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                 'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                 'Kansas', 'Minneapolis']
dfmain["District"] = pd.Categorical(dfmain["District"], categories=desired_order)
dummies = pd.get_dummies(dfmain["District"], prefix="inst", drop_first=True).astype(int)
dfmain = pd.concat([dfmain, dummies], axis=1)
speaker_vars = speakerBG + ["inst_"+var for var in desired_order[1:]]

finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]



In [171]:
anova_tables = {}
for finType in finTypeList[:1]: 
    financial_vars = []
    for finVar in finVarList[:1]: 
        financial_vars = financial_vars+[finVar+"_"+finType+"_lag"+str(i+1) for i in range(4)]
    results = {}
    for yvar in speechVarList:
        results[yvar] = anova_group_ss(dfmain, yvar, speaker_vars, financial_vars, macro_vars, topics=True, topic_vars=topicList)
    anova_tables[finType] = pd.DataFrame(results).reset_index().rename(columns={'index': 'group'})

for name, tbl in anova_tables.items():
    tbl['equity'] = name
big_table = pd.concat(anova_tables.values(), axis=0)
big_table.index = big_table['equity']
big_table = big_table.drop(columns = ['equity'])
big_table

,group,abstract,info,read,disunity,strain,strainUpper
equity,,,,,,,
SPX_index,speaker_pct,16.396%,33.569%,20.563%,56.366%,63.978%,56.400%
SPX_index,financial_pct,0.025%,0.037%,0.018%,0.033%,0.015%,0.023%
SPX_index,macro_pct,0.656%,0.272%,0.186%,0.133%,0.044%,0.219%
SPX_index,topic_pct,24.780%,3.005%,5.104%,13.299%,5.842%,2.005%


#### Main

In [84]:
# Combine all regressors
macro_vars = realAct1+realAct2+realAct3+inflat1+inflat2+inflaty1+inflaty2+unemp+mich # macroVarGTList
if not "inst_NewYork" in dfmain.columns:
    desired_order = ['BoardGovernors', 'NewYork', 'Atlanta', 'SanFrancisco', 'Chicago',
                     'Cleveland', 'Philadelphia', 'Richmond', 'StLouis', 'Dallas', 'Boston',
                     'Kansas', 'Minneapolis']
    dfmain["District"] = pd.Categorical(dfmain["District"], categories=desired_order)
    dummies = pd.get_dummies(dfmain["District"], prefix="inst", drop_first=True).astype(int)
    dfmain = pd.concat([dfmain, dummies], axis=1)
speaker_vars = speakerBG + ["inst_"+var for var in desired_order[1:]]

finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

anova_tables = {}
for finType in finTypeList: 
    financial_vars = []
    for finVar in finVarList: 
        financial_vars = financial_vars+[finVar+"_"+finType+"_lag"+str(i+1) for i in range(4)]
    results = {}
    for yvar in speechVarList:
        results[yvar] = anova_group_ss(dfmain, yvar, speaker_vars, financial_vars, macro_vars, topics=True, topic_vars=topicList)
    anova_tables[finType] = pd.DataFrame(results).reset_index().rename(columns={'index': 'group'})

for name, tbl in anova_tables.items():
    tbl['equity'] = name
big_table = pd.concat(anova_tables.values(), axis=0)
big_table.index = big_table['equity']
big_table = big_table.drop(columns = ['equity'])
big_table.to_csv("results/reg1/RegG1Anova1.csv")
big_table


,group,abstract,info,read,disunity,strain,strainUpper
equity,,,,,,,
SPX_index,speaker_pct,5.15\%,5.20\%,8.75\%,19.30\%,21.22\%,11.59\%
SPX_index,financial_pct,0.75\%,0.14\%,0.3\%,0.15\%,0.18\%,0.27\%
SPX_index,macro_pct,0.73\%,0.46\%,0.32\%,0.21\%,0.17\%,0.45\%
SPX_index,topic_pct,3.50\%,1.86\%,0.56\%,11.28\%,1.07\%,3.43\%
ES_futures,speaker_pct,7.27\%,10.39\%,20.90\%,15.03\%,28.94\%,17.70\%
ES_futures,financial_pct,0.46\%,0.9\%,0.15\%,0.31\%,0.28\%,0.73\%
ES_futures,macro_pct,1.43\%,0.93\%,0.50\%,0.29\%,0.65\%,0.99\%
ES_futures,topic_pct,3.91\%,2.05\%,38.05\%,25.08\%,2.89\%,6.37\%
SHY,speaker_pct,6.76\%,11.28\%,19.38\%,12.49\%,25.77\%,17.62\%


In [86]:
nameList = ['speaker','finance','macro','topics']
big_table['group'] = big_table['group'].map(dict(zip(big_table['group'].value_counts().index.tolist(),nameList)))
nameList = ["SPY","ES"]+big_table.index.value_counts().index.tolist()[2:]
nameTotal = []
for var in nameList:
    nameTotal += [var]+[""]*3
big_table.index = nameTotal
columnName = ["Group","Abstract","Informativeness","Readability","Disunity","Strain","Strain (Upperbound)"]
big_table.columns = columnName
latex = df_to_latex(big_table)
print(latex)


\begin{tabular}{lrrrrrrr}
\toprule
 & \makecell{Group} & \makecell{Abstract} & \makecell{Informativeness} & \makecell{Readability} & \makecell{Disunity} & \makecell{Strain} & \makecell{Strain \\ (Upperbound)} \\
\midrule
SPY & speaker & 5.15\% & 5.20\% & 8.75\% & 19.30\% & 21.22\% & 11.59\% \\
 & finance & 0.75\% & 0.14\% & 0.3\% & 0.15\% & 0.18\% & 0.27\% \\
 & macro & 0.73\% & 0.46\% & 0.32\% & 0.21\% & 0.17\% & 0.45\% \\
 & topics & 3.50\% & 1.86\% & 0.56\% & 11.28\% & 1.07\% & 3.43\% \\
ES & speaker & 7.27\% & 10.39\% & 20.90\% & 15.03\% & 28.94\% & 17.70\% \\
 & finance & 0.46\% & 0.9\% & 0.15\% & 0.31\% & 0.28\% & 0.73\% \\
 & macro & 1.43\% & 0.93\% & 0.50\% & 0.29\% & 0.65\% & 0.99\% \\
 & topics & 3.91\% & 2.05\% & 38.05\% & 25.08\% & 2.89\% & 6.37\% \\
SHY & speaker & 6.76\% & 11.28\% & 19.38\% & 12.49\% & 25.77\% & 17.62\% \\
 & finance & 1.1\% & 0.8\% & 0.41\% & 0.42\% & 0.59\% & 0.83\% \\
 & macro & 1.35\% & 0.70\% & 0.61\% & 0.35\% & 0.86\% & 1.22\% \\
 & topics & 4.08\% 

In [88]:
# green/tealbook?
macro_vars = macroVarGTList1 

anova_tables = {}
for finType in finTypeList: 
    financial_vars = []
    for finVar in finVarList: 
        financial_vars = financial_vars+[finVar+"_"+finType+"_lag"+str(i+1) for i in range(4)]
    results = {}
    for yvar in speechVarList:
        results[yvar] = anova_group_ss(dfmain, yvar, speaker_vars, financial_vars, macro_vars, topics=True, topic_vars=topicList)
    anova_tables[finType] = pd.DataFrame(results).reset_index().rename(columns={'index': 'group'})

for name, tbl in anova_tables.items():
    tbl['equity'] = name
big_table = pd.concat(anova_tables.values(), axis=0)
big_table.index = big_table['equity']
big_table = big_table.drop(columns = ['equity'])
big_table.to_csv("results/reg1/RegG1Anova2.csv")
big_table
        

,group,abstract,info,read,disunity,strain,strainUpper
equity,,,,,,,
SPX_index,speaker_pct,6.02\%,5.70\%,7.82\%,17.88\%,19.17\%,11.21\%
SPX_index,financial_pct,0.61\%,0.16\%,0.27\%,0.21\%,0.17\%,0.29\%
SPX_index,macro_pct,0.68\%,0.28\%,0.69\%,0.43\%,0.44\%,0.36\%
SPX_index,topic_pct,2.89\%,1.77\%,0.41\%,9.83\%,0.83\%,3.16\%
ES_futures,speaker_pct,7.83\%,9.88\%,20.52\%,17.25\%,27.54\%,16.66\%
ES_futures,financial_pct,0.44\%,0.77\%,0.1\%,0.32\%,0.39\%,0.79\%
ES_futures,macro_pct,0.95\%,0.20\%,0.38\%,0.59\%,0.80\%,0.28\%
ES_futures,topic_pct,3.42\%,2.16\%,36.11\%,22.86\%,2.18\%,6.00\%
SHY,speaker_pct,7.95\%,10.53\%,19.33\%,14.91\%,24.35\%,16.16\%


In [89]:
nameList = ['speaker','finance','macro','topics']
big_table['group'] = big_table['group'].map(dict(zip(big_table['group'].value_counts().index.tolist(),nameList)))
nameList = ["SPY","ES"]+big_table.index.value_counts().index.tolist()[2:]
nameTotal = []
for var in nameList:
    nameTotal += [var]+[""]*3
big_table.index = nameTotal
columnName = ["Group","Abstract","Informativeness","Readability","Disunity","Strain","Strain (Upperbound)"]
big_table.columns = columnName
latex = df_to_latex(big_table)
print(latex)

\begin{tabular}{lrrrrrrr}
\toprule
 & \makecell{Group} & \makecell{Abstract} & \makecell{Informativeness} & \makecell{Readability} & \makecell{Disunity} & \makecell{Strain} & \makecell{Strain \\ (Upperbound)} \\
\midrule
SPY & speaker & 6.02\% & 5.70\% & 7.82\% & 17.88\% & 19.17\% & 11.21\% \\
 & finance & 0.61\% & 0.16\% & 0.27\% & 0.21\% & 0.17\% & 0.29\% \\
 & macro & 0.68\% & 0.28\% & 0.69\% & 0.43\% & 0.44\% & 0.36\% \\
 & topics & 2.89\% & 1.77\% & 0.41\% & 9.83\% & 0.83\% & 3.16\% \\
ES & speaker & 7.83\% & 9.88\% & 20.52\% & 17.25\% & 27.54\% & 16.66\% \\
 & finance & 0.44\% & 0.77\% & 0.1\% & 0.32\% & 0.39\% & 0.79\% \\
 & macro & 0.95\% & 0.20\% & 0.38\% & 0.59\% & 0.80\% & 0.28\% \\
 & topics & 3.42\% & 2.16\% & 36.11\% & 22.86\% & 2.18\% & 6.00\% \\
SHY & speaker & 7.95\% & 10.53\% & 19.33\% & 14.91\% & 24.35\% & 16.16\% \\
 & finance & 0.61\% & 0.86\% & 0.32\% & 0.12\% & 0.48\% & 0.97\% \\
 & macro & 0.65\% & 0.21\% & 0.29\% & 0.48\% & 0.81\% & 0.44\% \\
 & topics & 3.82\%

In [90]:
# check duplicate column
dfmain.columns.duplicated().any()


False

### Second group of regressions

In [94]:
finTypeList = ["SPX_index","ES_futures","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
finVarList = ["ret", "abs_ret", "hl_range", "volume", "abn_ret_5", "abn_ret_22"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']
             
otherList = ['is_monday', 'is_friday','Chair']+topicList


In [97]:
inputList = speechVarList+realAct3+inflaty2+unemp+mich+otherList
finNameList = finTypeList
indexName = ["SPX","ES","SHY", "IEF", "TLT","UUP","VXX","VIX","GLD"]
columnName = ["Return","Absolute Return","High-low Range","Volume","AbnormalReturn (5days)","AbnormalReturn (22days)"]

res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True, sepNull = True)
res.to_csv("results/reg2/RegG2main.csv")
res.index = indexName
res.columns = columnName
latex = df_to_latex(res)
display(res)
print(latex)

res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True)
res.to_csv("results/reg2/RegG2robustSign.csv")
res.index = indexName
res.columns = columnName
latex = df_to_latex(res)
display(res)
print(latex)



['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper']


,Return,Absolute Return,High-low Range,Volume,AbnormalReturn (5days),AbnormalReturn (22days)
SPX,0,***,***,***,0,0
ES,0,0,0,***,0,0
SHY,0,***,***,***,0,0
IEF,*,***,**,***,0,0
TLT,0,***,***,***,0,0
UUP,0,0,**,0,0,0
VXX,0,0,*,***,0,0
VIX,0,0,***,error,0,0
GLD,0,0,0,***,0,0


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Return} & \makecell{Absolute \\ Return} & \makecell{High-low \\ Range} & \makecell{Volume} & \makecell{AbnormalReturn \\ (5days)} & \makecell{AbnormalReturn \\ (22days)} \\
\midrule
SPX & 0 & *** & *** & *** & 0 & 0 \\
ES & 0 & 0 & 0 & *** & 0 & 0 \\
SHY & 0 & *** & *** & *** & 0 & 0 \\
IEF & * & *** & ** & *** & 0 & 0 \\
TLT & 0 & *** & *** & *** & 0 & 0 \\
UUP & 0 & 0 & ** & 0 & 0 & 0 \\
VXX & 0 & 0 & * & *** & 0 & 0 \\
VIX & 0 & 0 & *** & error & 0 & 0 \\
GLD & 0 & 0 & 0 & *** & 0 & 0 \\
\bottomrule
\end{tabular}



,Return,Absolute Return,High-low Range,Volume,AbnormalReturn (5days),AbnormalReturn (22days)
SPX,0,--,--,++,0,0
ES,0,-,--,-,0,0
SHY,0,--,0,---,0,0
IEF,0,---,---,---,0,0
TLT,0,---,---,---,0,0
UUP,0,0,0,--,0,0
VXX,0,-,--,+,0,0
VIX,0,0,0,0,0,0
GLD,0,-,0,---,0,0


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Return} & \makecell{Absolute \\ Return} & \makecell{High-low \\ Range} & \makecell{Volume} & \makecell{AbnormalReturn \\ (5days)} & \makecell{AbnormalReturn \\ (22days)} \\
\midrule
SPX & 0 & -- & -- & ++ & 0 & 0 \\
ES & 0 & - & -- & - & 0 & 0 \\
SHY & 0 & -- & 0 & --- & 0 & 0 \\
IEF & 0 & --- & --- & --- & 0 & 0 \\
TLT & 0 & --- & --- & --- & 0 & 0 \\
UUP & 0 & 0 & 0 & -- & 0 & 0 \\
VXX & 0 & - & -- & + & 0 & 0 \\
VIX & 0 & 0 & 0 & 0 & 0 & 0 \\
GLD & 0 & - & 0 & --- & 0 & 0 \\
\bottomrule
\end{tabular}



In [100]:
finNameList = finTypeList

inputList = speechVarList+realAct3+inflaty1+unemp+mich+otherList
res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True, sepNull = True)
res.to_csv("results/reg2/RegG2robustMacro1.csv")
display(res)

inputList = speechVarList+realAct2+inflaty2+unemp+mich+otherList
res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True, sepNull = True)
res.to_csv("results/reg2/RegG2robustMacro2.csv")
display(res)

inputList = speechVarList+macroVarGTList+otherList
res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = True, sepNull = True)
res.to_csv("results/reg2/RegG2robustMacro3.csv")
res.index = indexName
res.columns = columnName
latex = df_to_latex(res)
display(res)
print(latex)


['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper']


,ret,abs_ret,hl_range,volume,abn_ret_5,abn_ret_22
SPX_index,0,***,***,***,0,0
ES_futures,0,0,0,***,0,0
SHY,0,***,***,***,0,0
IEF,0,**,0,***,0,0
TLT,0,***,***,***,0,0
UUP,0,0,**,0,0,0
VXX,0,*,**,0,0,0
VIX,0,0,***,error,0,0
GLD,0,0,*,***,0,0


['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper']


,ret,abs_ret,hl_range,volume,abn_ret_5,abn_ret_22
SPX_index,0,***,***,***,0,0
ES_futures,0,0,**,***,0,0
SHY,0,***,***,***,0,0
IEF,*,***,***,***,0,0
TLT,0,***,***,***,0,0
UUP,0,0,**,0,0,0
VXX,0,0,*,***,0,0
VIX,0,0,***,error,0,0
GLD,0,0,*,***,0,0


['abstract', 'info', 'read', 'disunity', 'strain', 'strainUpper']


,Return,Absolute Return,High-low Range,Volume,AbnormalReturn (5days),AbnormalReturn (22days)
SPX,0,***,***,***,0,0
ES,0,0,*,***,0,0
SHY,0,**,*,***,0,0
IEF,0,***,***,***,0,0
TLT,0,***,***,***,0,0
UUP,0,0,0,***,0,0
VXX,**,0,0,0,0,***
VIX,0,0,***,error,0,0
GLD,0,*,0,***,0,0


\begin{tabular}{lrrrrrr}
\toprule
 & \makecell{Return} & \makecell{Absolute \\ Return} & \makecell{High-low \\ Range} & \makecell{Volume} & \makecell{AbnormalReturn \\ (5days)} & \makecell{AbnormalReturn \\ (22days)} \\
\midrule
SPX & 0 & *** & *** & *** & 0 & 0 \\
ES & 0 & 0 & * & *** & 0 & 0 \\
SHY & 0 & ** & * & *** & 0 & 0 \\
IEF & 0 & *** & *** & *** & 0 & 0 \\
TLT & 0 & *** & *** & *** & 0 & 0 \\
UUP & 0 & 0 & 0 & *** & 0 & 0 \\
VXX & ** & 0 & 0 & 0 & 0 & *** \\
VIX & 0 & 0 & *** & error & 0 & 0 \\
GLD & 0 & * & 0 & *** & 0 & 0 \\
\bottomrule
\end{tabular}



In [102]:
# details
inputList = speechVarList+realAct3+inflaty2+unemp+mich+otherList
finNameList = finTypeList

res = OLSFin2(dfmain, finNameList, finVarList, inputList, Ftest = False, sepNull = False)



                               ret_SPX_index_fut1 ret_ES_futures_fut1 ret_SHY_fut1 ret_IEF_fut1 ret_TLT_fut1 ret_UUP_fut1 ret_VXX_fut1 ret_VIX_fut1 ret_GLD_fut1
----------------------------------------------------------------------------------------------------------------------------------------------------------------
abstract                       0.00               0.00                0.00         0.00         0.00         -0.00        -0.00        0.00         0.00        
                               (0.00)             (0.00)              (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)      
info                           0.00               0.00                0.00         0.00         -0.00        0.00         -0.00        -0.00        -0.00       
                               (0.00)             (0.00)              (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)       (0.00)      
read                           0.

### Third group: Intraday Regressions

In [296]:
windowList = [15,30,45,60]
finTypeList = ["SPY_BB","SHY_BB", "IEF_BB", "TLT_BB","VIX_BB"]
finVarList = ["ret", "absret", "range","vol"]

realAct1 = ['GDPC1_change', 'GDPC1_change_lag1']
realAct2 = ['GDPC1_low_q10', 'GDPC1_low_q10_lag1']
realAct3 = ['INDPRO_change', 'INDPRO_change_lag1']
inflat1 = ['PCEPILFE_change', 'PCEPILFE_change_lag1']
inflat2 = ['CPIAUCSL_change', 'CPIAUCSL_change_lag1']
inflaty1 = ['PCEPILFE_yoy', 'PCEPILFE_yoy_lag1']
inflaty2 = ['CPIAUCSL_yoy', 'CPIAUCSL_yoy_lag1']
mich = ['MICH', 'MICH_lag1']
unemp = ['UNRATE', 'UNRATE_lag1']

macroVarGTList = ['gRGDPB1', 'gRGDPF0', 'gRGDPF1', 
                  'gPGDPB1', 'gPGDPF0', 'gPGDPF1', 
                  'UNEMPB1', 'UNEMPF0', 'UNEMPF1', 
                  'HSTARTB1', 'HSTARTF0', 'HSTARTF1']
             
otherList = ['is_monday', 'is_friday','Chair','tod_midday','tod_morning']+topicList


In [308]:
finNameList = finTypeList[:-1]
inputList = speechVarList+otherList #+realAct3+inflaty2+unemp+mich
indexName = ["SPY","SHY","IEF","TLT","VIX"]

res1 = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = False)
res2 = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = False)
res = pd.concat([res1,res2])
res.to_csv("results/reg3/RegG3robustSign.csv")
res.index = indexName
latex = df_to_latex_intra(res)
display(res)
print(latex)

res1 = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = True)
res2 = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = True, Ftest = True, sepNull = True)
res = pd.concat([res1,res2])
res.to_csv("results/reg3/RegG3main.csv")
res.index = indexName
latex = df_to_latex_intra(res)
display(res)
print(latex)



,ret15,ret30,ret45,ret60,absret15,absret30,absret45,absret60,range15,range30,range45,range60,vol15,vol30,vol45,vol60
SPY,0,0,0,0,0,0,0,+,0,0,0,--,0,---,--,---
SHY,0,0,--,-,0,0,0,0,0,0,+,0,0,--,--,---
IEF,0,0,---,0,+++,++,++,++,++,0,0,0,0,---,--,--
TLT,0,0,--,0,+++,++,0,0,+++,++,++,0,0,--,0,---
VIX,0,0,0,0,0,0,0,0,0,0,0,0,NaN,NaN,NaN,NaN



    \begin{tabular}{lcccccccccccccccc}
    \toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\
    \midrule
    
SPY & 0 & 0 & 0 & 0 & 0 & 0 & 0 & + & 0 & 0 & 0 & -- & 0 & --- & -- & --- \\
SHY & 0 & 0 & -- & - & 0 & 0 & 0 & 0 & 0 & 0 & + & 0 & 0 & -- & -- & --- \\
IEF & 0 & 0 & --- & 0 & +++ & ++ & ++ & ++ & ++ & 0 & 0 & 0 & 0 & --- & -- & -- \\
TLT & 0 & 0 & -- & 0 & +++ & ++ & 0 & 0 & +++ & ++ & ++ & 0 & 0 & -- & 0 & --- \\
VIX & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & NaN & NaN & NaN & NaN \\

    \bottomrule
    \end{tabular}
    


,ret15,ret30,ret45,ret60,absret15,absret30,absret45,absret60,range15,range30,range45,range60,vol15,vol30,vol45,vol60
SPY,0,0,0,0,0,0,***,0,0,0,*,**,0,***,**,***
SHY,0,0,*,0,*,0,0,0,**,0,0,0,0,**,0,**
IEF,0,0,0,0,***,0,0,*,**,0,**,0,0,***,**,***
TLT,0,0,0,0,*,0,0,0,*,0,**,0,0,**,0,***
VIX,0,0,0,0,0,0,0,*,0,**,0,0,NaN,NaN,NaN,NaN



    \begin{tabular}{lcccccccccccccccc}
    \toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\
    \midrule
    
SPY & 0 & 0 & 0 & 0 & 0 & 0 & *** & 0 & 0 & 0 & * & ** & 0 & *** & ** & *** \\
SHY & 0 & 0 & * & 0 & * & 0 & 0 & 0 & ** & 0 & 0 & 0 & 0 & ** & 0 & ** \\
IEF & 0 & 0 & 0 & 0 & *** & 0 & 0 & * & ** & 0 & ** & 0 & 0 & *** & ** & *** \\
TLT & 0 & 0 & 0 & 0 & * & 0 & 0 & 0 & * & 0 & ** & 0 & 0 & ** & 0 & *** \\
VIX & 0 & 0 & 0 & 0 & 0 & 0 & 0 & * & 0 & ** & 0 & 0 & NaN & NaN & NaN & NaN \\

    \bottomrule
    \end{tabular}
    


In [310]:
res1 = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = False, Ftest = True, sepNull = False)
res2 = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = False, Ftest = True, sepNull = False)
res = pd.concat([res1,res2])
res.to_csv("results/reg3/RegG3robustSymmetrySign.csv")
res.index = indexName
latex = df_to_latex_intra(res)
display(res)
print(latex)

res1 = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = False, Ftest = True, sepNull = True)
res2 = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = False, Ftest = True, sepNull = True)
res = pd.concat([res1,res2])
res.to_csv("results/reg3/RegG3robustSymmetry.csv")
res.index = indexName
latex = df_to_latex_intra(res)
display(res)
print(latex)


,ret15,ret30,ret45,ret60,absret15,absret30,absret45,absret60,range15,range30,range45,range60,vol15,vol30,vol45,vol60
SPY,0,0,0,0,0,0,0,0,--,0,0,--,0,---,---,---
SHY,0,0,0,0,0,0,0,0,0,0,0,0,0,---,-,---
IEF,0,+,0,0,0,-,0,0,+++,0,0,0,0,0,0,---
TLT,0,++,0,0,0,-,0,0,++,0,0,-,0,---,0,---
VIX,0,0,0,0,-,0,0,0,0,-,-,--,NaN,NaN,NaN,NaN



    \begin{tabular}{lcccccccccccccccc}
    \toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\
    \midrule
    
SPY & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & -- & 0 & 0 & -- & 0 & --- & --- & --- \\
SHY & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & --- & - & --- \\
IEF & 0 & + & 0 & 0 & 0 & - & 0 & 0 & +++ & 0 & 0 & 0 & 0 & 0 & 0 & --- \\
TLT & 0 & ++ & 0 & 0 & 0 & - & 0 & 0 & ++ & 0 & 0 & - & 0 & --- & 0 & --- \\
VIX & 0 & 0 & 0 & 0 & - & 0 & 0 & 0 & 0 & - & - & -- & NaN & NaN & NaN & NaN \\

    \bottomrule
    \end{tabular}
    


,ret15,ret30,ret45,ret60,absret15,absret30,absret45,absret60,range15,range30,range45,range60,vol15,vol30,vol45,vol60
SPY,0,0,0,0,0,0,0,0,0,0,0,***,**,***,***,***
SHY,0,0,0,0,**,0,0,0,0,0,0,0,0,**,0,**
IEF,0,0,0,0,0,*,0,0,***,0,0,0,*,0,*,**
TLT,0,0,0,0,*,0,0,0,*,0,0,0,0,**,0,***
VIX,0,0,0,0,0,0,0,0,0,0,0,***,NaN,NaN,NaN,NaN



    \begin{tabular}{lcccccccccccccccc}
    \toprule
    & \multicolumn{4}{c}{Return} 
    & \multicolumn{4}{c}{Absolute Return} 
    & \multicolumn{4}{c}{High-low Range} 
    & \multicolumn{4}{c}{Volume} \\
    \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}
    & 15m & 30m & 45m & 60m 
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m
    & 15m & 30m & 45m & 60m \\
    \midrule
    
SPY & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & *** & ** & *** & *** & *** \\
SHY & 0 & 0 & 0 & 0 & ** & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & ** & 0 & ** \\
IEF & 0 & 0 & 0 & 0 & 0 & * & 0 & 0 & *** & 0 & 0 & 0 & * & 0 & * & ** \\
TLT & 0 & 0 & 0 & 0 & * & 0 & 0 & 0 & * & 0 & 0 & 0 & 0 & ** & 0 & *** \\
VIX & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & *** & NaN & NaN & NaN & NaN \\

    \bottomrule
    \end{tabular}
    


In [105]:
# the details
res = OLSFin3(dfmain, finNameList, finVarList, inputList, windowList, 
        windowAsym = True, Ftest = False, sepNull = False)

res = OLSFin3(dfmain, ["VIX_BB"], finVarList[:-1], inputList, windowList, 
        windowAsym = True, Ftest = False, sepNull = False)



                               SPY_BB_ret15 SPY_BB_ret30 SPY_BB_ret45 SPY_BB_ret60 SPY_BB_absret15 SPY_BB_absret30 SPY_BB_absret45 SPY_BB_absret60 SPY_BB_range15 SPY_BB_range30 SPY_BB_range45 SPY_BB_range60  SPY_BB_vol15   SPY_BB_vol30   SPY_BB_vol45   SPY_BB_vol60 
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
abstract                       -0.00        -0.00        -0.00        -0.00*       -0.00           -0.00*          -0.00           -0.00           -0.01          -0.01          0.00           -0.01          -67533.39**    -71816.50*     -18164.40      20130.56      
                               (0.00)       (0.00)       (0.00)       (0.00)       (0.00)          (0.00)          (0.00)          (0.00)          (0.01)         (0.01)         (0.01)         (0.01)

### Anything else?

In [54]:
# topic shares complexity relation
topic_cols = topicList
complexity_cols = speechVarList

# Long format
long = dfmain[topic_cols+complexity_cols].melt(
    id_vars=complexity_cols,
    value_vars=topic_cols,
    var_name="topic",
    value_name="topic_share"
)

# Weighted average complexity for each topic
summary = (
    long.groupby("topic")
        .apply(lambda g: (g[complexity_cols].T @ g["topic_share"]) / g["topic_share"].sum())
        .T
)
summary.round(3).to_csv("results/others/complexityTopics.csv")
summary.round(3)


C:\Users\Lynn\AppData\Local\Temp\ipykernel_9752\498963724.py:16: DeprecationWarning:DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


topic,community_development,fed_operations_payments,financial_stability_regulation,international_economics,monetary_policy_inflation,real_economy_productivity
abstract,-0.444,-0.015,1.152,0.437,-0.414,-0.525
info,-0.372,0.116,0.172,-0.033,-0.014,-0.102
read,-0.174,0.029,0.611,-0.067,-0.144,-0.259
disunity,-0.229,0.215,0.433,0.118,0.065,-0.314
strain,-0.088,0.119,0.253,0.017,0.032,-0.283
strainUpper,-0.057,0.412,0.118,-0.173,0.039,-0.175


In [55]:
print(df_to_latex(summary))

\begin{tabular}{lrrrrrr}
\toprule
topic & \makecell{community \\ development} & \makecell{fed \\ operations \\ payments} & \makecell{financial \\ stability \\ regulation} & \makecell{international \\ economics} & \makecell{monetary \\ policy \\ inflation} & \makecell{real \\ economy \\ productivity} \\
\midrule
abstract & -0.444 & -0.015 & 1.152 & 0.437 & -0.414 & -0.525 \\
info & -0.372 & 0.116 & 0.172 & -0.033 & -0.014 & -0.102 \\
read & -0.174 & 0.029 & 0.611 & -0.067 & -0.144 & -0.259 \\
disunity & -0.229 & 0.215 & 0.433 & 0.118 & 0.065 & -0.314 \\
strain & -0.088 & 0.119 & 0.253 & 0.017 & 0.032 & -0.283 \\
strainUpper & -0.057 & 0.412 & 0.118 & -0.173 & 0.039 & -0.175 \\
\bottomrule
\end{tabular}

